# 26T3-ENG-R2C1M1P1-CICP-P1: Assessment [Obrigatório]
## Velocidade e Qualidade com Estruturas de Dados e Algoritmos [26E3_1]

---

### Exercício 1: Busca em estruturas lineares e impacto de organização

**Introdução**  
Em sistemas reais, a diferença entre uma busca linear e uma busca logarítmica costuma determinar se uma funcionalidade é utilizável ou não sob carga. Desenvolvedores avançados precisam traduzir requisitos de produto em decisões de engenharia, escolhendo estruturas e algoritmos que entreguem previsibilidade de tempo. Neste exercício, você implementará buscas clássicas e conectará o desempenho à organização dos dados, medindo custos e descrevendo a complexidade em Big O com base em contagens observáveis. Além disso, você fará a ponte entre arrays e estruturas baseadas em nós, evidenciando como a mesma operação pode ter custos radicalmente distintos em listas encadeadas. O objetivo é que você demonstre domínio de implementação, instrumentação e análise, evitando explicações genéricas e fundamentando conclusões em evidências produzidas no notebook.

**Tarefa**  
1. Implemente `linear_search(arr, target)` e `binary_search(sorted_arr, target)` retornando a posição encontrada ou -1.
   - a. Instrumente ambas para contar comparações de chaves e retorne também esse contador;
   - b. Crie um gerador de vetores de tamanho $n$ em três cenários: ordenado, reverso, aleatório;
   - c. Execute para $n$ em pelo menos 5 escalas (por exemplo, $10^2$ a $10^6$) e registre as contagens.
2. Para o caso de `binary_search`, implemente uma etapa explícita de verificação de pré-condição (array ordenado) e defina o comportamento quando a pré-condição falhar.
   - a. Proponha uma estratégia de "falha rápida" com custo menor do que ordenar todo o vetor;
   - b. Justifique a estratégia em termos de Big O e em termos de risco de falso positivo.
3. Implemente uma `SinglyLinkedList` mínima (nó com `value` e `next`) e reescreva a busca linear para operar sobre essa lista.
   - a. Compare as contagens com a busca linear em array para os mesmos dados e explique a diferença prática.

*Tarefa 1: Implementação, Instrumentação e Experimentos*

In [1]:
import random
import pandas as pd

# 1.a. Algoritmos instrumentados para contagem de comparações
def linear_search(arr, target):
    comparisons = 0
    for i in range(len(arr)):
        comparisons += 1
        if arr[i] == target:
            return i, comparisons
    return -1, comparisons

def binary_search(sorted_arr, target):
    comparisons = 0
    left, right = 0, len(sorted_arr) - 1

    while left <= right:
        mid = (left + right) // 2
        comparisons += 1
        if sorted_arr[mid] == target:
            return mid, comparisons
        elif sorted_arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1

    return -1, comparisons

# 1.b. Gerador de vetores
def generate_vector(n, scenario):
    if scenario == 'sorted':
        return list(range(n))
    elif scenario == 'reverse':
        return list(range(n, 0, -1))
    elif scenario == 'random':
        vec = list(range(n))
        random.shuffle(vec)
        return vec
    else:
        raise ValueError("Cenário inválido. Escolha 'sorted', 'reverse' ou 'random'.")

# 1.c. Execução experimental em 5 escalas
scales = [10**2, 10**3, 10**4, 10**5, 10**6]
results = []

for n in scales:
    target = -1  # Pior caso (elemento inexistente) para expor o custo assintótico

    for scenario in ['sorted', 'reverse', 'random']:
        arr = generate_vector(n, scenario)

        # Busca Linear
        _, comp_linear = linear_search(arr, target)

        # Busca Binária (válida apenas se ordenado)
        if scenario == 'sorted':
            _, comp_binary = binary_search(arr, target)
        else:
            comp_binary = "N/A"

        results.append({
            'Tamanho (n)': n,
            'Cenário': scenario,
            'Comparações Linear': comp_linear,
            'Comparações Binária': comp_binary
        })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

 Tamanho (n) Cenário  Comparações Linear Comparações Binária
         100  sorted                 100                   6
         100 reverse                 100                 N/A
         100  random                 100                 N/A
        1000  sorted                1000                   9
        1000 reverse                1000                 N/A
        1000  random                1000                 N/A
       10000  sorted               10000                  13
       10000 reverse               10000                 N/A
       10000  random               10000                 N/A
      100000  sorted              100000                  16
      100000 reverse              100000                 N/A
      100000  random              100000                 N/A
     1000000  sorted             1000000                  19
     1000000 reverse             1000000                 N/A
     1000000  random             1000000                 N/A


*Tarefa 2: Verificação de Pré-Condição e Falha Rápida*

In [2]:
def is_sorted_fast_fail(arr, k=100):
    """
    Estratégia de Falha Rápida:
    1. Verifica extremidades O(1).
    2. Realiza amostragem de k pares aleatórios adjacentes O(k).
    """
    n = len(arr)
    if n <= 1:
        return True

    # Validação rápida de extremidades
    if arr[0] > arr[-1]:
        return False

    # Validação por amostragem
    sample_size = min(k, n - 1)
    sampled_indices = random.sample(range(n - 1), sample_size)
    for idx in sampled_indices:
        if arr[idx] > arr[idx + 1]:
            return False

    return True

def safe_binary_search(arr, target, k=100):
    if not is_sorted_fast_fail(arr, k):
        raise ValueError("Pré-condição falhou: o array não está devidamente ordenado.")
    return binary_search(arr, target)

*Justificativa Técnica*

- Complexidade Assintótica ($Big\ O$):
  - Verificar se um vetor está totalmente ordenado custa $O(n)$. Ordená-lo antes da busca custa $O(n \log n)$.
  - A estratégia de amostragem proposta possui custo de tempo $O(k)$ em que $k$ é uma constante fixa (ex: $k = 100$). Assintoticamente em relação ao tamanho da entrada $n$, trata-se de um custo $O(1)$, sendo dramaticamente inferior a $O(n)$ e $O(n \log n)$.
- Risco de Falso Positivo:
  - Por ser uma verificação probabilística e parcial, há o risco de o algoritmo declarar um vetor como ordenado quando ele apresenta inversões locais fora dos pontos amostrados.
  - Caso ocorra um falso positivo, a busca binária executará sobre dados desalinhados, podendo retornar $-1$ mesmo se o elemento estiver presente ou retornar um índice incorreto.

*Tarefa 3: SinglyLinkedList e Comparativo Prático*

In [3]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class SinglyLinkedList:
    def __init__(self):
        self.head = None

    def append(self, value):
        new_node = Node(value)
        if not self.head:
            self.head = new_node
            return
        curr = self.head
        while curr.next:
            curr = curr.next
        curr.next = new_node

def linear_search_sll(sll, target):
    comparisons = 0
    curr = sll.head
    idx = 0
    while curr:
        comparisons += 1
        if curr.value == target:
            return idx, comparisons
        curr = curr.next
        idx += 1
    return -1, comparisons

# Experimento comparativo
n = 10**4
data = generate_vector(n, 'random')

sll = SinglyLinkedList()
for val in data:
    sll.append(val)

_, comp_array = linear_search(data, -1)
_, comp_sll = linear_search_sll(sll, -1)

print(f"Comparações no Array: {comp_array}")
print(f"Comparações na SinglyLinkedList: {comp_sll}")

Comparações no Array: 10000
Comparações na SinglyLinkedList: 10000


*Comparação e Explicação Prática*
- Contagem de Comparações de Chaves: É identicamente igual entre o Array e a SinglyLinkedList para uma mesma sequência de dados. Em ambos os casos, no pior cenário de busca linear, o algoritmo realiza exatamente $n$ comparações ($O(n)$).
- Diferença Prática de Desempenho (Tempo Real de CPU):
  - Array (Lista em Python): Armazena referências de forma contígua em memória. Isso favorece a localidade espacial de referência, permitindo que a CPU utilize mecanismos de prefetching e aproveite os caches L1/L2 com alto índice de cache hits.
  - SinglyLinkedList: Cada nó é instanciado em posições esparsas da memória Heap. A navegação exige um salto de ponteiro a cada elemento (curr = curr.next), gerando frequentes cache misses e maior sobrecarrega (overhead) de memória devido à alocação de ponteiros. Na prática, o tempo de execução do tempo de CPU na lista encadeada é significativamente superior ao do array, mesmo com o mesmo número de comparações de chaves.

### Exercício 2: Ordenações quadráticas com instrumentação e diagnóstico

**Introdução**  
Ordenar dados é uma operação central em engenharia de software, porém o ponto crítico não é "saber ordenar", e sim reconhecer quando uma implementação é inadequada para o volume e o padrão de entrada. Algoritmos como Bubble Sort, Selection Sort e Insertion Sort têm complexidades conhecidas, mas um engenheiro precisa conseguir demonstrar esse custo com medições internas do algoritmo, como número de cópias e comparações. Neste exercício, você implementará essas ordenações com instrumentação rigorosa, testará em diferentes distribuições e produzirá uma análise que conecte comportamento observado e Big O. Ao final, você integrará a ordenação com uma estrutura em árvore para validar invariantes de ordenação por meio de travessias, reforçando o vínculo entre arrays e estruturas dinâmicas.

**Tarefa**  
1. Implemente `bubble_sort`, `selection_sort` e `insertion_sort` sobre listas Python, retornando também:
   - a. número de comparações de chaves;
   - b. número de cópias (considere uma troca como 3 cópias).
2. Projete um experimento reprodutível com pelo menos quatro padrões de entrada: ordenado, reverso, quase ordenado, aleatório.
   - a. Para cada padrão, execute com ao menos 4 tamanhos crescentes e registre comparações e cópias;
   - b. Identifique qual algoritmo se beneficia mais do caso "quase ordenado" e explique por que isso ocorre.
3. Construa uma BinarySearch Tree (BST) de inteiros e use-a para validar ordenação.
   - a. Insira todos os elementos do array original na BST, faça uma travessia in-order e verifique que o resultado é ordenado;
   - b. Compare o custo total de ordenação por BST (inserções + in-order) com Insertion Sort, usando Big O e evidências do seu experimento.

*Tarefa 1: Algoritmos de Ordenação Instrumentados*

In [4]:
def bubble_sort(arr):
    arr_copy = arr.copy()
    n = len(arr_copy)
    comparisons = 0
    copies = 0

    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            comparisons += 1
            if arr_copy[j] > arr_copy[j + 1]:
                # Troca (swap) equivale a 3 cópias de elementos
                arr_copy[j], arr_copy[j + 1] = arr_copy[j + 1], arr_copy[j]
                copies += 3
                swapped = True
        if not swapped:
            break

    return arr_copy, comparisons, copies


def selection_sort(arr):
    arr_copy = arr.copy()
    n = len(arr_copy)
    comparisons = 0
    copies = 0

    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            comparisons += 1
            if arr_copy[j] < arr_copy[min_idx]:
                min_idx = j

        if min_idx != i:
            # Troca entre o elemento atual e o mínimo encontrado (3 cópias)
            arr_copy[i], arr_copy[min_idx] = arr_copy[min_idx], arr_copy[i]
            copies += 3

    return arr_copy, comparisons, copies


def insertion_sort(arr):
    arr_copy = arr.copy()
    n = len(arr_copy)
    comparisons = 0
    copies = 0

    for i in range(1, n):
        key = arr_copy[i]
        copies += 1  # Guarda a chave em variável temporária
        j = i - 1

        while j >= 0:
            comparisons += 1
            if arr_copy[j] > key:
                arr_copy[j + 1] = arr_copy[j]
                copies += 1  # Deslocamento de elemento
                j -= 1
            else:
                break

        arr_copy[j + 1] = key
        copies += 1  # Inserção da chave na posição correta

    return arr_copy, comparisons, copies

*Tarefa 2: Experimento Reprodutível e Análise*

In [5]:
import random
import pandas as pd

def generate_data(n, pattern):
    if pattern == 'sorted':
        return list(range(n))
    elif pattern == 'reverse':
        return list(range(n, 0, -1))
    elif pattern == 'nearly_sorted':
        arr = list(range(n))
        # Provoca trocas em aproximadamente 5% dos elementos
        swaps = max(1, int(n * 0.05))
        for _ in range(swaps):
            i, j = random.sample(range(n), 2)
            arr[i], arr[j] = arr[j], arr[i]
        return arr
    elif pattern == 'random':
        arr = list(range(n))
        random.shuffle(arr)
        return arr
    else:
        raise ValueError("Padrão inválido.")

sizes = [100, 300, 500, 1000]
patterns = ['sorted', 'reverse', 'nearly_sorted', 'random']
exp_results = []

for size in sizes:
    for pat in patterns:
        data = generate_data(size, pat)

        _, comp_b, copy_b = bubble_sort(data)
        _, comp_s, copy_s = selection_sort(data)
        _, comp_i, copy_i = insertion_sort(data)

        exp_results.append({
            'Tamanho (N)': size,
            'Padrão': pat,
            'Bubble (Comp/Cóp)': f"{comp_b} / {copy_b}",
            'Selection (Comp/Cóp)': f"{comp_s} / {copy_s}",
            'Insertion (Comp/Cóp)': f"{comp_i} / {copy_i}"
        })

df_exp = pd.DataFrame(exp_results)
print(df_exp.to_string(index=False))

 Tamanho (N)        Padrão Bubble (Comp/Cóp) Selection (Comp/Cóp) Insertion (Comp/Cóp)
         100        sorted            99 / 0             4950 / 0             99 / 198
         100       reverse      4950 / 14850           4950 / 150          4950 / 5148
         100 nearly_sorted       3869 / 1173            4950 / 15            490 / 589
         100        random       4830 / 7170           4950 / 282          2487 / 2588
         300        sorted           299 / 0            44850 / 0            299 / 598
         300       reverse    44850 / 134550          44850 / 450        44850 / 45448
         300 nearly_sorted      41022 / 7431           44850 / 45          2774 / 3075
         300        random     44550 / 67611          44850 / 885        22830 / 23135
         500        sorted           499 / 0           124750 / 0            499 / 998
         500       reverse   124750 / 374250         124750 / 750      124750 / 125748
         500 nearly_sorted    121975 / 1663

*Análise do Caso "Quase Ordenado*
- Algoritmo Beneficiado: O Insertion Sort.   
- Justificação Técnica:
  - No Insertion Sort, o laço interno while interrompe a verificação assim que encontra um elemento menor ou igual à chave. Quando o vetor está "quase ordenado", a maioria dos elementos já se encontra próxima da sua posição final.
  - Isso reduz drasticamente o número de comparações e deslocamentos internos por elemento para $O(1)$ no caso ideal, aproximando o custo total de tempo de $O(n)$.   
  - Em contrapartida, o Selection Sort realiza sempre $\frac{n(n-1)}{2}$ comparações ($O(n^2)$) independentemente da ordenação prévia. O Bubble Sort otimizado deteta a ausência de trocas, mas ainda assim efetua mais comparações do que o Insertion Sort para pequenas perturbações locais.

*Tarefa 3: Ordenação e Validação via Binary Search Tree (BST)*

In [6]:
class BSTNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None
        self.insert_ops = 0  # Contabilizador de comparações na inserção

    def insert(self, key):
        if not self.root:
            self.root = BSTNode(key)
        else:
            self._insert_recursive(self.root, key)

    def _insert_recursive(self, node, key):
        self.insert_ops += 1
        if key < node.key:
            if node.left is None:
                node.left = BSTNode(key)
            else:
                self._insert_recursive(node.left, key)
        else:
            if node.right is None:
                node.right = BSTNode(key)
            else:
                self._insert_recursive(node.right, key)

    def inorder(self):
        result = []
        self._inorder_recursive(self.root, result)
        return result

    def _inorder_recursive(self, node, result):
        if node:
            self._inorder_recursive(node.left, result)
            result.append(node.key)
            self._inorder_recursive(node.right, result)


# Validação e Experimento Comparativo com Insertion Sort
test_data = generate_data(500, 'random')

# 3.a. Inserção na BST e verificação via travessia in-order
bst = BST()
for val in test_data:
    bst.insert(val)

bst_sorted_data = bst.inorder()
is_valid = bst_sorted_data == sorted(test_data)

# Execução do Insertion Sort para dados idênticos
_, ins_comp, ins_copies = insertion_sort(test_data)

print(f"Resultado da BST está ordenado? {is_valid}")
print(f"Operações de inserção (comparações) na BST: {bst.insert_ops}")
print(f"Comparações no Insertion Sort: {ins_comp}")

Resultado da BST está ordenado? True
Operações de inserção (comparações) na BST: 4946
Comparações no Insertion Sort: 64192


*Comparação de Custo Assintótico e Prático (BST vs Insertion Sort)*
- Custo da BST (Inserções + Travessia In-order):
  - Caso Médio (Entrada Aleatória): A inserção de $n$ elementos numa BST não balanceada tem custo de tempo $O(n \log n)$. A travessia in-order posterior percorre cada nó exatamente uma vez em $O(n)$. O custo total resultante é $O(n \log n)$.   
  - Pior Caso (Entrada Ordenada ou Reversa): A BST degrada numa lista encadeada (árvore degenerada) com altura $h = n$. O custo de inserção eleva-se para $O(n^2)$, resultando num custo global de $O(n^2)$.
- Insertion Sort:
  - Apresenta custo de tempo $O(n)$ para o melhor caso (ordenado) e $O(n^2)$ para casos médios e piores (aleatório/reverso).  
- Conclusão com Base em Evidências: Em vetores aleatórios de média/grande escala, o método BST supera o Insertion Sort por atingir complexidade $O(n \log n)$ contra o custo quadrático $O(n^2)$ do Insertion Sort. No entanto, em vetores já ordenados, o Insertion Sort é superior ($O(n)$) enquanto a BST sofre degradação de desempenho ($O(n^2)$).



### Exercício 3: Otimização orientada a Big O com requisitos de desempenho

**Introdução**  
Otimizar não é "tornar mais rápido" de forma vaga, é reduzir o custo assintótico e os fatores dominantes sob restrições explícitas. Em ambientes profissionais, você receberá um código funcional que falha em escala, e sua responsabilidade será redesenhá-lo com base em análise e instrumentação. Neste exercício, você partirá de uma solução deliberadamente ineficiente para deduplicação e seleção de estatísticas, reescrevendo-a para reduzir complexidade e justificar tecnicamente cada decisão. O exercício exige que você demonstre capacidade de reconhecer gargalos, propor estruturas adequadas e validar corretude com testes. Para consolidar a integração com estruturas dinâmicas, você deverá suportar uma variante em árvore, comparando a eficiência de operações e evidenciando trade-offs de implementação.

**Tarefa**  
1. Implemente `deduplicate_slow(arr)` com abordagem ingênua $O(n^2)$ e depois implemente `deduplicate_fast(arr)` com $O(n)$.
   - a. Em `deduplicate_fast`, você deve preservar ordem relativa dos primeiros aparecimentos;
   - b. Instrumente ambas para contar comparações e escreva testes que provem equivalência de saída.
2. Implemente `k_smallest(arr, k)` de duas formas:
   - a. versão A: ordena todo o array e pega os $k$ primeiros;
   - b. versão B: usa QuickSelect para encontrar o $k$-ésimo e particionar sem ordenar totalmente;
   - c. Compare as duas em termos de Big O e em contagens de operações para diferentes tamanhos e valores de $k$.
3. Integre uma variante em árvore:
   - a. Insira os valores em uma BST e implemente `k_smallest_bst(root, k)` por travessia controlada;
   - b. Meça a eficiência esperada e discuta o pior caso, incluindo quando a BST degrada para lista encadeada.

*Tarefa 1: Deduplicação Ineficiente ($O(n^2)$) e Otimizada ($O(n)$)*

In [7]:
import random

# 1.a & 1.b: Abordagem $O(n^2)$ por pesquisa linear na lista de saída
def deduplicate_slow(arr):
    seen = []
    comparisons = 0
    for item in arr:
        found = False
        for x in seen:
            comparisons += 1
            if x == item:
                found = True
                break
        if not found:
            seen.append(item)
    return seen, comparisons

# 1.a & 1.b: Abordagem $O(n)$ preservando a ordem com Tabela de Dispersão (Set)
def deduplicate_fast(arr):
    seen_set = set()
    result = []
    comparisons = 0
    for item in arr:
        comparisons += 1 # Verificação/Hash lookup no conjunto
        if item not in seen_set:
            seen_set.add(item)
            result.append(item)
    return result, comparisons

# Testes automatizados para comprovar equivalência e comparar operações
def test_deduplication():
    test_cases = [
        [1, 2, 2, 3, 4, 4, 4, 5],
        [5, 4, 3, 2, 1],
        [1, 1, 1, 1, 1],
        [random.randint(1, 50) for _ in range(1000)]
    ]

    for idx, arr in enumerate(test_cases, 1):
        slow_res, slow_comp = deduplicate_slow(arr)
        fast_res, fast_comp = deduplicate_fast(arr)

        # Validação de equivalência
        assert slow_res == fast_res, f"Falha de equivalência no caso {idx}"

        print(f"Caso {idx} (N={len(arr)}): Resultados idênticos.")
        print(f"  - Lento O(n²): {slow_comp} comparações")
        print(f"  - Rápido O(n): {fast_comp} comparações\n")

test_deduplication()

Caso 1 (N=8): Resultados idênticos.
  - Lento O(n²): 20 comparações
  - Rápido O(n): 8 comparações

Caso 2 (N=5): Resultados idênticos.
  - Lento O(n²): 10 comparações
  - Rápido O(n): 5 comparações

Caso 3 (N=5): Resultados idênticos.
  - Lento O(n²): 4 comparações
  - Rápido O(n): 5 comparações

Caso 4 (N=1000): Resultados idênticos.
  - Lento O(n²): 24997 comparações
  - Rápido O(n): 1000 comparações



*Tarefa 2: Seleção dos $k$ Menores Elementos (k_smallest)*

In [8]:
# 2.a: Versão A - Ordenação Total O(n log n)
def k_smallest_sort(arr, k):
    arr_copy = arr.copy()
    comparisons = 0

    # Simulação de ordenação com contagem (usando Python sorted / Timsort)
    # Timsort faz no máximo n * log2(n) comparações no caso médio/pior
    import math
    n = len(arr_copy)
    comparisons = int(n * math.log2(n)) if n > 0 else 0

    sorted_arr = sorted(arr_copy)
    return sorted_arr[:k], comparisons

# 2.b: Versão B - QuickSelect O(n) médio
def quickselect_partition(arr, low, high, stats):
    pivot = arr[high]
    i = low - 1
    for j in range(low, high):
        stats['comparisons'] += 1
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
            stats['swaps'] += 1
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    stats['swaps'] += 1
    return i + 1

def quickselect_helper(arr, low, high, k_idx, stats):
    if low <= high:
        pivot_idx = quickselect_partition(arr, low, high, stats)
        if pivot_idx == k_idx:
            return
        elif pivot_idx > k_idx:
            quickselect_helper(arr, low, pivot_idx - 1, k_idx, stats)
        else:
            quickselect_helper(arr, pivot_idx + 1, high, k_idx, stats)

def k_smallest_quickselect(arr, k):
    arr_copy = arr.copy()
    stats = {'comparisons': 0, 'swaps': 0}
    if k > 0 and k <= len(arr_copy):
        quickselect_helper(arr_copy, 0, len(arr_copy) - 1, k - 1, stats)
    return sorted(arr_copy[:k]), stats['comparisons'] # Retorna os k menores particionados

# 2.c: Experimento comparativo entre Versão A e Versão B
import pandas as pd

sizes = [500, 2000, 10000]
k_ratios = [0.05, 0.25, 0.50]
k_results = []

for n in sizes:
    data = [random.randint(1, 100000) for _ in range(n)]
    for ratio in k_ratios:
        k = int(n * ratio)
        _, comp_a = k_smallest_sort(data, k)
        _, comp_b = k_smallest_quickselect(data, k)

        k_results.append({
            'Tamanho (N)': n,
            'k': k,
            'Proporção k/N': f"{int(ratio*100)}%",
            'Versão A (Sort) - Comparações Est.': comp_a,
            'Versão B (QuickSelect) - Comparações': comp_b
        })

df_k = pd.DataFrame(k_results)
print(df_k.to_string(index=False))

 Tamanho (N)    k Proporção k/N  Versão A (Sort) - Comparações Est.  Versão B (QuickSelect) - Comparações
         500   25            5%                                4482                                  1281
         500  125           25%                                4482                                  1811
         500  250           50%                                4482                                  1277
        2000  100            5%                               21931                                  5342
        2000  500           25%                               21931                                  6398
        2000 1000           50%                               21931                                  3966
       10000  500            5%                              132877                                 27454
       10000 2500           25%                              132877                                 22021
       10000 5000           50%               

*Comparativo de Complexidade ($Big\ O$)*
- Versão A (Ordenação Total): Possui complexidade temporal $O(n \log n)$. Ordena todo o vetor independentemente do valor de $k$, realizando trabalho desnecessário para $k \ll n$.   
- Versão B (QuickSelect): Possui complexidade temporal média de $O(n)$. Descarta recursivamente uma das partições, reduzindo a área de busca a cada passo ($\approx n + \frac{n}{2} + \frac{n}{4} + \dots = 2n$). No entanto, o seu pior caso assintótico é $O(n^2)$ se os pivôs forem mal escolhidos.

*Tarefa 3: Seleção de $k$-Menores em BST e Análise*

In [9]:
class BSTNode:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None

    def insert(self, val):
        if not self.root:
            self.root = BSTNode(val)
        else:
            self._insert_rec(self.root, val)

    def _insert_rec(self, node, val):
        if val < node.val:
            if node.left is None:
                node.left = BSTNode(val)
            else:
                self._insert_rec(node.left, val)
        else:
            if node.right is None:
                node.right = BSTNode(val)
            else:
                self._insert_rec(node.right, val)

# 3.a: Travessia in-order interrompida ao atingir k elementos
def k_smallest_bst(root, k):
    result = []
    visited_nodes = 0

    def inorder(node):
        nonlocal visited_nodes
        if not node or len(result) == k:
            return

        inorder(node.left)

        if len(result) < k:
            result.append(node.val)
            visited_nodes += 1
            inorder(node.right)

    inorder(root)
    return result, visited_nodes

# Exemplo de teste com dados aleatórios
data_tree = [random.randint(1, 1000) for _ in range(100)]
bst = BST()
for x in data_tree:
    bst.insert(x)

k_val = 10
res_bst, nodes_visited = k_smallest_bst(bst.root, k_val)
print(f"Os {k_val} menores elementos via BST: {res_bst}")
print(f"Nós visitados na travessia controlada: {nodes_visited}")

Os 10 menores elementos via BST: [16, 32, 38, 39, 53, 58, 76, 89, 94, 104]
Nós visitados na travessia controlada: 10


*Análise de Eficiência e Degradação da BST*
- Eficiência Esperada:
  - Construção da Árvore: Para $n$ elementos inseridos aleatoriamente, a altura esperada é $O(\log n)$, custando $O(n \log n)$ no total.  
  - Consulta dos $k$ Menores: A travessia in-order interrompida percorre apenas a subárvore esquerda relevante e atinge os $k$ primeiros nós em tempo $O(h + k)$, em que $h$ é a altura da árvore.
- Eficiência Esperada:
  - Se os dados forem inseridos de forma estritamente ordenada ou inversa, a BST degrada numa lista encadeada (árvore desbalanceada de altura $h = n$).
  - Na BST degenerada, a inserção atinge o custo quadrático $O(n^2)$. A busca pelos $k$ menores passa a exigir a travessia sequencial de $k$ nós profundos com custo $O(k)$ sobre uma estrutura linear de acesso $O(n)$.     


*Complemento da Tarefa 3: Busca e Deleção na BST (folha, um filho e dois filhos)*

In [10]:
# Complemento: busca, deleção e validação da BST
# Reaproveita BSTNode (atributo .val) e BST da Tarefa 3 acima.
# Convenção de duplicatas herdada de _insert_rec: valores MENORES à esquerda
# e valores MAIORES OU IGUAIS à direita  ->  left < node.val <= right.
import random

def bst_search(root, val):
    """Busca iterativa: devolve (nó ou None, comparações). Custo O(h)."""
    comparisons, node = 0, root
    while node:
        comparisons += 1
        if val == node.val:
            return node, comparisons
        node = node.left if val < node.val else node.right
    return None, comparisons

def _min_node(node):
    while node.left:
        node = node.left
    return node

def bst_delete(root, val):
    """Remove UMA ocorrência de val. Devolve (nova_raiz, caso_tratado).
    Casos: 'folha', 'um_filho', 'dois_filhos' ou 'nao_encontrado'."""
    parent, node = None, root
    while node and node.val != val:                      # 1) localizar o nó
        parent, node = node, (node.left if val < node.val else node.right)
    if node is None:
        return root, 'nao_encontrado'

    if node.left and node.right:                         # 2) dois filhos
        # Sucessor in-order = menor valor da subárvore direita. Copiar o valor
        # dele para o nó preserva a ordem: tudo à esquerda < sucessor <= resto da direita.
        succ_parent, succ = node, node.right
        while succ.left:
            succ_parent, succ = succ, succ.left
        node.val = succ.val
        # O sucessor nunca tem filho esquerdo: removê-lo cai no caso folha/um filho.
        if succ_parent is node:
            succ_parent.right = succ.right
        else:
            succ_parent.left = succ.right
        return root, 'dois_filhos'

    caso = 'folha' if (node.left is None and node.right is None) else 'um_filho'
    child = node.left if node.left else node.right       # 3) folha (None) ou filho único
    if parent is None:                                   # removendo a raiz
        return child, caso
    if parent.left is node:
        parent.left = child
    else:
        parent.right = child
    return root, caso

def is_valid_bst(node, low=float('-inf'), high=float('inf')):
    """Invariante: todo valor da subárvore esquerda < nó <= subárvore direita.
    Verifica limites herdados de TODOS os ancestrais, não só do pai."""
    if node is None:
        return True
    if not (low <= node.val < high):
        return False
    return is_valid_bst(node.left, low, node.val) and is_valid_bst(node.right, node.val, high)

def inorder_vals(node):
    return inorder_vals(node.left) + [node.val] + inorder_vals(node.right) if node else []

def show(node):
    """Representação textual compacta: valor(esq, dir)."""
    if node is None:
        return '·'
    if node.left is None and node.right is None:
        return str(node.val)
    return f"{node.val}({show(node.left)}, {show(node.right)})"

# --- Demonstração dos três casos em uma árvore conhecida ---
arvore = BST()
for v in [50, 30, 70, 20, 40, 60, 80, 35, 65]:
    arvore.insert(v)
print("Árvore inicial:          ", show(arvore.root))

for alvo, esperado in [(20, 'folha'), (60, 'um_filho'), (70, 'dois_filhos'), (50, 'dois_filhos'), (99, 'nao_encontrado')]:
    arvore.root, caso = bst_delete(arvore.root, alvo)
    assert caso == esperado, (alvo, caso)
    valida = is_valid_bst(arvore.root)
    ordenada = inorder_vals(arvore.root) == sorted(inorder_vals(arvore.root))
    print(f"delete({alvo:>2}) caso={caso:<14} válida={valida} in-order ordenado={ordenada} -> {show(arvore.root)}")
    assert valida and ordenada

# --- Teste de estresse: inserções e deleções aleatórias com duplicatas ---
random.seed(7)
arvore, referencia = BST(), []
for _ in range(600):
    v = random.randint(1, 150)                 # faixa pequena força duplicatas
    arvore.insert(v); referencia.append(v)
contagem = {'folha': 0, 'um_filho': 0, 'dois_filhos': 0}
for _ in range(400):
    v = random.choice(referencia)
    arvore.root, caso = bst_delete(arvore.root, v)
    referencia.remove(v)
    contagem[caso] += 1
    assert is_valid_bst(arvore.root), "propriedade de ordenação violada"
    assert inorder_vals(arvore.root) == sorted(referencia), "conteúdo divergente"
print(f"\nEstresse: 600 inserções e 400 deleções aleatórias (com duplicatas)")
print(f"Casos exercitados: {contagem}")
print(f"Invariante verificada após CADA deleção: OK | elementos restantes = {len(referencia)}")
no, comps = bst_search(arvore.root, referencia[0])
print(f"Busca de {referencia[0]} após as deleções: encontrado={no is not None} em {comps} comparações")

Árvore inicial:           50(30(20, 40(35, ·)), 70(60(·, 65), 80))
delete(20) caso=folha          válida=True in-order ordenado=True -> 50(30(·, 40(35, ·)), 70(60(·, 65), 80))
delete(60) caso=um_filho       válida=True in-order ordenado=True -> 50(30(·, 40(35, ·)), 70(65, 80))
delete(70) caso=dois_filhos    válida=True in-order ordenado=True -> 50(30(·, 40(35, ·)), 80(65, ·))
delete(50) caso=dois_filhos    válida=True in-order ordenado=True -> 65(30(·, 40(35, ·)), 80)
delete(99) caso=nao_encontrado válida=True in-order ordenado=True -> 65(30(·, 40(35, ·)), 80)

Estresse: 600 inserções e 400 deleções aleatórias (com duplicatas)
Casos exercitados: {'folha': 4, 'um_filho': 45, 'dois_filhos': 351}
Invariante verificada após CADA deleção: OK | elementos restantes = 200
Busca de 145 após as deleções: encontrado=True em 6 comparações


*Análise da Deleção e da Preservação da Propriedade de Ordenação*
- Os três casos de remoção:
  - Folha: o ponteiro do pai passa a ser `None` (exemplo: `delete(20)`).
  - Um filho: o único filho ocupa o lugar do nó removido (exemplo: `delete(60)`, em que o 65 sobe).
  - Dois filhos: o valor do nó é substituído pelo do sucessor in-order (o menor valor da subárvore direita), e o sucessor é removido do lugar original. Como ele nunca tem filho esquerdo, essa segunda remoção sempre cai no caso folha ou um filho (exemplos: `delete(70)` e a remoção da raiz com `delete(50)`).
- Por que a ordenação é preservada: o sucessor é o menor valor da subárvore direita. Assim, todos os valores da esquerda continuam menores que ele, e todos os valores restantes da direita continuam maiores ou iguais. A convenção de duplicatas herdada de `_insert_rec` (esquerda $<$ nó $\le$ direita) também é mantida.
- Validação: `is_valid_bst` verifica os limites herdados de todos os ancestrais, e não só a comparação com o pai. Um teste só com o pai não detecta um valor que viola a ordem de um avô. Além disso, conferimos que a travessia in-order continua ordenada.
- Evidência: no teste de estresse com 600 inserções e 400 deleções aleatórias com duplicatas, os três casos foram exercitados (4 folhas, 45 com um filho e 351 com dois filhos). A invariante e o conteúdo, comparado a uma lista de referência, foram verificados após cada deleção.
- Complexidade: localizar o nó custa $O(h)$, e localizar o sucessor custa mais $O(h)$, então a deleção custa $O(h)$. Isso é $O(\log n)$ no caso esperado e $O(n)$ quando a árvore degrada para uma lista encadeada.

### Exercício 4: Hashtable com colisões e encadeamento por lista

**Introdução**  
Hashtables são a base de índices, caches, memoization e inúmeras estruturas de alto desempenho. O ponto de maturidade técnica é entender que "$O(1)$" é uma promessa condicional, e que colisões, fator de carga e estratégia de resolução determinam o desempenho real. Neste exercício, você implementará uma hashtable do zero, com hashing, redimensionamento e tratamento explícito de colisões. A exigência é que a colisão seja resolvida por encadeamento, usando uma lista encadeada por bucket, o que conecta diretamente estruturas baseadas em nós com acesso eficiente. Você também deverá fornecer uma análise de complexidade dependente do fator de carga e demonstrar, com testes, que a estrutura mantém corretude após rehash. A qualidade da entrega será avaliada pela precisão da implementação e pela clareza das evidências.

**Tarefa**  
1. Implemente `HashTableChained` com métodos `put(key, value)`, `get(key)`, `delete(key)` e `__len__()`.
   - a. Cada bucket deve ser uma lista encadeada de pares `(key, value)`;
   - b. Implemente atualização de valor para chave existente sem inserir duplicata.
2. Implemente redimensionamento automático baseado em fator de carga.
   - a. Defina um limiar e faça rehash para uma capacidade maior;
   - b. Demonstre por teste que todas as chaves permanecem acessíveis após o rehash.
3. Instrumente e avalie desempenho.
   - a. Conte comparações dentro dos buckets e relacione o custo esperado com o fator de carga;
   - b. Construa um cenário adversarial com chaves que colidem e discuta o impacto em Big O.

*Tarefa 1 e Tarefa 2: Implementação da HashTableChained com Redimensionamento Automático*

In [11]:
class HashNode:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.next = None

class HashTableChained:
    def __init__(self, initial_capacity=8, load_factor_threshold=0.75):
        self.capacity = initial_capacity
        self.load_factor_threshold = load_factor_threshold
        self.buckets = [None] * self.capacity
        self.size = 0
        self.last_op_comparisons = 0

    def _hash(self, key):
        # Função de hashing utilizando o hash embutido adaptado à capacidade atual
        return hash(key) % self.capacity

    def __len__(self):
        return self.size

    def get_load_factor(self):
        return self.size / self.capacity

    def put(self, key, value):
        index = self._hash(key)
        curr = self.buckets[index]

        # 1.b: Atualização de valor para chave existente sem inserir duplicata
        while curr:
            if curr.key == key:
                curr.value = value
                return
            curr = curr.next

        # Inserção no início da lista encadeada do bucket
        new_node = HashNode(key, value)
        new_node.next = self.buckets[index]
        self.buckets[index] = new_node
        self.size += 1

        # 2.a: Redimensionamento automático baseado no fator de carga
        if self.get_load_factor() > self.load_factor_threshold:
            self._resize(self.capacity * 2)

    def get(self, key):
        index = self._hash(key)
        curr = self.buckets[index]
        self.last_op_comparisons = 0

        while curr:
            self.last_op_comparisons += 1
            if curr.key == key:
                return curr.value
            curr = curr.next
        return None

    def delete(self, key):
        index = self._hash(key)
        curr = self.buckets[index]
        prev = None
        self.last_op_comparisons = 0

        while curr:
            self.last_op_comparisons += 1
            if curr.key == key:
                if prev:
                    prev.next = curr.next
                else:
                    self.buckets[index] = curr.next
                self.size -= 1
                return True
            prev = curr
            curr = curr.next
        return False

    def _resize(self, new_capacity):
        old_buckets = self.buckets
        self.capacity = new_capacity
        self.buckets = [None] * self.capacity
        self.size = 0

        # Rehash de todos os elementos para a nova estrutura
        for head in old_buckets:
            curr = head
            while curr:
                # Reinsere cada elemento ajustando para o novo índice
                index = self._hash(curr.key)
                new_node = HashNode(curr.key, curr.value)
                new_node.next = self.buckets[index]
                self.buckets[index] = new_node
                self.size += 1
                curr = curr.next


# 2.b: Teste que demonstra acessibilidade das chaves após Rehash
def test_rehash():
    ht = HashTableChained(initial_capacity=4, load_factor_threshold=0.75)
    test_data = {f"key_{i}": i * 10 for i in range(20)}

    # Inserção de dados provocando múltiplos rehashes
    initial_cap = ht.capacity
    for k, v in test_data.items():
        ht.put(k, v)

    print(f"Capacidade inicial: {initial_cap}")
    print(f"Capacidade final pós-rehash: {ht.capacity}")
    print(f"Fator de Carga final: {ht.get_load_factor():.2f}")

    # Verificação de acessibilidade
    all_accessible = all(ht.get(k) == v for k, v in test_data.items())
    print(f"Todas as {len(test_data)} chaves continuam acessíveis? {all_accessible}\n")

test_rehash()

Capacidade inicial: 4
Capacidade final pós-rehash: 32
Fator de Carga final: 0.62
Todas as 20 chaves continuam acessíveis? True



*Tarefa 3: Instrumentação e Avaliação de Desempenho*

In [12]:
class AdversarialKey:
    """Chave customizada para forçar colisões no mesmo bucket."""
    def __init__(self, val):
        self.val = val

    def __hash__(self):
        return 42  # Força todos os elementos a colidirem no mesmo hash

    def __eq__(self, other):
        return isinstance(other, AdversarialKey) and self.val == other.val

    def __repr__(self):
        return f"AdvKey({self.val})"


# 3.a & 3.b: Experimento Comparativo (Cenário Normal vs Cenário Adversarial)
def evaluate_hashtable_performance():
    n = 500

    # 1. Cenário Normal (Chaves distribuídas)
    ht_normal = HashTableChained(initial_capacity=16, load_factor_threshold=0.75)
    for i in range(n):
        ht_normal.put(f"normal_key_{i}", i)

    normal_comparisons = []
    for i in range(n):
        ht_normal.get(f"normal_key_{i}")
        normal_comparisons.append(ht_normal.last_op_comparisons)

    avg_comp_normal = sum(normal_comparisons) / len(normal_comparisons)

    # 2. Cenário Adversarial (Todas as chaves colidem no mesmo bucket)
    ht_adversarial = HashTableChained(initial_capacity=16, load_factor_threshold=0.75)
    for i in range(n):
        ht_adversarial.put(AdversarialKey(i), i)

    adv_comparisons = []
    for i in range(n):
        ht_adversarial.get(AdversarialKey(i))
        adv_comparisons.append(ht_adversarial.last_op_comparisons)

    avg_comp_adv = sum(adv_comparisons) / len(adv_comparisons)

    print("--- Resultados de Avaliação de Desempenho ---")
    print(f"Cenário Normal   - Média de Comparações/Busca: {avg_comp_normal:.2f}")
    print(f"Cenário Adversarial - Média de Comparações/Busca: {avg_comp_adv:.2f}")

evaluate_hashtable_performance()

--- Resultados de Avaliação de Desempenho ---
Cenário Normal   - Média de Comparações/Busca: 1.24
Cenário Adversarial - Média de Comparações/Busca: 250.50


*Análise de Complexidade e Impacto em Big O*
- Relação com o Fator de Carga ($\alpha = \frac{n}{m}$):
  - O custo esperado de busca, inserção e remoção em uma Hashtable com encadeamento por lista é $O(1 + \alpha)$, onde $\alpha$ representa a extensão média de elementos por bucket.  
  - Mantendo o fator de carga delimitado por um limiar fixo (ex: $\alpha \le 0.75$) através do rehash dinâmico, o número médio de elementos por bucket permanece constante, garantindo tempo amortizado $O(1)$.
- Impacto do Cenário Adversarial:
  - Quando todas as $n$ chaves geram o mesmo índice de hash (colisão total), a estrutura do bucket degenera em uma única lista encadeada.   
  - O tempo de busca deixa de ser $O(1)$ e passa a exigir travessia linear de tamanho $n$, elevando a complexidade temporal ao pior caso $O(n)$.   

### Exercício 5: Pilha e fila com invariantes e integração com árvore de expressão

**Introdução**  
Pilhas e filas aparecem em parsing, execução de tarefas, sistemas de mensagens e avaliação de expressões. O domínio esperado aqui vai além de "usar list como stack": você deve construir estruturas com invariantes claros, tratamento de underflow e overflow e testes que evidenciem comportamento. Neste exercício, você implementará Stack e Queue com capacidade configurável e erros explícitos, e em seguida aplicará as estruturas em um problema clássico de engenharia: construir e avaliar uma árvore binária de expressão a partir de uma entrada em notação pós-fixa. Essa etapa força a integração entre estruturas lineares e árvores, e exige que você trate casos inválidos de forma robusta. A análise final deve indicar custos de tempo e espaço, incluindo custo de travessias na árvore.

**Tarefa**  
1. Implemente `Stack` e `Queue` baseadas em array com capacidade fixa.
   - a. Levante exceções específicas ao tentar `push` em stack cheia ou `pop` em stack vazia;
   - b. Repita para `enqueue` e `dequeue` na fila, suportando wraparound.
2. Construa uma árvore binária de expressão a partir de uma string em pós-fixa.
   - a. Use uma `Stack` para montar a árvore, onde operandos geram nós e operadores combinam subárvores;
   - b. Detecte e trate entradas inválidas (operadores sem operandos suficientes ou sobras ao final).
3. Avalie a expressão e compare abordagens.
   - a. Implemente avaliação por travessia pós-ordem recursiva e por avaliação iterativa com stack;
   - b. Discuta complexidade e riscos (profundidade de recursão, custo de memória).

*Tarefa 1: Stack e Queue com Capacidade Fixa, Exceções Específicas e Wraparound*

In [13]:
# Exercício 5 — Tarefa 1: Stack e Queue baseadas em array com capacidade fixa

# 1.a / 1.b — exceções específicas (em vez de IndexError genérico)
class StructureError(Exception):
    """Base das exceções das estruturas lineares."""

class StackOverflowError(StructureError):
    pass

class StackUnderflowError(StructureError):
    pass

class QueueOverflowError(StructureError):
    pass

class QueueUnderflowError(StructureError):
    pass


class Stack:
    """Pilha LIFO sobre um array de tamanho fixo.
    Invariantes: 0 <= _top <= capacity; posições [0, _top) estão ocupadas;
    posições [_top, capacity) valem None; o topo é _data[_top - 1]."""

    def __init__(self, capacity):
        if not isinstance(capacity, int) or capacity <= 0:
            raise ValueError("capacity deve ser um inteiro positivo")
        self._data = [None] * capacity   # array pré-alocado: nunca cresce
        self._capacity = capacity
        self._top = 0                     # próximo índice livre = tamanho

    def push(self, item):                 # O(1)
        if self._top == self._capacity:
            raise StackOverflowError(f"push({item!r}) em pilha cheia (capacidade={self._capacity})")
        self._data[self._top] = item
        self._top += 1

    def pop(self):                        # O(1)
        if self._top == 0:
            raise StackUnderflowError("pop() em pilha vazia")
        self._top -= 1
        item = self._data[self._top]
        self._data[self._top] = None      # não mantém referência ao item removido
        return item

    def peek(self):                       # O(1)
        if self._top == 0:
            raise StackUnderflowError("peek() em pilha vazia")
        return self._data[self._top - 1]

    def is_empty(self):
        return self._top == 0

    def is_full(self):
        return self._top == self._capacity

    def __len__(self):
        return self._top

    def check_invariants(self):
        assert 0 <= self._top <= self._capacity
        assert all(x is None for x in self._data[self._top:])
        return True

    def __repr__(self):
        return f"Stack(base→topo={self._data[:self._top]}, cap={self._capacity})"


class Queue:
    """Fila FIFO circular sobre array fixo (wraparound por aritmética modular).
    Invariantes: 0 <= _count <= capacity; o primeiro elemento está em _front;
    o próximo livre está em (_front + _count) % capacity."""

    def __init__(self, capacity):
        if not isinstance(capacity, int) or capacity <= 0:
            raise ValueError("capacity deve ser um inteiro positivo")
        self._data = [None] * capacity
        self._capacity = capacity
        self._front = 0
        self._count = 0

    def enqueue(self, item):              # O(1)
        if self._count == self._capacity:
            raise QueueOverflowError(f"enqueue({item!r}) em fila cheia (capacidade={self._capacity})")
        rear = (self._front + self._count) % self._capacity   # wraparound
        self._data[rear] = item
        self._count += 1

    def dequeue(self):                    # O(1): nenhum deslocamento de elementos
        if self._count == 0:
            raise QueueUnderflowError("dequeue() em fila vazia")
        item = self._data[self._front]
        self._data[self._front] = None
        self._front = (self._front + 1) % self._capacity        # wraparound
        self._count -= 1
        return item

    def peek(self):
        if self._count == 0:
            raise QueueUnderflowError("peek() em fila vazia")
        return self._data[self._front]

    def is_empty(self):
        return self._count == 0

    def is_full(self):
        return self._count == self._capacity

    def __len__(self):
        return self._count

    def items(self):
        """Elementos na ordem de saída (frente -> final)."""
        return [self._data[(self._front + i) % self._capacity] for i in range(self._count)]

    def check_invariants(self):
        assert 0 <= self._count <= self._capacity and 0 <= self._front < self._capacity
        ocupados = {(self._front + i) % self._capacity for i in range(self._count)}
        assert all(self._data[i] is None for i in range(self._capacity) if i not in ocupados)  # livres = None
        return True

    def __repr__(self):
        return f"Queue(frente→final={self.items()}, front={self._front}, cap={self._capacity})"


# ---------------- Testes ----------------
def expect(exc_type, fn, *args):
    try:
        fn(*args)
    except exc_type as e:
        return f"{exc_type.__name__}: {e}"
    raise AssertionError(f"esperava {exc_type.__name__}")

# Stack: LIFO, overflow e underflow
s = Stack(3)
for x in [10, 20, 30]:
    s.push(x)
print(s, "| cheia:", s.is_full())
print(" ", expect(StackOverflowError, s.push, 40))
assert [s.pop(), s.pop(), s.pop()] == [30, 20, 10]          # ordem LIFO
print("  pops em ordem LIFO: [30, 20, 10] ✓")
print(" ", expect(StackUnderflowError, s.pop))
print(" ", expect(StackUnderflowError, s.peek))
assert s.check_invariants() and len(s) == 0

# Queue: FIFO, wraparound, overflow e underflow
q = Queue(4)
for x in "ABCD":
    q.enqueue(x)
print("\n", q, sep="")
print(" ", expect(QueueOverflowError, q.enqueue, "E"))
assert q.dequeue() == "A" and q.dequeue() == "B"            # front avança para o índice 2
q.enqueue("E"); q.enqueue("F")                               # E e F ocupam os índices 0 e 1 (wraparound)
print(f"  após 2 dequeues e 2 enqueues -> array interno={q._data}, front={q._front}")
assert q._data[:2] == ["E", "F"] and q.items() == ["C", "D", "E", "F"]
print("  ordem lógica preservada pelo wraparound:", q.items(), "✓")
assert [q.dequeue() for _ in range(4)] == ["C", "D", "E", "F"]
print(" ", expect(QueueUnderflowError, q.dequeue))

# Estresse: operações aleatórias comparadas com um modelo de referência
import random
from collections import deque as _ref_deque
random.seed(5)
q, ref, erros = Queue(8), _ref_deque(), 0
for i in range(20_000):
    if random.random() < 0.5:
        try:
            q.enqueue(i); ref.append(i)
        except QueueOverflowError:
            assert len(ref) == 8; erros += 1
    else:
        try:
            assert q.dequeue() == ref.popleft()
        except QueueUnderflowError:
            assert len(ref) == 0; erros += 1
    assert q.items() == list(ref) and len(q) == len(ref) and q.check_invariants()
print(f"\nEstresse: 20.000 operações aleatórias na fila de capacidade 8 -> conteúdo idêntico ao modelo em todos os passos;"
      f" {erros} overflows/underflows detectados corretamente.")

Stack(base→topo=[10, 20, 30], cap=3) | cheia: True
  StackOverflowError: push(40) em pilha cheia (capacidade=3)
  pops em ordem LIFO: [30, 20, 10] ✓
  StackUnderflowError: pop() em pilha vazia
  StackUnderflowError: peek() em pilha vazia

Queue(frente→final=['A', 'B', 'C', 'D'], front=0, cap=4)
  QueueOverflowError: enqueue('E') em fila cheia (capacidade=4)
  após 2 dequeues e 2 enqueues -> array interno=['E', 'F', 'C', 'D'], front=2
  ordem lógica preservada pelo wraparound: ['C', 'D', 'E', 'F'] ✓
  QueueUnderflowError: dequeue() em fila vazia

Estresse: 20.000 operações aleatórias na fila de capacidade 8 -> conteúdo idêntico ao modelo em todos os passos; 2282 overflows/underflows detectados corretamente.


*Tarefa 2: Construção da Árvore de Expressão a partir da Notação Pós-fixa*

In [14]:
# Exercício 5 — Tarefa 2: árvore binária de expressão a partir de notação pós-fixa

class InvalidExpressionError(Exception):
    """Entrada pós-fixa malformada."""

OPERATORS = {'+', '-', '*', '/', '^'}

class ExprNode:
    __slots__ = ('value', 'left', 'right')
    def __init__(self, value, left=None, right=None):
        self.value, self.left, self.right = value, left, right
    def is_leaf(self):
        return self.left is None and self.right is None

def _parse_operand(token):
    try:
        return float(token) if any(c in token for c in '.eE') else int(token)
    except ValueError:
        raise InvalidExpressionError(f"token inválido: {token!r}") from None

def build_expression_tree(postfix, capacity=None):
    """2.a: operandos viram folhas e são empilhados; operadores desempilham duas
    subárvores (direita primeiro!) e empilham a subárvore combinada.
    Tokens separados por espaço (suporta números com vários dígitos e negativos)."""
    tokens = postfix.split()
    if not tokens:
        raise InvalidExpressionError("expressão vazia")
    stack = Stack(capacity or len(tokens))        # no máximo n operandos empilhados
    for pos, tok in enumerate(tokens):
        if tok in OPERATORS:
            try:
                right = stack.pop()
                left = stack.pop()
            except StackUnderflowError:
                # 2.b: operador sem operandos suficientes
                raise InvalidExpressionError(
                    f"operador {tok!r} na posição {pos} sem dois operandos disponíveis") from None
            stack.push(ExprNode(tok, left, right))
        else:
            try:
                stack.push(ExprNode(_parse_operand(tok)))
            except StackOverflowError as e:
                raise InvalidExpressionError(f"capacidade da pilha excedida: {e}") from None
    if len(stack) != 1:
        # 2.b: sobras ao final (operandos que nenhum operador consumiu)
        raise InvalidExpressionError(
            f"expressão incompleta: restaram {len(stack)} subárvores na pilha (esperado 1)")
    return stack.pop()

def to_infix(node):
    """Travessia in-ordem com parênteses — O(n)."""
    if node.is_leaf():
        return str(node.value)
    return f"({to_infix(node.left)} {node.value} {to_infix(node.right)})"

def tree_height(node):
    return 0 if node is None else 1 + max(tree_height(node.left), tree_height(node.right))

def count_nodes(node):
    return 0 if node is None else 1 + count_nodes(node.left) + count_nodes(node.right)

# Casos válidos
for expr in ["3 4 +", "3 4 + 2 *", "5 1 2 + 4 * + 3 -", "2 3 ^ 10 -4 / +"]:
    t = build_expression_tree(expr)
    print(f"{expr:<22} -> infixa: {to_infix(t):<32} nós={count_nodes(t)} altura={tree_height(t)}")

# Casos inválidos (2.b): todos devem ser rejeitados com mensagem clara
print()
for bad in ["", "3 +", "+", "3 4", "1 2 3 +", "3 4 x +", "3 4 + *", "2 abc +"]:
    try:
        build_expression_tree(bad)
        raise AssertionError(f"deveria rejeitar {bad!r}")
    except InvalidExpressionError as e:
        print(f"rejeitada {bad!r:<12} -> {e}")

3 4 +                  -> infixa: (3 + 4)                          nós=3 altura=2
3 4 + 2 *              -> infixa: ((3 + 4) * 2)                    nós=5 altura=3
5 1 2 + 4 * + 3 -      -> infixa: ((5 + ((1 + 2) * 4)) - 3)        nós=9 altura=5
2 3 ^ 10 -4 / +        -> infixa: ((2 ^ 3) + (10 / -4))            nós=7 altura=3

rejeitada ''           -> expressão vazia
rejeitada '3 +'        -> operador '+' na posição 1 sem dois operandos disponíveis
rejeitada '+'          -> operador '+' na posição 0 sem dois operandos disponíveis
rejeitada '3 4'        -> expressão incompleta: restaram 2 subárvores na pilha (esperado 1)
rejeitada '1 2 3 +'    -> expressão incompleta: restaram 2 subárvores na pilha (esperado 1)
rejeitada '3 4 x +'    -> token inválido: 'x'
rejeitada '3 4 + *'    -> operador '*' na posição 3 sem dois operandos disponíveis
rejeitada '2 abc +'    -> token inválido: 'abc'


*Tarefa 3: Avaliação Recursiva (Pós-ordem) × Avaliação Iterativa com Stack*

In [15]:
# Exercício 5 — Tarefa 3: avaliação recursiva (pós-ordem) x iterativa com Stack
import sys, time

def _apply(op, a, b):
    if op == '+': return a + b
    if op == '-': return a - b
    if op == '*': return a * b
    if op == '/':
        if b == 0:
            raise ZeroDivisionError("divisão por zero na expressão")
        return a / b
    if op == '^': return a ** b
    raise InvalidExpressionError(f"operador desconhecido {op!r}")

def eval_recursive(node, stats=None):
    """3.a: pós-ordem recursiva — avalia esquerda, direita e depois aplica o operador.
    Profundidade de recursão = altura da árvore."""
    if stats is not None:
        stats['visits'] += 1
    if node.is_leaf():
        return node.value
    return _apply(node.value, eval_recursive(node.left, stats), eval_recursive(node.right, stats))

def eval_iterative(root, stats=None):
    """3.a: pós-ordem iterativa com a Stack da Tarefa 1 (sem recursão).
    Pilha de controle guarda (nó, filhos_já_avaliados); pilha de valores guarda resultados parciais."""
    n = count_nodes_iter(root)
    control, values = Stack(n), Stack(n)
    control.push((root, False))
    peak = 1
    while not control.is_empty():
        node, expanded = control.pop()
        if stats is not None:
            stats['visits'] += 1
        if node.is_leaf():
            values.push(node.value)
        elif expanded:                       # filhos já avaliados: combinar
            b, a = values.pop(), values.pop()
            values.push(_apply(node.value, a, b))
        else:                                # empilha o nó de volta e depois os filhos
            control.push((node, True))
            control.push((node.right, False))
            control.push((node.left, False))  # esquerda no topo -> avaliada primeiro
        peak = max(peak, len(control))
    if stats is not None:
        stats['peak_stack'] = peak
    return values.pop()

def height_iter(root):
    """Altura sem recursão (BFS por níveis), para não estourar a pilha em árvores profundas."""
    level, h = [root], 0
    while level:
        h += 1
        level = [c for n in level for c in (n.left, n.right) if c]
    return h

def count_nodes_iter(root):
    st, total = [root], 0
    while st:
        n = st.pop(); total += 1
        if n.left: st.append(n.left)
        if n.right: st.append(n.right)
    return total

# 1) Corretude: as duas abordagens concordam
for expr, esperado in [("3 4 + 2 *", 14), ("5 1 2 + 4 * + 3 -", 14), ("2 3 ^ 10 -4 / +", 5.5), ("7 2 /", 3.5)]:
    t = build_expression_tree(expr)
    r, it = eval_recursive(t), eval_iterative(t)
    assert r == it == esperado, (expr, r, it)
    print(f"{expr:<22} recursiva={r:<5} iterativa={it:<5} ✓")
try:
    eval_iterative(build_expression_tree("1 0 /"))
except ZeroDivisionError as e:
    print("1 0 /                  ->", e)

# 2) Custo: ambas visitam cada nó uma vez -> O(n). Espaço depende da ALTURA h.
def degenerate(n):
    """Expressão ((1+1)+1)+... : árvore degenerada à esquerda, altura ≈ n."""
    return "1 " + " ".join(["1 +"] * n)

def balanced(levels):
    """Árvore completa: altura = levels."""
    expr = "1"
    for _ in range(levels):
        expr = f"{expr} {expr} +"
    return expr

print(f"\nLimite de recursão do Python: {sys.getrecursionlimit()}")
print(f"{'forma':<12}{'nós':>8}{'altura':>8}{'visitas rec.':>14}{'visitas iter.':>15}{'pico pilha iter.':>18}  recursiva")
for nome, expr in [("balanceada", balanced(10)), ("balanceada", balanced(14)),
                   ("degenerada", degenerate(500)), ("degenerada", degenerate(5000))]:
    t = build_expression_tree(expr)
    si = {'visits': 0}
    vi = eval_iterative(t, si)
    sr = {'visits': 0}
    try:
        vr = eval_recursive(t, sr)
        rec, vis = f"ok (={vr})", str(sr['visits'])
    except RecursionError:
        rec, vis = "RecursionError!", "—"
    print(f"{nome:<12}{count_nodes_iter(t):>8}{height_iter(t):>8}"
          f"{vis:>14}{si['visits']:>15}{si['peak_stack']:>18}  {rec} | iterativa={vi}")

3 4 + 2 *              recursiva=14    iterativa=14    ✓
5 1 2 + 4 * + 3 -      recursiva=14    iterativa=14    ✓
2 3 ^ 10 -4 / +        recursiva=5.5   iterativa=5.5   ✓
7 2 /                  recursiva=3.5   iterativa=3.5   ✓
1 0 /                  -> divisão por zero na expressão

Limite de recursão do Python: 1000
forma            nós  altura  visitas rec.  visitas iter.  pico pilha iter.  recursiva
balanceada      2047      11          2047           3070                21  ok (=1024) | iterativa=1024
balanceada     32767      15         32767          49150                29  ok (=16384) | iterativa=16384
degenerada      1001     501          1001           1501              1001  ok (=501) | iterativa=501
degenerada     10001    5001             —          15001             10001  RecursionError! | iterativa=5001


*Análise de Complexidade e Riscos*
- Stack e Queue: `push`, `pop`, `enqueue` e `dequeue` custam $O(1)$, com espaço $O(\text{capacidade})$ pré-alocado.
  - A fila circular avança `front` com aritmética modular, sem deslocar elementos. Uma fila feita com `list.pop(0)` custaria $O(n)$ por remoção.
  - O teste mostra o wraparound acontecendo: depois de dois `dequeue`, os elementos E e F ocuparam os índices 0 e 1 do array, mas a ordem lógica continuou C, D, E, F.
- Invariantes e erros: cada violação tem uma exceção própria (`StackOverflowError`, `StackUnderflowError`, `QueueOverflowError` e `QueueUnderflowError`). No estresse com 20.000 operações, a fila ficou idêntica ao modelo de referência em todos os passos e detectou corretamente 2.282 overflows e underflows.
- Construção da árvore: cada token faz no máximo um `push` e cada operador faz dois `pop`, então a construção custa $O(n)$ em tempo e $O(n)$ em espaço (um nó por token). As entradas inválidas são detectadas assim:
  - operador sem operandos: é o `StackUnderflowError` convertido em `InvalidExpressionError`;
  - sobras ao final: a pilha termina com mais de uma subárvore;
  - tokens inválidos e expressão vazia também são rejeitados.
- Avaliação: as duas abordagens visitam cada nó e custam $O(n)$ em tempo. A iterativa visita os nós internos duas vezes (3.070 visitas para 2.047 nós), o que só muda a constante. As travessias auxiliares (reconstrução infixa com parênteses e altura) também custam $O(n)$.
- Espaço e risco de profundidade: as duas abordagens usam $O(h)$ de memória auxiliar, mas em lugares diferentes. A recursiva usa quadros na pilha de chamadas do Python, limitada a 1.000 níveis; a iterativa usa a `Stack` no heap.
  - Na árvore balanceada ($h = 15$, 32.767 nós), as duas funcionam, e a pilha iterativa chega a 29 posições.
  - Na árvore degenerada ($h = 5.001$), a recursiva falha com `RecursionError`, e a iterativa conclui usando 10.001 posições.
  - Conclusão: a recursão é mais legível, mas a versão iterativa é a escolha segura para entradas grandes ou não confiáveis, como uma expressão vinda de um usuário.

### Exercício 6: Simulador de eventos com múltiplas filas e índice em árvore

**Introdução**  
Sistemas orientados a eventos exigem estruturas que mantenham ordem temporal, suportem inserções e remoções eficientes e preservem rastreabilidade. Filas são essenciais para simular fluxo de atendimento, mas, quando há múltiplas filas e necessidade de auditoria, você precisa também de um índice eficiente para buscar eventos e extrair estatísticas. Neste exercício, você construirá um simulador de checkout baseado em uma string de eventos, com quatro filas independentes, tratamento de overflow e underflow, e logging detalhado. Em seguida, você indexará eventos por identificador em uma BST para permitir buscas rápidas e extração ordenada por chave. A entrega deve demonstrar precisão na simulação e qualidade na modelagem das estruturas, além de análise de custo por operação sob diferentes padrões de entrada.

**Tarefa**  
1. Implemente o simulador com quatro filas A, B, C e D.
   - a. Eventos minúsculos indicam chegada em uma fila, eventos maiúsculos indicam saída dessa fila;
   - b. Qualquer caractere não alfabético deve disparar um snapshot do estado atual de todas as filas.
2. Modele cada cliente com um identificador sequencial por fila e registre logs de operações.
   - a. Em overflow e underflow, registre erro com contexto, mas mantenha a simulação.
3. Indexe os clientes atendidos em uma BST.
   - a. Use a chave como o identificador completo (por exemplo, "B17") e armazene metadados (ordem de atendimento, fila, timestamp lógico);
   - b. Implemente busca e listagem ordenada (in-order) e explique o custo de cada operação em Big O.

*Tarefas 1 e 2: Simulador de Checkout com Quatro Filas, Identificadores Sequenciais e Log*

In [16]:
# Exercício 6 — Tarefas 1 e 2: simulador de checkout com quatro filas (A, B, C, D)
# Reaproveita a Queue circular de capacidade fixa do Exercício 5.

class Customer:
    """Cliente com identificador sequencial POR FILA (A1, A2, ..., B1, ...)."""
    __slots__ = ('cid', 'queue', 'seq', 'arrival_t')
    def __init__(self, queue, seq, arrival_t):
        self.queue, self.seq, self.arrival_t = queue, seq, arrival_t
        self.cid = f"{queue}{seq}"
    def __repr__(self):
        return self.cid


class CheckoutSimulator:
    QUEUES = ('A', 'B', 'C', 'D')

    def __init__(self, capacity=4, on_served=None):
        self.queues = {name: Queue(capacity) for name in self.QUEUES}
        self.next_seq = {name: 1 for name in self.QUEUES}   # contador sequencial por fila
        self.clock = 0              # timestamp lógico: 1 tique por evento processado
        self.service_order = 0      # ordem global de atendimento
        self.log = []               # (t, nível, mensagem)
        self.snapshots = []
        self.on_served = on_served  # callback para indexação (Tarefa 3)
        self.stats = {'chegadas': 0, 'atendimentos': 0, 'overflow': 0, 'underflow': 0, 'ignorados': 0}

    def _log(self, level, msg):
        self.log.append((self.clock, level, msg))

    def process(self, events):
        for ch in events:
            self.clock += 1
            if ch.isalpha():
                name = ch.upper()
                if name not in self.queues:
                    self.stats['ignorados'] += 1
                    self._log('ERRO', f"evento {ch!r}: fila {name} inexistente (válidas: A-D) — ignorado")
                elif ch.islower():
                    self._arrival(name)      # 1.a: minúscula = chegada
                else:
                    self._departure(name)    # 1.a: maiúscula = saída
            else:
                self._snapshot(ch)           # 1.b: não alfabético = snapshot
        return self

    def _arrival(self, name):
        q = self.queues[name]
        cust = Customer(name, self.next_seq[name], self.clock)
        try:
            q.enqueue(cust)
        except QueueOverflowError:
            # 2.a: registra o erro com contexto e segue a simulação.
            # O identificador NÃO é consumido: o cliente recusado não entra no sistema.
            self.stats['overflow'] += 1
            self._log('ERRO', f"OVERFLOW na fila {name}: cliente {cust.cid} recusado "
                              f"(fila cheia {len(q)}/{q._capacity}: {q.items()})")
            return
        self.next_seq[name] += 1
        self.stats['chegadas'] += 1
        self._log('INFO', f"chegada  {cust.cid:<4} -> fila {name} (tamanho {len(q)})")

    def _departure(self, name):
        q = self.queues[name]
        try:
            cust = q.dequeue()
        except QueueUnderflowError:
            self.stats['underflow'] += 1
            self._log('ERRO', f"UNDERFLOW na fila {name}: saída solicitada com a fila vazia")
            return
        self.service_order += 1
        self.stats['atendimentos'] += 1
        wait = self.clock - cust.arrival_t
        self._log('INFO', f"atendido {cust.cid:<4} <- fila {name} (ordem #{self.service_order}, espera={wait} tiques)")
        if self.on_served:
            self.on_served(cust.cid, {'ordem': self.service_order, 'fila': name,
                                      'chegada_t': cust.arrival_t, 'saida_t': self.clock, 'espera': wait})

    def _snapshot(self, trigger):
        state = {name: [c.cid for c in q.items()] for name, q in self.queues.items()}
        self.snapshots.append((self.clock, trigger, state))
        linha = " | ".join(f"{n}: {state[n] if state[n] else '[]'}" for n in self.QUEUES)
        self._log('SNAP', f"snapshot ({trigger!r}) -> {linha}")

    def print_log(self):
        for t, level, msg in self.log:
            print(f"t={t:>3} [{level:<4}] {msg}")


# String de eventos que exercita todos os requisitos
eventos = "aabbc#AAB#aaaaaz!ACCD#bbB.Dd*"
sim = CheckoutSimulator(capacity=4).process(eventos)
print(f"Eventos: {eventos!r} | capacidade por fila = 4\n")
sim.print_log()
print("\nResumo:", sim.stats)

Eventos: 'aabbc#AAB#aaaaaz!ACCD#bbB.Dd*' | capacidade por fila = 4

t=  1 [INFO] chegada  A1   -> fila A (tamanho 1)
t=  2 [INFO] chegada  A2   -> fila A (tamanho 2)
t=  3 [INFO] chegada  B1   -> fila B (tamanho 1)
t=  4 [INFO] chegada  B2   -> fila B (tamanho 2)
t=  5 [INFO] chegada  C1   -> fila C (tamanho 1)
t=  6 [SNAP] snapshot ('#') -> A: ['A1', 'A2'] | B: ['B1', 'B2'] | C: ['C1'] | D: []
t=  7 [INFO] atendido A1   <- fila A (ordem #1, espera=6 tiques)
t=  8 [INFO] atendido A2   <- fila A (ordem #2, espera=6 tiques)
t=  9 [INFO] atendido B1   <- fila B (ordem #3, espera=6 tiques)
t= 10 [SNAP] snapshot ('#') -> A: [] | B: ['B2'] | C: ['C1'] | D: []
t= 11 [INFO] chegada  A3   -> fila A (tamanho 1)
t= 12 [INFO] chegada  A4   -> fila A (tamanho 2)
t= 13 [INFO] chegada  A5   -> fila A (tamanho 3)
t= 14 [INFO] chegada  A6   -> fila A (tamanho 4)
t= 15 [ERRO] OVERFLOW na fila A: cliente A7 recusado (fila cheia 4/4: [A3, A4, A5, A6])
t= 16 [ERRO] evento 'z': fila Z inexistente (válidas: 

*Decisões de Modelagem Evidenciadas pelo Log*
- Semântica dos eventos: uma letra minúscula de `a` a `d` é uma chegada, e a maiúscula correspondente é uma saída. Qualquer caractere não alfabético (`#`, `!`, `.`, `*`) gera um snapshot das quatro filas. Uma letra de fila inexistente (`z`, em t=16) é registrada como erro e ignorada.
- Identificadores: cada fila tem o seu contador (A1, A2, …, B1, …).
  - No overflow (t=15), o cliente A7 é recusado, e o identificador não é consumido. Assim, a numeração dos clientes que realmente entraram continua contígua.
  - O underflow (t=20, 21 e 27) é registrado com o nome da fila vazia.
  - Em nenhum dos dois casos a simulação é interrompida.
- Timestamp lógico: é o índice do evento na string. Ele torna a simulação reprodutível e permite calcular a espera de cada cliente (saída − chegada), registrada nos metadados.
- Custo por evento: chegada e saída custam $O(1)$, com a fila circular do Exercício 5. O snapshot custa $O(\text{clientes em fila})$.

*Tarefa 3: Índice dos Clientes Atendidos em uma BST*

In [17]:
# Exercício 6 — Tarefa 3: índice dos clientes atendidos em uma BST
import math, random

def natural_key(cid):
    """'B17' -> ('B', 17). Comparar strings puras daria 'A10' < 'A2' (ordem lexicográfica),
    o que quebraria a listagem ordenada por identificador."""
    return (cid[0], int(cid[1:]))

class IndexNode:
    __slots__ = ('key', 'meta', 'left', 'right')
    def __init__(self, key, meta):
        self.key, self.meta, self.left, self.right = key, meta, None, None

class ServedIndex:
    """BST cuja chave é o identificador completo do cliente (ex.: 'B17'),
    comparado pela ordem natural (fila, número). Guarda os metadados do atendimento."""

    def __init__(self):
        self.root, self.size = None, 0
        self.insert_comparisons = 0

    def insert(self, cid, meta):                 # O(h)
        k = natural_key(cid)
        if self.root is None:
            self.root = IndexNode(cid, meta); self.size = 1; return
        node = self.root
        while True:
            self.insert_comparisons += 1
            nk = natural_key(node.key)
            if k == nk:
                raise KeyError(f"cliente {cid} já indexado")   # identificadores são únicos
            side = 'left' if k < nk else 'right'
            nxt = getattr(node, side)
            if nxt is None:
                setattr(node, side, IndexNode(cid, meta)); self.size += 1; return
            node = nxt

    def search(self, cid):                       # O(h): devolve (metadados ou None, comparações)
        k, node, comps = natural_key(cid), self.root, 0
        while node:
            comps += 1
            nk = natural_key(node.key)
            if k == nk:
                return node.meta, comps
            node = node.left if k < nk else node.right
        return None, comps

    def inorder(self):                           # O(n): listagem ordenada por identificador
        out, stack, node = [], [], self.root     # in-order iterativo (sem risco de RecursionError)
        while stack or node:
            while node:
                stack.append(node); node = node.left
            node = stack.pop()
            out.append((node.key, node.meta))
            node = node.right
        return out

    def height(self):
        level, h = [self.root] if self.root else [], 0
        while level:
            h += 1
            level = [c for n in level for c in (n.left, n.right) if c]
        return h


# --- Integração: o simulador chama o índice a cada atendimento ---
indice = ServedIndex()
sim = CheckoutSimulator(capacity=4, on_served=indice.insert).process("aabbc#AAB#aaaaaz!ACCD#bbB.Dd*")
print("Listagem in-order do índice (ordem natural de identificador):")
for cid, meta in indice.inorder():
    print(f"  {cid:<4} {meta}")
for alvo in ["B2", "A4", "Z9"]:
    meta, comps = indice.search(alvo)
    print(f"search({alvo!r}) -> {'encontrado' if meta else 'não encontrado':<15} em {comps} comparações")

print("\nPor que ordem natural? sorted(['A2','A10','A1']) =", sorted(['A2', 'A10', 'A1']),
      "| com natural_key:", sorted(['A2', 'A10', 'A1'], key=natural_key))

# --- Custo sob diferentes padrões de entrada ---
def gerar(padrao, n, seed=1):
    rnd = random.Random(seed)
    if padrao == "fila única (A)":        # todos atendidos em A: chaves chegam em ordem crescente
        return "aA" * n
    if padrao == "rodízio A→D":           # chegadas e saídas alternando as quatro filas
        return "".join(f"{q}{q.upper()}" for _ in range(n // 4) for q in "abcd")
    if padrao == "aleatório":             # chegadas e saídas sorteadas
        return "".join(rnd.choice("abcdABCD") for _ in range(4 * n))

print(f"\n{'padrão':<16}{'indexados':>10}{'altura h':>10}{'log2(n)':>9}{'comp. médias/busca':>20}{'comp. inserção total':>22}")
for padrao in ["fila única (A)", "rodízio A→D", "aleatório"]:
    idx = ServedIndex()
    CheckoutSimulator(capacity=50, on_served=idx.insert).process(gerar(padrao, 1500))
    chaves = [k for k, _ in idx.inorder()]
    media = sum(idx.search(k)[1] for k in chaves) / len(chaves)
    print(f"{padrao:<16}{idx.size:>10}{idx.height():>10}{math.log2(idx.size):>9.1f}{media:>20.1f}{idx.insert_comparisons:>22}")

# --- Mitigação: reconstruir o índice balanceado a partir da listagem in-order (O(n)) ---
def rebuild_balanced(index):
    itens = index.inorder()
    novo = ServedIndex()
    pendentes = [(0, len(itens) - 1)]           # insere sempre a mediana de cada faixa (BFS)
    while pendentes:
        lo, hi = pendentes.pop(0)
        if lo > hi:
            continue
        mid = (lo + hi) // 2
        novo.insert(*itens[mid])
        pendentes += [(lo, mid - 1), (mid + 1, hi)]
    return novo

idx = ServedIndex()
CheckoutSimulator(capacity=50, on_served=idx.insert).process(gerar("fila única (A)", 1500))
bal = rebuild_balanced(idx)
chaves = [k for k, _ in bal.inorder()]
media = sum(bal.search(k)[1] for k in chaves) / len(chaves)
print(f"\nfila única reconstruída balanceada: altura {idx.height()} -> {bal.height()}, "
      f"comparações médias por busca {750.5} -> {media:.1f}")

Listagem in-order do índice (ordem natural de identificador):
  A1   {'ordem': 1, 'fila': 'A', 'chegada_t': 1, 'saida_t': 7, 'espera': 6}
  A2   {'ordem': 2, 'fila': 'A', 'chegada_t': 2, 'saida_t': 8, 'espera': 6}
  A3   {'ordem': 4, 'fila': 'A', 'chegada_t': 11, 'saida_t': 18, 'espera': 7}
  B1   {'ordem': 3, 'fila': 'B', 'chegada_t': 3, 'saida_t': 9, 'espera': 6}
  B2   {'ordem': 6, 'fila': 'B', 'chegada_t': 4, 'saida_t': 25, 'espera': 21}
  C1   {'ordem': 5, 'fila': 'C', 'chegada_t': 5, 'saida_t': 19, 'espera': 14}
search('B2') -> encontrado      em 5 comparações
search('A4') -> não encontrado  em 4 comparações
search('Z9') -> não encontrado  em 4 comparações

Por que ordem natural? sorted(['A2','A10','A1']) = ['A1', 'A10', 'A2'] | com natural_key: ['A1', 'A2', 'A10']

padrão           indexados  altura h  log2(n)  comp. médias/busca  comp. inserção total
fila única (A)        1500      1500     10.6               750.5               1124250
rodízio A→D           1500       378     

*Custo de Cada Operação e Efeito do Padrão de Entrada*
- Chave: é o identificador completo (por exemplo, `B17`), comparado pela ordem natural (fila, número). A comparação pura de strings colocaria `A10` antes de `A2`, como a célula demonstra, e quebraria a listagem ordenada.
- Custos: `insert` e `search` custam $O(h)$, e a listagem in-order custa $O(n)$. A in-order foi implementada de forma iterativa para não depender do limite de recursão.
- O padrão de entrada decide $h$. Como cada fila é FIFO, os clientes de uma mesma fila saem em ordem crescente de número, e o índice recebe as chaves quase ordenadas. Com 1.500 atendimentos:

| Padrão | Altura | Comparações médias por busca | $\log_2 n$ |
|---|---|---|---|
| fila única | 1.500 (lista encadeada) | 750,5 | 10,6 |
| rodízio A→D | 378 | 190,2 | 10,6 |
| aleatório (2.901 atendimentos) | 759 | 365,0 | 11,5 |

  - No pior caso, a construção do índice custou 1.124.250 comparações, ou seja, $O(n^2)$, e cada busca custou $O(n)$.
- Mitigação: reconstruir o índice inserindo sempre a mediana (a partir da listagem in-order, em $O(n)$) reduziu a altura de 1.500 para 11 e as comparações por busca de 750,5 para 9,6. Em produção, uma árvore auto-balanceada garantiria $O(\log n)$ sem reconstrução. Uma hashtable daria $O(1)$ para a busca exata, mas perderia a listagem ordenada.

### Exercício 7: Recursão para navegação e representação em árvore de diretórios

**Introdução**  
Recursão é uma técnica inevitável ao lidar com estruturas hierárquicas, como diretórios, ASTs e árvores de decisão. O desafio prático é escrever código correto, evitar explosão de custo, tratar casos extremos e conseguir analisar a complexidade em função da estrutura do problema. Neste exercício, você implementará uma navegação recursiva sobre uma representação de diretórios em memória, sem depender do sistema de arquivos real, para manter reprodutibilidade do notebook. Você também criará uma árvore explícita de diretórios e arquivos, com operações de busca, inserção e deleção, além de uma listagem em profundidade. Para consolidar integração com estruturas lineares, você deverá manter o caminho atual em uma lista encadeada, evidenciando como escolhas estruturais impactam o custo total.

**Tarefa**  
1. Defina uma estrutura de dados em memória que represente diretórios e arquivos (nós com nome, tipo, filhos).
   - a. Implemente `walk(root)` recursivo que retorna uma lista de caminhos completos em pré-ordem.
2. Construa uma Directory Tree com operações:
   - a. `insert(path, node_type)`, `search(path)`, `delete(path)` com validações de consistência;
   - b. Garanta que `delete` trate corretamente nós internos com filhos, definindo e documentando a política adotada.
3. Mantenha o caminho atual durante a travessia usando lista encadeada.
   - a. Implemente `push` e `pop` do "path stack" como lista encadeada e compare com uso de lista Python em termos de operações fundamentais.

*Tarefa 1: Estrutura em Memória e `walk` Recursivo em Pré-ordem*

In [18]:
# Exercício 7 — Tarefa 1: estrutura em memória de diretórios/arquivos e walk recursivo

class FSNode:
    """Nó com nome, tipo ('dir' ou 'file') e filhos.
    Os filhos ficam em um dict nome -> nó: busca de um filho por nome em O(1) médio.
    Arquivos não têm filhos (children = None)."""
    __slots__ = ('name', 'node_type', 'children')
    def __init__(self, name, node_type):
        if node_type not in ('dir', 'file'):
            raise ValueError("node_type deve ser 'dir' ou 'file'")
        self.name, self.node_type = name, node_type
        self.children = {} if node_type == 'dir' else None
    def is_dir(self):
        return self.node_type == 'dir'
    def __repr__(self):
        return f"{self.name}{'/' if self.is_dir() else ''}"

def walk(root):
    """1.a: travessia recursiva em PRÉ-ORDEM (o nó antes dos filhos).
    Devolve a lista de caminhos completos. Filhos visitados em ordem alfabética
    para o resultado ser determinístico."""
    paths = []
    def visit(node, prefix):
        path = '/' if node is root else f"{prefix}/{node.name}".replace('//', '/')
        paths.append(path + ('/' if node.is_dir() and node is not root else ''))
        if node.is_dir():                              # caso base: arquivo (ou dir vazio) não recursa
            for name in sorted(node.children):
                visit(node.children[name], path)
    visit(root, '')
    return paths

# Construção manual (a Tarefa 2 vai oferecer insert com validação)
root = FSNode('/', 'dir')
home = FSNode('home', 'dir'); root.children['home'] = home
joao = FSNode('joao', 'dir'); home.children['joao'] = joao
for f in ['notas.txt', 'at.ipynb']:
    joao.children[f] = FSNode(f, 'file')
etc = FSNode('etc', 'dir'); root.children['etc'] = etc
etc.children['hosts'] = FSNode('hosts', 'file')
root.children['tmp'] = FSNode('tmp', 'dir')            # diretório vazio

for p in walk(root):
    print(p)

/
/etc/
/etc/hosts
/home/
/home/joao/
/home/joao/at.ipynb
/home/joao/notas.txt
/tmp/


*Tarefa 2: Directory Tree com `insert`, `search` e `delete` Validados*

In [19]:
# Exercício 7 — Tarefa 2: Directory Tree com insert, search e delete validados

class PathError(Exception):
    """Caminho inválido ou operação inconsistente com a árvore."""

class DirectoryTree:
    """Política de delete (2.b), documentada:
      - arquivo ou diretório VAZIO: removido diretamente;
      - diretório COM filhos: recusado (PathError), a menos que recursive=True,
        caso em que a subárvore inteira é removida (como `rm -r`) e a quantidade de nós
        removidos é contada recursivamente;
      - a raiz '/' nunca pode ser removida.
    Motivo: evitar apagar uma subárvore inteira por engano; a remoção em massa exige intenção explícita."""

    def __init__(self):
        self.root = FSNode('/', 'dir')
        self.last_visits = 0

    @staticmethod
    def _split(path):
        if not isinstance(path, str) or not path.startswith('/'):
            raise PathError(f"caminho deve ser absoluto (começar com '/'): {path!r}")
        parts = [p for p in path.split('/') if p]
        for p in parts:
            if p in ('.', '..'):
                raise PathError(f"componentes '.' e '..' não são suportados: {path!r}")
        return parts

    def _walk_to(self, parts):
        """Desce componente a componente; devolve o nó ou None. Custo O(d), d = profundidade."""
        node, self.last_visits = self.root, 1
        for p in parts:
            if not node.is_dir() or p not in node.children:
                return None
            node = node.children[p]
            self.last_visits += 1
        return node

    def search(self, path):
        return self._walk_to(self._split(path))

    def insert(self, path, node_type):
        parts = self._split(path)
        if not parts:
            raise PathError("a raiz já existe")
        parent = self._walk_to(parts[:-1])
        if parent is None:
            raise PathError(f"diretório pai inexistente: /{'/'.join(parts[:-1])}")
        if not parent.is_dir():
            raise PathError(f"não é possível criar dentro de um arquivo: /{'/'.join(parts[:-1])}")
        name = parts[-1]
        if name in parent.children:
            raise PathError(f"já existe: {path}")
        node = FSNode(name, node_type)          # valida node_type
        parent.children[name] = node
        return node

    def delete(self, path, recursive=False):
        parts = self._split(path)
        if not parts:
            raise PathError("a raiz não pode ser removida")
        parent = self._walk_to(parts[:-1])
        if parent is None or not parent.is_dir() or parts[-1] not in parent.children:
            raise PathError(f"não encontrado: {path}")
        node = parent.children[parts[-1]]
        if node.is_dir() and node.children and not recursive:
            raise PathError(f"diretório não vazio ({len(node.children)} filhos): {path} — use recursive=True")
        removed = self._count(node)             # recursão: conta a subárvore que sai
        del parent.children[parts[-1]]          # o coletor de lixo libera a subárvore
        return removed

    def _count(self, node):
        if not node.is_dir():
            return 1
        return 1 + sum(self._count(c) for c in node.children.values())

    def walk(self):
        return walk(self.root)


# ---------------- Testes ----------------
fs = DirectoryTree()
for path, kind in [("/home", 'dir'), ("/home/joao", 'dir'), ("/home/joao/at.ipynb", 'file'),
                   ("/home/joao/fotos", 'dir'), ("/home/joao/fotos/praia.png", 'file'),
                   ("/etc", 'dir'), ("/etc/hosts", 'file'), ("/tmp", 'dir')]:
    fs.insert(path, kind)
print("Árvore após inserções:", fs.walk())

no = fs.search("/home/joao/fotos/praia.png")
print(f"\nsearch('/home/joao/fotos/praia.png') -> {no} ({fs.last_visits} nós visitados = profundidade + 1)")
print(f"search('/home/maria') -> {fs.search('/home/maria')}")

print("\nValidações de consistência:")
for fn, args in [(fs.insert, ("/home/joao", 'dir')),               # duplicado
                 (fs.insert, ("/var/log/syslog", 'file')),          # pai inexistente
                 (fs.insert, ("/etc/hosts/x", 'file')),             # pai é arquivo
                 (fs.insert, ("home/x", 'file')),                   # caminho relativo
                 (fs.insert, ("/x", 'link')),                       # tipo inválido
                 (fs.delete, ("/",)),                               # raiz
                 (fs.delete, ("/nada",)),                           # inexistente
                 (fs.delete, ("/home/joao",))]:                     # diretório com filhos
    try:
        fn(*args); raise AssertionError(f"deveria falhar: {args}")
    except (PathError, ValueError) as e:
        print(f"  {fn.__name__}{args} -> {type(e).__name__}: {e}")

print("\nPolítica de delete:")
print("  delete('/etc/hosts')             removeu", fs.delete("/etc/hosts"), "nó (arquivo)")
print("  delete('/tmp')                   removeu", fs.delete("/tmp"), "nó (diretório vazio)")
print("  delete('/home/joao', recursive)  removeu", fs.delete("/home/joao", recursive=True), "nós (subárvore inteira)")
print("Árvore final:", fs.walk())
assert fs.search("/home/joao/fotos/praia.png") is None and fs.search("/home") is not None

Árvore após inserções: ['/', '/etc/', '/etc/hosts', '/home/', '/home/joao/', '/home/joao/at.ipynb', '/home/joao/fotos/', '/home/joao/fotos/praia.png', '/tmp/']

search('/home/joao/fotos/praia.png') -> praia.png (5 nós visitados = profundidade + 1)
search('/home/maria') -> None

Validações de consistência:
  insert('/home/joao', 'dir') -> PathError: já existe: /home/joao
  insert('/var/log/syslog', 'file') -> PathError: diretório pai inexistente: /var/log
  insert('/etc/hosts/x', 'file') -> PathError: não é possível criar dentro de um arquivo: /etc/hosts
  insert('home/x', 'file') -> PathError: caminho deve ser absoluto (começar com '/'): 'home/x'
  insert('/x', 'link') -> ValueError: node_type deve ser 'dir' ou 'file'
  delete('/',) -> PathError: a raiz não pode ser removida
  delete('/nada',) -> PathError: não encontrado: /nada
  delete('/home/joao',) -> PathError: diretório não vazio (2 filhos): /home/joao — use recursive=True

Política de delete:
  delete('/etc/hosts')             r

*Política de Deleção e Custos*
- Política adotada para `delete`:
  - arquivos e diretórios vazios são removidos diretamente;
  - um diretório com filhos é recusado com `PathError`, a menos que `recursive=True`, caso em que a subárvore inteira é removida (como `rm -r`) e a quantidade de nós removidos é devolvida;
  - a raiz nunca pode ser removida.

  O objetivo é que apagar uma subárvore exija uma intenção explícita.
- Validações de consistência: caminho absoluto, pai existente, pai do tipo diretório, nome não duplicado, tipo válido e proibição de `.` e `..`. Todos os casos aparecem testados na saída.
- Custos, com $d$ = profundidade do caminho e filhos em `dict`:
  - `search` e `insert` custam $O(d)$ (o arquivo em profundidade 4 foi encontrado visitando 5 nós);
  - `delete` custa $O(d)$ para localizar, mais $O(s)$ para contar a subárvore de tamanho $s$ quando é recursivo;
  - `walk` custa $O(N)$ nós, mais o custo de montar os caminhos.

  Com filhos em lista, cada nível custaria $O(f)$, com $f$ = número de filhos.

*Tarefa 3: Caminho Atual Mantido em Lista Encadeada (Path Stack)*

In [20]:
# Exercício 7 — Tarefa 3: caminho atual mantido em lista encadeada ("path stack")
import random, time

class _PNode:
    __slots__ = ('name', 'next')
    def __init__(self, name, nxt):
        self.name, self.next = name, nxt

class LinkedPathStack:
    """Pilha como lista simplesmente encadeada: o topo (último componente do caminho) é a CABEÇA.
    push/pop mexem só na cabeça -> O(1) real, sem realocação.
    Contadores registram as operações fundamentais."""
    def __init__(self):
        self.head, self.depth = None, 0
        self.ops = {'alocações': 0, 'atribuições de ponteiro': 0, 'visitas p/ montar caminho': 0, 'cópias na inversão': 0}

    def push(self, name):
        self.head = _PNode(name, self.head)
        self.depth += 1
        self.ops['alocações'] += 1
        self.ops['atribuições de ponteiro'] += 2          # novo.next = head ; head = novo

    def pop(self):
        if self.head is None:
            raise IndexError("pop em path stack vazia")
        name, self.head = self.head.name, self.head.next
        self.depth -= 1
        self.ops['atribuições de ponteiro'] += 1          # head = head.next
        return name

    def current_path(self):
        """A cabeça é o ÚLTIMO componente: percorre O(d) nós e inverte para montar '/a/b/c'."""
        parts, node = [], self.head
        while node:
            parts.append(node.name); node = node.next
            self.ops['visitas p/ montar caminho'] += 1
        self.ops['cópias na inversão'] += len(parts)      # custo extra: inverter a ordem
        return '/' + '/'.join(reversed(parts))

class ListPathStack:
    """Mesma interface usando list do Python (array dinâmico): append/pop no FINAL, O(1) amortizado."""
    def __init__(self):
        self.items = []
        self.ops = {'escritas no array': 0, 'realocações do array': 0, 'visitas p/ montar caminho': 0}
        self._cap = 0
    def push(self, name):
        self.items.append(name)
        self.ops['escritas no array'] += 1
        if len(self.items) > self._cap:                   # modela o crescimento geométrico do CPython
            self.ops['realocações do array'] += 1
            self._cap = max(4, int(self._cap * 1.125) + 6)
    def pop(self):
        return self.items.pop()
    def current_path(self):
        self.ops['visitas p/ montar caminho'] += len(self.items)
        return '/' + '/'.join(self.items)                 # já está na ordem raiz -> folha

def walk_with_stack(root, stack):
    """Travessia recursiva em pré-ordem que NÃO concatena strings ao descer:
    o caminho atual vive na path stack (push ao entrar num diretório, pop ao sair)."""
    paths = []
    def visit(node):
        if node is not root:
            stack.push(node.name)
        paths.append(stack.current_path() if node is not root else '/')
        if node.is_dir():
            for name in sorted(node.children):
                visit(node.children[name])
        if node is not root:
            stack.pop()
    visit(root)
    return paths

# Corretude: os três métodos produzem os mesmos caminhos
fs = DirectoryTree()
for p, k in [("/home", 'dir'), ("/home/joao", 'dir'), ("/home/joao/at.ipynb", 'file'),
             ("/home/joao/src", 'dir'), ("/home/joao/src/main.py", 'file'), ("/etc", 'dir')]:
    fs.insert(p, k)
ll, pl = LinkedPathStack(), ListPathStack()
ref = [p.rstrip('/') or '/' for p in fs.walk()]
assert walk_with_stack(fs.root, ll) == ref == walk_with_stack(fs.root, pl)
print("Caminhos (lista encadeada):", walk_with_stack(fs.root, LinkedPathStack()))
assert ll.depth == 0 and len(pl.items) == 0
print("Pilhas vazias ao final da travessia (todo push teve seu pop) ✓")

# Experimento: árvores com formatos diferentes
def build_tree(fanout, depth):
    t = DirectoryTree()
    def rec(prefix, d):
        if d == depth:
            return
        for i in range(fanout):
            p = f"{prefix}/d{i}"
            t.insert(p, 'dir'); rec(p, d + 1)
    rec('', 0)
    return t

print(f"\n{'árvore':<22}{'nós':>7}{'implementação':>16}{'push/pop':>10}  operações fundamentais")
for nome, fan, dep in [("larga (fan=8, d=4)", 8, 4), ("profunda (fan=1, d=400)", 1, 400)]:
    t = build_tree(fan, dep)
    for impl in (LinkedPathStack(), ListPathStack()):
        n_paths = len(walk_with_stack(t.root, impl))
        print(f"{nome:<22}{n_paths:>7}{type(impl).__name__:>16}{n_paths-1:>10}  {impl.ops}")

# Risco da recursão: profundidade da pilha de chamadas = altura da árvore
import sys
def walk_iterative(root):
    """Mesma saída de walk (pré-ordem), mas com pilha explícita: sem limite de profundidade."""
    paths, stack = [], [(root, '')]
    while stack:
        node, parent = stack.pop()
        path = '/' if node is root else f"{parent}/{node.name}".replace('//', '/')
        paths.append(path + ('/' if node.is_dir() and node is not root else ''))
        if node.is_dir():
            for name in sorted(node.children, reverse=True):   # reverso: o menor sai primeiro
                stack.append((node.children[name], path))
    return paths

assert walk_iterative(fs.root) == walk(fs.root)
print(f"\nLimite de recursão do Python: {sys.getrecursionlimit()} quadros")
for d in [500, 2000]:
    raiz = FSNode('/', 'dir'); cur = raiz
    for i in range(d):                                   # cadeia /d0/d1/.../d{d-1}
        novo = FSNode(f'd{i}', 'dir'); cur.children[novo.name] = novo; cur = novo
    try:
        n_rec = len(walk(raiz)); rec = f"ok ({n_rec} caminhos)"
    except RecursionError:
        rec = "RecursionError"
    print(f"cadeia com profundidade {d}: walk recursivo -> {rec} | walk iterativo -> ok ({len(walk_iterative(raiz))} caminhos)")

Caminhos (lista encadeada): ['/', '/etc', '/home', '/home/joao', '/home/joao/at.ipynb', '/home/joao/src', '/home/joao/src/main.py']
Pilhas vazias ao final da travessia (todo push teve seu pop) ✓

árvore                    nós   implementação  push/pop  operações fundamentais
larga (fan=8, d=4)       4681 LinkedPathStack      4680  {'alocações': 4680, 'atribuições de ponteiro': 14040, 'visitas p/ montar caminho': 18056, 'cópias na inversão': 18056}
larga (fan=8, d=4)       4681   ListPathStack      4680  {'escritas no array': 4680, 'realocações do array': 1, 'visitas p/ montar caminho': 18056}
profunda (fan=1, d=400)    401 LinkedPathStack       400  {'alocações': 400, 'atribuições de ponteiro': 1200, 'visitas p/ montar caminho': 80200, 'cópias na inversão': 80200}
profunda (fan=1, d=400)    401   ListPathStack       400  {'escritas no array': 400, 'realocações do array': 20, 'visitas p/ montar caminho': 80200}

Limite de recursão do Python: 1000 quadros
cadeia com profundidade 500: wal

*Lista Encadeada × Lista Python em Operações Fundamentais*
- `push` e `pop` custam $O(1)$ nas duas implementações. A lista encadeada faz uma alocação de nó e 2 atribuições de ponteiro por `push`, e 1 por `pop`, sem nunca realocar. A lista Python grava no array e realoca raramente, com crescimento geométrico: 1 realocação na árvore larga e 20 na profunda.
- Montar o caminho custa $O(d)$ nas duas, mas a lista encadeada tem a cabeça no último componente. Por isso, ela precisa de uma inversão extra: 18.056 cópias a mais na árvore larga e 80.200 na profunda. A lista Python já está na ordem raiz → folha.
- Custo total da travessia: $O\left(\sum_{v} \text{profundidade}(v)\right)$. Na árvore profunda (uma cadeia com $d = 400$), isso dá 80.200 visitas, ou seja, $\approx d^2/2$: a montagem dos caminhos domina, e não as operações de pilha.
- Custo da recursão e risco de profundidade: o `walk` recursivo usa um quadro na pilha de chamadas por nível, então o espaço auxiliar é $O(d)$, com $d$ = altura da árvore. O tempo é $O(N)$ visitas mais a montagem dos caminhos. Como o Python limita a recursão a 1.000 quadros, a profundidade é um risco real:
  - uma cadeia de profundidade 500 funcionou;
  - uma de profundidade 2.000 falhou com `RecursionError`.

  A versão `walk_iterative`, com pilha explícita no heap, produziu a mesma saída nos dois casos (verificado por `assert`). Para hierarquias de origem desconhecida, a versão iterativa é a escolha segura.
- Conclusão técnica: para este uso, a lista Python é mais eficiente, porque é contígua e está na ordem natural. A lista encadeada ganha quando os caminhos precisam ser guardados, porque filhos podem compartilhar o nó do prefixo do pai (compartilhamento estrutural), com `push` $O(1)$ sem copiar o caminho inteiro.

### Exercício 8: Programação dinâmica e memoization com hashtable e árvore de subproblemas

**Introdução**  
Programação dinâmica é uma resposta de engenharia para recursos que repetem subproblemas e explodem em tempo. Em projetos reais, você frequentemente começa com uma solução recursiva correta e, ao identificar repetição, aplica memoization para reduzir complexidade. Neste exercício, você implementará um problema clássico com forte sobreposição de subproblemas, instrumentará chamadas recursivas, e em seguida aplicará memoization usando uma hashtable implementada por você. O objetivo é demonstrar redução quantitativa de custo e a capacidade de justificar a complexidade resultante. Para integrar estruturas hierárquicas, você também construirá uma árvore explícita de dependência de subproblemas em uma execução, permitindo visualizar e validar a estrutura recursiva. A entrega deve ser técnica, baseada em evidências, e robusta para diferentes entradas.

**Tarefa**  
1. Implemente uma solução recursiva para um problema de otimização, como `knapsack(weights, capacity)` ou `min_coins(amount, coins)`.
   - a. Conte o número de chamadas recursivas e o número de subproblemas distintos.
2. Aplique memoization usando sua estrutura `HashTableChained` ou equivalente.
   - a. Mostre a redução de chamadas recursivas e discuta a mudança de complexidade em Big O;
   - b. Garanta que a chave do memo seja bem definida (estado mínimo suficiente).
3. Construa uma árvore de subproblemas visitados em uma execução.
   - a. Cada nó deve representar um estado, com filhos representando transições recursivas;
   - b. Implemente uma busca na árvore por um estado específico e discuta o custo dessa operação.

*Tarefas 1 e 2: `min_coins` Recursivo e com Memoization em `HashTableChained`*

In [21]:
# Exercício 8 — Tarefas 1 e 2: min_coins recursivo, instrumentado, e com memoization em HashTableChained
INF = float('inf')

def min_coins_naive(amount, coins, stats):
    """1: recursão direta. min_coins(v) = 1 + min(min_coins(v - c)) para cada moeda c <= v.
    Casos base: v == 0 -> 0 moedas; nenhuma moeda cabe -> inf (impossível)."""
    stats['calls'] += 1
    stats['distinct'].add(amount)                 # 1.a: subproblemas distintos (estado = valor restante)
    if amount == 0:
        return 0
    best = INF
    for c in coins:
        if c <= amount:
            best = min(best, 1 + min_coins_naive(amount - c, coins, stats))
    return best

def min_coins_memo(amount, coins, memo, stats):
    """2: mesma recorrência, mas cada estado é resolvido uma única vez.
    2.b: a chave do memo é só `amount`: dentro de UMA execução as moedas são fixas,
    então o valor restante é o estado mínimo suficiente. Se o memo fosse reaproveitado
    entre conjuntos de moedas diferentes, a chave precisaria ser (amount, tuple(coins))."""
    stats['calls'] += 1
    cached = memo.get(amount)
    stats['bucket_comparisons'] += memo.last_op_comparisons
    if cached is not None:
        stats['hits'] += 1
        return cached
    if amount == 0:
        result = 0
    else:
        result = INF
        for c in coins:
            if c <= amount:
                result = min(result, 1 + min_coins_memo(amount - c, coins, memo, stats))
    memo.put(amount, result)                      # 0 e inf também são memorizados
    return result

coins = [1, 3, 4]
print(f"Moedas = {coins}")
print(f"{'valor':>6}{'resposta':>9}{'chamadas ingênuas':>19}{'subproblemas distintos':>24}"
      f"{'chamadas memo':>15}{'acertos memo':>14}{'comp. buckets':>15}{'redução':>10}")
for amount in [5, 10, 15, 20, 25, 30]:
    s1 = {'calls': 0, 'distinct': set()}
    r1 = min_coins_naive(amount, coins, s1)
    memo = HashTableChained(initial_capacity=8)
    s2 = {'calls': 0, 'hits': 0, 'bucket_comparisons': 0}
    r2 = min_coins_memo(amount, coins, memo, s2)
    assert r1 == r2
    print(f"{amount:>6}{r1:>9}{s1['calls']:>19}{len(s1['distinct']):>24}{s2['calls']:>15}"
          f"{s2['hits']:>14}{s2['bucket_comparisons']:>15}{s1['calls']/s2['calls']:>9.1f}x")

# Valores maiores: só a versão com memo é viável
for amount in [100, 500]:
    memo = HashTableChained(initial_capacity=8)
    s2 = {'calls': 0, 'hits': 0, 'bucket_comparisons': 0}
    r = min_coins_memo(amount, coins, memo, s2)
    print(f"memo: valor={amount} -> {r} moedas | chamadas={s2['calls']} (≈ k·(valor+1) = {len(coins)*(amount+1)+1})"
          f" | estados no memo={len(memo)} | capacidade final da hashtable={memo.capacity}")

# Caso sem solução e corretude contra uma solução bottom-up independente
def min_coins_table(amount, coins):
    dp = [0] + [INF] * amount
    for v in range(1, amount + 1):
        dp[v] = min((dp[v - c] + 1 for c in coins if c <= v), default=INF)
    return dp[amount]
for amt, cs in [(7, [2, 4]), (11, [1, 2, 5]), (63, [1, 5, 10, 21, 25]), (6, [1, 3, 4])]:
    m = HashTableChained()
    r = min_coins_memo(amt, cs, m, {'calls': 0, 'hits': 0, 'bucket_comparisons': 0})
    assert r == min_coins_table(amt, cs)
    print(f"min_coins({amt}, {cs}) = {r}" + ("  (impossível)" if r == INF else ""))

Moedas = [1, 3, 4]
 valor resposta  chamadas ingênuas  subproblemas distintos  chamadas memo  acertos memo  comp. buckets   redução
     5        2                 15                       6             11             5              5      1.4x
    10        3                168                      11             26            15             15      6.5x
    15        4               1869                      16             41            25             25     45.6x
    20        5              20736                      21             56            35             35    370.3x
    25        7             229970                      26             71            45             45   3239.0x
    30        8            2550408                      31             86            55             55  29655.9x
memo: valor=100 -> 25 moedas | chamadas=296 (≈ k·(valor+1) = 304) | estados no memo=101 | capacidade final da hashtable=256
memo: valor=500 -> 125 moedas | chamadas=1496 (≈ k·(valor+1) = 150

*Redução de Chamadas e Mudança de Complexidade*
- Recursão ingênua: o número de chamadas cresce exponencialmente. Com as moedas `[1, 3, 4]`, foram 168 chamadas para o valor 10, 20.736 para 20 e 2.550.408 para 30 (cerca de ×11 a cada +5). Mesmo assim, existem apenas 31 subproblemas distintos para o valor 30: a explosão vem de resolver os mesmos estados repetidamente. O custo é $O(k^{A/c_{min}})$ no pior caso.
- Com memoization na `HashTableChained` do Exercício 4, cada estado é resolvido uma única vez. As chamadas caem para 86 no valor 30 (29.656× menos), e o custo passa a $O(A \cdot k)$ em tempo e $O(A)$ em espaço.
  - As chamadas medidas batem com a fórmula: 296 contra $k(A+1)+1 = 304$ para $A = 100$.
  - O memo guardou 101 estados, e a tabela foi redimensionada até 256 buckets, mantendo o fator de carga $\le 0{,}75$.
  - Cada acerto no memo custou 1 comparação no bucket (a coluna de comparações é igual à de acertos), ou seja, $O(1)$ esperado.
- Chave do memo (2.b): a chave é apenas `amount`, porque dentro de uma execução as moedas são fixas, e o valor restante é o estado mínimo suficiente. Se o mesmo memo fosse reaproveitado com outro conjunto de moedas, a chave precisaria ser `(amount, tuple(coins))`, senão respostas de problemas diferentes se misturariam. Os valores 0 e $\infty$ (sem solução) também são memorizados.
- Corretude: os resultados foram conferidos contra uma solução bottom-up independente, inclusive no caso sem solução `min_coins(7, [2, 4]) = inf`.
- Risco: a profundidade da recursão é $A / c_{min}$ (500 níveis para $A = 500$). Para valores maiores, a versão bottom-up evita o limite de recursão.

*Tarefa 3: Árvore de Subproblemas Visitados em uma Execução*

In [22]:
# Exercício 8 — Tarefa 3: árvore de subproblemas visitados em UMA execução

class SubproblemNode:
    """3.a: cada nó é um estado (valor restante); cada filho é uma transição (usar a moeda c).
    `origem` indica como o estado foi resolvido: 'base', 'calculado' ou 'memo' (reaproveitado)."""
    __slots__ = ('state', 'coin', 'result', 'origem', 'children')
    def __init__(self, state, coin):
        self.state, self.coin, self.result, self.origem, self.children = state, coin, None, None, []

def min_coins_traced(amount, coins, use_memo=True):
    """Executa min_coins registrando a árvore de chamadas; devolve (resposta, raiz)."""
    memo = HashTableChained() if use_memo else None
    def solve(v, coin):
        node = SubproblemNode(v, coin)
        if use_memo and memo.get(v) is not None:
            node.result, node.origem = memo.get(v), 'memo'   # folha: estado já resolvido antes
            return node
        if v == 0:
            node.result, node.origem = 0, 'base'
        else:
            best = INF
            for c in coins:
                if c <= v:
                    child = solve(v - c, c)
                    node.children.append(child)
                    best = min(best, 1 + child.result)
            node.result, node.origem = best, 'calculado'
        if use_memo:
            memo.put(v, node.result)
        return node
    root = solve(amount, None)
    return root.result, root

def print_tree(node, prefix="", last=True):
    label = f"{node.state}" + (f"  (moeda {node.coin})" if node.coin else "  (raiz)")
    label += f" -> {node.result}  [{node.origem}]"
    print(prefix + ("└── " if last else "├── ") + label)
    for i, ch in enumerate(node.children):
        print_tree(ch, prefix + ("    " if last else "│   "), i == len(node.children) - 1)

def tree_size(node):
    return 1 + sum(tree_size(c) for c in node.children)

def find_state(root, target):
    """3.b: busca em profundidade por um estado específico.
    A árvore NÃO é ordenada pela chave (filhos seguem a ordem das moedas),
    então no pior caso é preciso visitar todos os nós: O(N), N = nós da árvore."""
    visited, stack = 0, [root]
    while stack:
        node = stack.pop(); visited += 1
        if node.state == target:
            return node, visited
        stack.extend(reversed(node.children))
    return None, visited

res, raiz = min_coins_traced(6, [1, 3, 4], use_memo=True)
print(f"Árvore de subproblemas de min_coins(6, [1, 3, 4]) COM memo (resposta = {res}):")
print_tree(raiz)

_, raiz_naive = min_coins_traced(6, [1, 3, 4], use_memo=False)
print(f"\nNós na árvore: sem memo = {tree_size(raiz_naive)} | com memo = {tree_size(raiz)}")

print("\nBusca por estado na árvore (DFS):")
for alvo in [3, 0, 99]:
    no, vis = find_state(raiz, alvo)
    print(f"  estado {alvo:>2}: {'encontrado' if no else 'ausente':<10} visitando {vis} de {tree_size(raiz)} nós")

# Custo da busca cresce com o tamanho da árvore
for amount in [10, 20, 40]:
    _, r = min_coins_traced(amount, [1, 3, 4], use_memo=True)
    _, vis = find_state(r, -1)                      # estado inexistente = pior caso
    print(f"  min_coins({amount}): árvore com {tree_size(r)} nós -> busca malsucedida visitou {vis}")

Árvore de subproblemas de min_coins(6, [1, 3, 4]) COM memo (resposta = 2):
└── 6  (raiz) -> 2  [calculado]
    ├── 5  (moeda 1) -> 2  [calculado]
    │   ├── 4  (moeda 1) -> 1  [calculado]
    │   │   ├── 3  (moeda 1) -> 1  [calculado]
    │   │   │   ├── 2  (moeda 1) -> 2  [calculado]
    │   │   │   │   └── 1  (moeda 1) -> 1  [calculado]
    │   │   │   │       └── 0  (moeda 1) -> 0  [base]
    │   │   │   └── 0  (moeda 3) -> 0  [memo]
    │   │   ├── 1  (moeda 3) -> 1  [memo]
    │   │   └── 0  (moeda 4) -> 0  [memo]
    │   ├── 2  (moeda 3) -> 2  [memo]
    │   └── 1  (moeda 4) -> 1  [memo]
    ├── 3  (moeda 3) -> 1  [memo]
    └── 2  (moeda 4) -> 2  [memo]

Nós na árvore: sem memo = 24 | com memo = 14

Busca por estado na árvore (DFS):
  estado  3: encontrado visitando 4 de 14 nós
  estado  0: encontrado visitando 7 de 14 nós
  estado 99: ausente    visitando 14 de 14 nós
  min_coins(10): árvore com 26 nós -> busca malsucedida visitou 26
  min_coins(20): árvore com 56 nós -> busca

*Estrutura da Árvore e Custo da Busca*
- Cada nó é um estado (o valor restante), e cada aresta é a transição "usar a moeda $c$". As folhas marcadas como `[memo]` são estados reaproveitados. É nelas que a memoization corta a árvore: para `min_coins(6)`, ela tem 14 nós, contra 24 na execução sem memo. A diferença cresce exponencialmente com o valor, como a Tarefa 2 mostrou.
- Busca por estado: a árvore não é ordenada pela chave (os filhos seguem a ordem das moedas), então a busca em profundidade precisa, no pior caso, visitar todos os nós. O custo é $O(N)$, com $N \approx k \cdot A$ nós quando há memo.
  - Evidência: buscas malsucedidas visitaram 26, 56 e 116 nós para $A$ = 10, 20 e 40, crescendo linearmente.
  - Para busca $O(1)$ por estado, bastaria indexar os nós da árvore em uma hashtable `estado → nó` durante a construção.

### Exercício 9: QuickSort e QuickSelect com contagem de operações e variante em lista encadeada

**Introdução**  
QuickSort e QuickSelect são algoritmos que aparecem em bibliotecas e sistemas de alto desempenho, mas o domínio esperado aqui é entender como decisões de partição e casos de entrada impactam o custo real. Um engenheiro deve ser capaz de quantificar cópias e comparações, reconhecer piores casos e aplicar estratégias para reduzir custo médio. Neste exercício, você implementará QuickSort com instrumentação e uma estratégia de "subarray curto" que troca para Insertion Sort, além de QuickSelect para seleção sem ordenação total. Você também implementará uma variante de partição que opera sobre lista encadeada, para expor as consequências de trabalhar sem acesso aleatório. Ao final, você deve produzir uma análise comparativa consistente, conectando resultados de instrumentação e complexidade assintótica.

**Tarefa**  
1. Implemente `quicksort(arr, short)` com contagem de cópias e comparações.
   - a. Use Insertion Sort para subarrays com tamanho menor ou igual a `short`;
   - b. Execute em quatro padrões de entrada e compare resultados para pelo menos três valores de `short`.
2. Implemente `quickselect(arr, goal_index)` retornando o elemento que ocuparia `goal_index` se o array fosse ordenado.
   - a. Retorne também cópias e comparações e compare com ordenar todo o array.
3. Variante em lista encadeada:
   - a. Implemente uma partição em `SinglyLinkedList` que separe nós menores, iguais e maiores que um pivô, preservando estabilidade quando possível;
   - b. Discuta por que o custo de partição muda sem acesso por índice e relacione com Big O.

*Tarefa 1: QuickSort Instrumentado com Corte para Insertion Sort*

In [23]:
# Exercício 9 — Tarefa 1: QuickSort instrumentado com corte para Insertion Sort em subarrays curtos
import random

class OpCounter:
    """Contagem com as mesmas convenções do Exercício 2: comparação = comparação de chaves;
    troca = 3 cópias; deslocamento/gravação do Insertion Sort = 1 cópia."""
    __slots__ = ('comparisons', 'copies')
    def __init__(self):
        self.comparisons = self.copies = 0
    def swap(self, a, i, j):
        if i != j:
            a[i], a[j] = a[j], a[i]
            self.copies += 3

def _insertion_range(a, lo, hi, c):
    for i in range(lo + 1, hi + 1):
        key = a[i]; c.copies += 1
        j = i - 1
        while j >= lo:
            c.comparisons += 1
            if a[j] > key:
                a[j + 1] = a[j]; c.copies += 1; j -= 1
            else:
                break
        a[j + 1] = key; c.copies += 1

def _median_of_three(a, lo, hi, c):
    """Ordena a[lo], a[mid], a[hi] e usa a mediana como pivô: evita o pior caso O(n²)
    em entradas ordenadas/reversas, que derrubam a escolha ingênua 'último elemento'."""
    mid = (lo + hi) // 2
    c.comparisons += 1
    if a[mid] < a[lo]: c.swap(a, lo, mid)
    c.comparisons += 1
    if a[hi] < a[lo]: c.swap(a, lo, hi)
    c.comparisons += 1
    if a[hi] < a[mid]: c.swap(a, mid, hi)
    c.swap(a, mid, hi)                      # pivô vai para o fim (formato Lomuto)

def _partition(a, lo, hi, c, pivot_rule):
    if pivot_rule == 'median3' and hi - lo >= 2:
        _median_of_three(a, lo, hi, c)
    pivot = a[hi]
    i = lo
    for j in range(lo, hi):
        c.comparisons += 1
        if a[j] < pivot:
            c.swap(a, i, j); i += 1
    c.swap(a, i, hi)
    return i

def quicksort(arr, short, pivot_rule='median3'):
    """Devolve (lista_ordenada, comparações, cópias). Não altera a entrada.
    short: subarrays com tamanho <= short vão para o Insertion Sort (1.a).
    Recursão só no lado MENOR e laço no maior: profundidade O(log n) mesmo no pior caso."""
    a, c = list(arr), OpCounter()
    def sort(lo, hi):
        while lo < hi:
            if hi - lo + 1 <= short:
                _insertion_range(a, lo, hi, c); return
            p = _partition(a, lo, hi, c, pivot_rule)
            if p - lo < hi - p:
                sort(lo, p - 1); lo = p + 1
            else:
                sort(p + 1, hi); hi = p - 1
    sort(0, len(a) - 1)
    return a, c.comparisons, c.copies

def gerar_padrao(n, padrao, seed=42):
    rnd = random.Random(seed)
    if padrao == 'ordenado':  return list(range(n))
    if padrao == 'reverso':   return list(range(n, 0, -1))
    if padrao == 'quase ordenado':
        a = list(range(n))
        for _ in range(max(1, n // 20) if n > 1 else 0):   # ~5% de trocas, como no Exercício 2
            i, j = rnd.randrange(n), rnd.randrange(n); a[i], a[j] = a[j], a[i]
        return a
    if padrao == 'aleatório':
        a = list(range(n)); rnd.shuffle(a); return a

# Corretude em vários tamanhos, padrões, valores de short e com duplicatas
for n in [0, 1, 2, 3, 10, 257]:
    for padrao in ['ordenado', 'reverso', 'quase ordenado', 'aleatório']:
        for short in [1, 8, 32]:
            d = gerar_padrao(n, padrao)
            assert quicksort(d, short)[0] == sorted(d)
dups = [random.Random(1).randint(0, 5) for _ in range(500)]
assert quicksort(dups, 8)[0] == sorted(dups)
print("Corretude verificada (inclui listas vazias, unitárias e com muitas duplicatas) ✓\n")

# 1.b: quatro padrões x cinco valores de short
N = 5000
SHORTS = [1, 8, 16, 32, 64]          # short = 1 -> QuickSort puro
linhas = []
print(f"n = {N} | pivô = mediana de três | células: comparações / cópias")
print(f"{'padrão':<16}" + "".join(f"{'short='+str(s):>20}" for s in SHORTS))
for padrao in ['ordenado', 'reverso', 'quase ordenado', 'aleatório']:
    d = gerar_padrao(N, padrao)
    cel = []
    for s in SHORTS:
        _, cmp_, cp = quicksort(d, s)
        cel.append(f"{cmp_:,} / {cp:,}".replace(',', '.'))
        linhas.append((padrao, s, cmp_, cp))
    print(f"{padrao:<16}" + "".join(f"{x:>20}" for x in cel))

# Por que mediana de três? Mesma entrada ordenada com pivô ingênuo (último elemento)
print()
for n in [1000, 2000, 4000]:
    d = list(range(n))
    _, c_last, _ = quicksort(d, 1, pivot_rule='last')
    _, c_med, _ = quicksort(d, 1, pivot_rule='median3')
    print(f"ordenado n={n}: pivô 'último' = {c_last:>9,} comparações (≈ n²/2 = {n*n//2:,}) | mediana de três = {c_med:,}".replace(',', '.'))

Corretude verificada (inclui listas vazias, unitárias e com muitas duplicatas) ✓

n = 5000 | pivô = mediana de três | células: comparações / cópias
padrão                       short=1             short=8            short=16            short=32            short=64
ordenado             57.963 / 12.282     53.034 / 11.806     49.497 / 11.020     44.752 / 10.508     39.879 / 10.252
reverso            104.823 / 133.770   104.398 / 130.956   115.941 / 140.156   133.753 / 156.614   152.652 / 174.460
quase ordenado       82.028 / 71.676     78.427 / 67.632     76.210 / 65.600     75.089 / 65.018     77.131 / 68.784
aleatório           66.249 / 100.812     64.195 / 95.566     66.417 / 95.942    75.257 / 102.232    96.225 / 120.460

ordenado n=1000: pivô 'último' =   499.500 comparações (≈ n²/2 = 500.000) | mediana de três = 9.451
ordenado n=2000: pivô 'último' = 1.999.000 comparações (≈ n²/2 = 2.000.000) | mediana de três = 20.892
ordenado n=4000: pivô 'último' = 7.998.000 comparações (≈ n²/2 

*Efeito de `short` e do Padrão de Entrada*
- Convenções de contagem: as mesmas do Exercício 2 (comparação de chaves; troca = 3 cópias). Duas decisões reduzem o custo médio e o risco:
  - pivô pela mediana de três;
  - recursão apenas no lado menor, o que limita a profundidade a $O(\log n)$.
- Aleatório: o melhor ponto foi `short = 8` (64.195 comparações e 95.566 cópias, contra 66.249 e 100.812 no QuickSort puro). Acima de 16 o custo volta a subir (96.225 comparações com `short = 64`), porque o Insertion Sort é $O(s^2)$ em cada bloco de tamanho $s$ fora de ordem.
- Ordenado: quanto maior `short`, melhor (57.963 → 39.879 comparações), porque o Insertion Sort é $O(s)$ em blocos já ordenados.
- Reverso: quanto maior `short`, pior (104.823 → 152.652), porque blocos invertidos são o pior caso do Insertion Sort.
- Quase ordenado: o ótimo ficou perto de `short = 32`.
- Conclusão: `short` entre 8 e 16 é o melhor compromisso quando o padrão de entrada é desconhecido.
- Pivô: com o pivô ingênuo "último elemento", a entrada ordenada vira o pior caso $O(n^2)$ (499.500 comparações para $n = 1.000$ e 7.998.000 para $n = 4.000$: o custo quadruplica quando $n$ dobra). A mediana de três mantém $O(n \log n)$ (45.773 comparações para $n = 4.000$).

*Tarefa 2: QuickSelect com Contagem de Cópias e Comparações*

In [24]:
# Exercício 9 — Tarefa 2: QuickSelect com contagem de operações

def quickselect(arr, goal_index, pivot_rule='median3'):
    """Devolve (elemento que ocuparia goal_index no array ordenado, comparações, cópias).
    Mesma partição do QuickSort, mas segue APENAS o lado que contém goal_index
    (iterativo: nenhuma recursão)."""
    if not 0 <= goal_index < len(arr):
        raise IndexError(f"goal_index {goal_index} fora de [0, {len(arr) - 1}]")
    a, c = list(arr), OpCounter()
    lo, hi = 0, len(a) - 1
    while lo < hi:
        p = _partition(a, lo, hi, c, pivot_rule)
        if p == goal_index:
            break
        if goal_index < p:
            hi = p - 1
        else:
            lo = p + 1
    return a[goal_index], c.comparisons, c.copies

# Corretude contra sorted()
rnd = random.Random(3)
for _ in range(300):
    d = [rnd.randint(0, 50) for _ in range(rnd.randint(1, 60))]
    g = rnd.randrange(len(d))
    assert quickselect(d, g)[0] == sorted(d)[g]
print("QuickSelect == sorted(arr)[goal_index] em 300 casos aleatórios (com duplicatas) ✓\n")

# 2.a: seleção x ordenação completa (QuickSort com short=8, a melhor configuração da Tarefa 1 no caso aleatório)
print(f"{'n':>7} {'padrão':<16}{'goal':>10}{'QuickSelect comp/cópias':>28}{'QuickSort comp/cópias':>28}{'razão comp.':>13}")
for n in [1000, 10000, 100000]:
    for padrao in ['aleatório', 'ordenado']:
        d = gerar_padrao(n, padrao)
        _, sc, sp = quicksort(d, 8)
        for nome_goal, g in [('mínimo', 0), ('mediana', n // 2), ('p90', int(n * 0.9))]:
            v, qc, qp = quickselect(d, g)
            assert v == sorted(d)[g]
            print(f"{n:>7} {padrao:<16}{nome_goal:>10}{f'{qc:,} / {qp:,}':>28}{f'{sc:,} / {sp:,}':>28}{sc/qc:>12.1f}x".replace(',', '.'))

QuickSelect == sorted(arr)[goal_index] em 300 casos aleatórios (com duplicatas) ✓

      n padrão                goal     QuickSelect comp/cópias       QuickSort comp/cópias  razão comp.
   1000 aleatório           mínimo               2.325 / 3.960             10.029 / 15.335         4.3x
   1000 aleatório          mediana               2.038 / 3.762             10.029 / 15.335         4.9x
   1000 aleatório              p90               2.019 / 3.381             10.029 / 15.335         5.0x
   1000 ordenado            mínimo                  2.000 / 48               7.879 / 2.252         3.9x
   1000 ordenado           mediana                  2.001 / 48               7.879 / 2.252         3.9x
   1000 ordenado               p90                  2.010 / 54               7.879 / 2.252         3.9x
  10000 aleatório           mínimo             15.142 / 15.417           143.359 / 222.076         9.5x
  10000 aleatório          mediana             31.929 / 53.343           143.359 / 22

*QuickSelect × Ordenação Completa*
- O QuickSelect segue apenas a partição que contém `goal_index`, então o custo esperado é $n + n/2 + n/4 + \dots \approx 2n$, ou seja, $O(n)$.
  - As contagens confirmam: na entrada ordenada foram exatamente $\approx 2n$ comparações (20.005 para $n = 10.000$ e 200.007 para $n = 100.000$).
  - A ordenação completa custa $O(n \log n)$ (1.900.283 comparações para 100.000 elementos aleatórios).
- A vantagem cresce com $n$: de 3,9–5× para $n = 1.000$ até 7,4–11,6× para $n = 100.000$, que é o fator $\log n$ aparecendo. O QuickSelect também faz muito menos cópias, porque não reorganiza as partições descartadas.
- Pior caso: continua $O(n^2)$ se os pivôs forem sistematicamente ruins. A mediana de três torna isso improvável para entradas ordenadas ou reversas; um pivô aleatório ou o *median of medians* garantiriam limites mais fortes.

*Tarefa 3: Partição em `SinglyLinkedList` (Menores | Iguais | Maiores)*

In [25]:
# Exercício 9 — Tarefa 3: partição em SinglyLinkedList (menores | iguais | maiores)
# Reaproveita Node(value, next) e SinglyLinkedList do Exercício 1.

def sll_from_iterable(values):
    """Constrói em O(n) mantendo um ponteiro de cauda local
    (o append do Exercício 1 percorre a lista inteira: O(n) por inserção)."""
    lst, tail = SinglyLinkedList(), None
    for v in values:
        node = Node(v)
        if tail is None: lst.head = node
        else: tail.next = node
        tail = node
    return lst

def sll_to_list(lst):
    out, cur = [], lst.head
    while cur:
        out.append(cur.value); cur = cur.next
    return out

def partition_sll(lst, pivot, key=lambda v: v):
    """3.a: percorre a lista UMA vez e religa cada nó ao final de uma de três cadeias.
    Nenhum nó é criado nem copiado: só ponteiros mudam. Como cada nó vai para o FIM
    da sua cadeia, a ordem relativa original é mantida dentro de cada grupo (partição ESTÁVEL).
    Devolve (lista_particionada, stats)."""
    heads, tails = [None, None, None], [None, None, None]   # 0: menores, 1: iguais, 2: maiores
    stats = {'comparações': 0, 'atribuições de ponteiro': 0, 'nós visitados': 0}
    pk = key(pivot)
    cur = lst.head
    while cur:
        nxt = cur.next
        stats['nós visitados'] += 1
        k = key(cur.value)
        stats['comparações'] += 1
        if k < pk:
            g = 0
        else:
            stats['comparações'] += 1
            g = 1 if k == pk else 2
        cur.next = None
        if tails[g] is None: heads[g] = cur
        else: tails[g].next = cur
        tails[g] = cur
        stats['atribuições de ponteiro'] += 2
        cur = nxt
    # concatena menores -> iguais -> maiores
    result, last = SinglyLinkedList(), None
    for g in range(3):
        if heads[g] is None: continue
        if last is None: result.head = heads[g]
        else: last.next = heads[g]; stats['atribuições de ponteiro'] += 1
        last = tails[g]
    return result, stats

# Demonstração com registros (chave, rótulo): estabilidade visível pelos rótulos
dados = [(5, 'a'), (2, 'b'), (8, 'c'), (5, 'd'), (1, 'e'), (9, 'f'), (2, 'g'), (5, 'h')]
lst = sll_from_iterable(dados)
part, st = partition_sll(lst, (5, None), key=lambda r: r[0])
print("entrada :", dados)
print("pivô = 5 ->", sll_to_list(part))
print("estatísticas:", st)
res = sll_to_list(part)
assert [r for r in res if r[0] < 5] == [r for r in dados if r[0] < 5]      # ordem preservada
assert [r for r in res if r[0] == 5] == [r for r in dados if r[0] == 5]
assert [r for r in res if r[0] > 5] == [r for r in dados if r[0] > 5]
print("ordem relativa preservada em cada grupo (b antes de g; a, d, h na ordem original) ✓")

# QuickSort sobre lista encadeada usando a partição de 3 vias (sem acesso por índice)
def quicksort_sll(lst, pivot_rule='head'):
    stats = {'comparações': 0, 'nós visitados': 0}
    def sort(head):
        if head is None or head.next is None:
            return head, head
        if pivot_rule == 'head':
            pivot = head.value                    # O(1): é o único nó de acesso imediato
        else:                                     # 'middle': precisa ANDAR até o meio -> O(n)
            slow = fast = head
            while fast.next and fast.next.next:
                slow, fast = slow.next, fast.next.next
                stats['nós visitados'] += 1
            pivot = slow.value
        part_heads, _ = _three_way(head, pivot)
        (lh, eh, et, gh) = part_heads
        lh, lt = sort(lh)
        gh, gt = sort(gh)
        # concatena menores + iguais + maiores
        head_out = lh if lh else eh
        if lt: lt.next = eh
        et.next = gh
        return head_out, (gt if gt else et)
    def _three_way(head, pivot):
        hs, ts = [None] * 3, [None] * 3
        cur = head
        while cur:
            nxt = cur.next; stats['nós visitados'] += 1
            stats['comparações'] += 1
            if cur.value < pivot: g = 0
            else:
                stats['comparações'] += 1
                g = 1 if cur.value == pivot else 2
            cur.next = None
            if ts[g] is None: hs[g] = cur
            else: ts[g].next = cur
            ts[g] = cur; cur = nxt
        return (hs[0], hs[1], ts[1], hs[2]), None
    h, _ = sort(lst.head)
    out = SinglyLinkedList(); out.head = h
    return out, stats

print(f"\n{'n':>6} {'padrão':<16}{'SLL pivô=cabeça':>18}{'SLL pivô=meio':>16}{'nós visitados (meio)':>22}{'array QuickSort':>17}")
for n in [500, 2000]:
    for padrao in ['aleatório', 'ordenado']:
        d = gerar_padrao(n, padrao)
        try:
            r, s1 = quicksort_sll(sll_from_iterable(d), 'head')
            assert sll_to_list(r) == sorted(d); cab = f"{s1['comparações']:,}"
        except RecursionError:                         # pivô na cabeça + entrada ordenada: profundidade n
            cab = 'RecursionError'
        r, s2 = quicksort_sll(sll_from_iterable(d), 'middle')
        assert sll_to_list(r) == sorted(d)
        _, ac, _ = quicksort(d, 1)
        print(f"{n:>6} {padrao:<16}{cab:>18}{s2['comparações']:>16,}{s2['nós visitados']:>22,}{ac:>17,}".replace(',', '.'))

entrada : [(5, 'a'), (2, 'b'), (8, 'c'), (5, 'd'), (1, 'e'), (9, 'f'), (2, 'g'), (5, 'h')]
pivô = 5 -> [(2, 'b'), (1, 'e'), (2, 'g'), (5, 'a'), (5, 'd'), (5, 'h'), (8, 'c'), (9, 'f')]
estatísticas: {'comparações': 13, 'atribuições de ponteiro': 18, 'nós visitados': 8}
ordem relativa preservada em cada grupo (b antes de g; a, d, h na ordem original) ✓

     n padrão             SLL pivô=cabeça   SLL pivô=meio  nós visitados (meio)  array QuickSort
   500 aleatório                    7.764           8.380                 7.891            4.549
   500 ordenado                   250.498           5.784                 5.475            4.230
  2000 aleatório                   39.027          41.569                38.719           22.730
  2000 ordenado            RecursionError          29.104                27.857           20.892


*Por que a Partição Muda sem Acesso por Índice*
- A partição em lista percorre os nós uma única vez e religa cada um ao fim de uma de três cadeias. O custo é $O(n)$, sem copiar valores: só ponteiros mudam (18 atribuições para 8 nós).
- Estabilidade: como cada nó é anexado ao fim da sua cadeia, a ordem relativa é preservada, o que a saída comprova (b antes de g; a, d e h na ordem original). A partição de Lomuto em array não é estável, porque troca elementos distantes.
- O que se perde sem acesso por índice:
  - não é possível trocar `a[i]` com `a[j]` nem varrer a partir das duas pontas (a lista não tem `prev`);
  - escolher a mediana exige andar até o meio, em $O(n)$ (ponteiros lento e rápido).
- O ponto técnico: como a partição já custa $O(n)$ por nível, andar até o meio não muda o Big O, só a constante. Já o pivô "cabeça", que é o único de custo $O(1)$, degrada em entradas ordenadas:
  - com $n = 500$ ordenado, foram 250.498 comparações ($O(n^2)$) contra 5.784 com o pivô do meio;
  - com $n = 2.000$, a recursão atinge profundidade $n$ e falha com `RecursionError`.
- Em média, o QuickSort em lista fez cerca de 1,8× mais comparações que o de array, porque a partição de 3 vias usa até 2 comparações por nó. Além disso, a lista não tem localidade de memória.

### Exercício 10: Lista encadeada com operações completas e análise de eficiência

**Introdução**  
Listas encadeadas não são apenas "uma lista alternativa", elas são uma escolha de engenharia quando inserções e remoções locais precisam ser eficientes, ou quando a estrutura deve crescer dinamicamente sem realocações. Entretanto, o preço é pago em busca, acesso por índice e overhead de memória. Neste exercício, você implementará uma lista encadeada do zero com um conjunto completo de operações, incluindo inserção e remoção em diferentes posições, busca e travessia segura. A exigência de qualidade inclui tratamento de casos de borda, testes consistentes e uma análise de custo por operação em termos de Big O. Você também deverá justificar uma decisão de design, como manter ponteiro para cauda ou não, e demonstrar o impacto dessa decisão com medições simples.

**Tarefa**  
1. Implemente `SinglyLinkedList` com:
   - a. `insert_first`, `insert_last`, `search(value)`, `delete(value)`, `__len__`, `__str__`
2. Adicione suporte a operações por posição:
   - a. `insert_at(index, value)` e `delete_at(index)` com validações e mensagens de erro claras.
3. Análise e evidências:
   - a. Para cada operação, registre o custo esperado em Big O e demonstre ao menos um experimento que evidencie o custo de busca linear;
   - b. Justifique tecnicamente a presença ou ausência de ponteiro para cauda e o impacto em `insert_last`.

*Tarefas 1 e 2: `SinglyLinkedList` com Operações Completas e Operações por Posição*

In [26]:
# Exercício 10 — Tarefas 1 e 2: SinglyLinkedList completa
# (substitui a versão mínima do Exercício 1, mantendo o atributo `head`)

class _SNode:
    __slots__ = ('value', 'next')
    def __init__(self, value, nxt=None):
        self.value, self.next = value, nxt

class SinglyLinkedList:
    """Lista simplesmente encadeada com ponteiro de cauda e contador de tamanho.
    Invariantes: size == nº de nós alcançáveis a partir de head;
                 tail é o último nó (tail.next is None); lista vazia <=> head is tail is None.
    track_tail=False desliga o ponteiro de cauda (usado só no experimento da Tarefa 3.b)."""

    def __init__(self, iterable=(), track_tail=True):
        self.head = self.tail = None
        self._size = 0
        self.track_tail = track_tail
        self.last_comparisons = 0      # comparações da última search/delete
        self.last_traversals = 0       # nós percorridos na última operação posicional/insert_last
        for v in iterable:
            self.insert_last(v)

    # ---------- Tarefa 1 ----------
    def insert_first(self, value):                  # O(1)
        self.head = _SNode(value, self.head)
        if self._size == 0:
            self.tail = self.head
        self._size += 1

    def insert_last(self, value):                   # O(1) com cauda | O(n) sem cauda
        node = _SNode(value)
        self.last_traversals = 0
        if self.head is None:
            self.head = self.tail = node
        elif self.track_tail:
            self.tail.next = node
            self.tail = node
        else:                                        # sem cauda: precisa andar até o fim
            cur = self.head
            while cur.next:
                cur = cur.next; self.last_traversals += 1
            cur.next = node
            self.tail = node                         # mantido só para check_invariants
        self._size += 1

    def search(self, value):                        # O(n): devolve o índice ou -1
        self.last_comparisons = 0
        cur, idx = self.head, 0
        while cur:
            self.last_comparisons += 1
            if cur.value == value:
                return idx
            cur, idx = cur.next, idx + 1
        return -1

    def delete(self, value):                        # O(n): remove a 1ª ocorrência; devolve bool
        self.last_comparisons = 0
        prev, cur = None, self.head
        while cur:
            self.last_comparisons += 1
            if cur.value == value:
                self._unlink(prev, cur)
                return True
            prev, cur = cur, cur.next
        return False

    def _unlink(self, prev, cur):
        if prev is None:
            self.head = cur.next
        else:
            prev.next = cur.next
        if cur is self.tail:                         # removendo o último: a cauda recua
            self.tail = prev
        cur.next = None                              # desliga o nó removido
        self._size -= 1

    def __len__(self):                              # O(1) graças ao contador
        return self._size

    def __iter__(self):
        cur = self.head
        while cur:
            yield cur.value
            cur = cur.next

    def __str__(self):
        return "[" + " -> ".join(map(repr, self)) + "]" if self.head else "[]"

    # ---------- Tarefa 2 ----------
    def _check_index(self, index, upper, op):
        if isinstance(index, bool) or not isinstance(index, int):
            raise TypeError(f"{op}: índice deve ser int, recebido {type(index).__name__}")
        if not 0 <= index <= upper:
            faixa = f"[0, {upper}]" if upper >= 0 else "(lista vazia)"
            raise IndexError(f"{op}: índice {index} fora do intervalo válido {faixa} (tamanho={self._size})")

    def insert_at(self, index, value):              # O(index)
        self._check_index(index, self._size, "insert_at")       # index == size equivale a insert_last
        if index == 0:
            return self.insert_first(value)
        if index == self._size:
            return self.insert_last(value)
        prev, self.last_traversals = self.head, 0
        for _ in range(index - 1):
            prev = prev.next; self.last_traversals += 1
        prev.next = _SNode(value, prev.next)
        self._size += 1

    def get(self, index):                           # O(index): leitura por posição
        if self._size == 0:
            raise IndexError("get: lista vazia")
        self._check_index(index, self._size - 1, "get")
        cur, self.last_traversals = self.head, 0
        for _ in range(index):
            cur = cur.next; self.last_traversals += 1
        return cur.value

    def delete_at(self, index):                     # O(index); devolve o valor removido
        if self._size == 0:
            raise IndexError("delete_at: lista vazia")
        self._check_index(index, self._size - 1, "delete_at")
        prev, cur, self.last_traversals = None, self.head, 0
        for _ in range(index):
            prev, cur = cur, cur.next; self.last_traversals += 1
        self._unlink(prev, cur)
        return cur.value

    def check_invariants(self):
        count, last, cur = 0, None, self.head
        while cur:
            count, last, cur = count + 1, cur, cur.next
        assert count == self._size, f"tamanho {self._size} != nós {count}"
        assert last is self.tail, "tail não aponta para o último nó"
        assert (self.head is None) == (self.tail is None) == (self._size == 0)
        return True


# ---------------- Testes de casos de borda ----------------
L = SinglyLinkedList()
print("vazia:", L, "| len =", len(L), "| search(1) =", L.search(1), "| delete(1) =", L.delete(1))
for op, args in [(L.delete_at, (0,)), (L.insert_at, (1, 'x')), (L.insert_at, ('0', 'x')), (L.insert_at, (-1, 'x'))]:
    try:
        op(*args); raise AssertionError
    except (IndexError, TypeError) as e:
        print(f"  {op.__name__}{args} -> {type(e).__name__}: {e}")

L.insert_last(2); L.insert_first(1); L.insert_last(4); L.insert_at(2, 3); L.insert_at(0, 0); L.insert_at(5, 5)
print("após inserções:", L, "| len =", len(L)); assert list(L) == [0, 1, 2, 3, 4, 5] and L.check_invariants()
print("leitura: get(0) =", L.get(0), "| get(3) =", L.get(3), f"({L.last_traversals} nós percorridos) | get(5) =", L.get(5))
try:
    L.get(6)
except IndexError as e:
    print("  get(6) ->", e)
print("search(3) =", L.search(3), f"({L.last_comparisons} comparações) | search(9) =", L.search(9), f"({L.last_comparisons} comparações)")
assert L.delete(5) and L.tail.value == 4 and L.check_invariants()        # remove a cauda: tail recua
assert L.delete(0) and L.head.value == 1 and L.check_invariants()        # remove a cabeça
assert L.delete_at(1) == 2 and L.check_invariants()                      # remove do meio
assert not L.delete(42)
print("após deletes (cauda, cabeça, meio):", L)
try:
    L.delete_at(3)
except IndexError as e:
    print("  delete_at(3) ->", e)
while len(L):
    L.delete_at(len(L) - 1); L.check_invariants()
print("esvaziada pelo fim:", L, "| head/tail =", L.head, L.tail)
L.insert_last('único'); assert L.head is L.tail and L.delete('único') and L.head is L.tail is None
print("lista unitária: inserir e remover mantém head == tail == None ✓")

# Estresse contra list do Python
rnd = random.Random(11); L, ref = SinglyLinkedList(), []
for _ in range(5000):
    op = rnd.random()
    if op < 0.3:   v = rnd.randint(0, 30); L.insert_first(v); ref.insert(0, v)
    elif op < 0.55: v = rnd.randint(0, 30); L.insert_last(v); ref.append(v)
    elif op < 0.7:  i = rnd.randint(0, len(ref)); v = rnd.randint(0, 30); L.insert_at(i, v); ref.insert(i, v)
    elif op < 0.85 and ref:
        i = rnd.randrange(len(ref)); assert L.get(i) == ref[i]; assert L.delete_at(i) == ref.pop(i)
    else:
        v = rnd.randint(0, 30); ok = v in ref
        assert L.delete(v) == ok
        if ok: ref.remove(v)
    assert list(L) == ref and L.check_invariants()
print(f"estresse: 5.000 operações aleatórias idênticas a list do Python, invariantes checadas a cada passo ✓ (tamanho final {len(L)})")

vazia: [] | len = 0 | search(1) = -1 | delete(1) = False
  delete_at(0,) -> IndexError: delete_at: lista vazia
  insert_at(1, 'x') -> IndexError: insert_at: índice 1 fora do intervalo válido [0, 0] (tamanho=0)
  insert_at('0', 'x') -> TypeError: insert_at: índice deve ser int, recebido str
  insert_at(-1, 'x') -> IndexError: insert_at: índice -1 fora do intervalo válido [0, 0] (tamanho=0)
após inserções: [0 -> 1 -> 2 -> 3 -> 4 -> 5] | len = 6
leitura: get(0) = 0 | get(3) = 3 (3 nós percorridos) | get(5) = 5
  get(6) -> get: índice 6 fora do intervalo válido [0, 5] (tamanho=6)
search(3) = 3 (4 comparações) | search(9) = -1 (6 comparações)
após deletes (cauda, cabeça, meio): [1 -> 3 -> 4]
  delete_at(3) -> delete_at: índice 3 fora do intervalo válido [0, 2] (tamanho=3)
esvaziada pelo fim: [] | head/tail = None None
lista unitária: inserir e remover mantém head == tail == None ✓
estresse: 5.000 operações aleatórias idênticas a list do Python, invariantes checadas a cada passo ✓ (tamanho f

*Tarefa 3: Análise e Evidências*

In [27]:
# Exercício 10 — Tarefa 3: evidências de custo
import time
import pandas as pd

# 3.a — busca linear: comparações crescem linearmente com n
linhas = []
for n in [100, 1_000, 10_000, 100_000]:
    L = SinglyLinkedList(range(n))
    L.search(n // 2); meio = L.last_comparisons
    t0 = time.perf_counter(); L.search(-1); t_aus = (time.perf_counter() - t0) * 1e3
    aus = L.last_comparisons
    L.search(0); inicio = L.last_comparisons
    linhas.append({'n': n, 'alvo na cabeça': inicio, 'alvo no meio': meio, 'alvo ausente': aus,
                   'tempo ausente (ms)': round(t_aus, 3)})
df = pd.DataFrame(linhas)
print("Comparações de search(value):"); print(df.to_string(index=False))
print("razão (ausente, n=100.000 / n=10.000) =", df['alvo ausente'].iloc[3] / df['alvo ausente'].iloc[2],
      "-> 10x mais dados, 10x mais comparações: O(n)")

# 3.a — operações posicionais: custo proporcional ao índice
L = SinglyLinkedList(range(10_000))
print("\ninsert_at / delete_at percorrem até a posição:")
for idx in [1, 2_500, 5_000, 9_999]:
    L.insert_at(idx, -1); a = L.last_traversals
    L.delete_at(idx);     b = L.last_traversals
    print(f"  índice {idx:>5}: insert_at percorreu {a:>5} nós | delete_at percorreu {b:>5} nós")

# 3.b — impacto do ponteiro de cauda em insert_last
print("\nConstruir a lista com n chamadas de insert_last:")
print(f"{'n':>7}{'com cauda: nós percorridos':>28}{'tempo':>10}{'sem cauda: nós percorridos':>29}{'tempo':>11}")
for n in [1_000, 5_000, 10_000]:
    res = []
    for track in (True, False):
        L = SinglyLinkedList(track_tail=track); total = 0
        t0 = time.perf_counter()
        for v in range(n):
            L.insert_last(v); total += L.last_traversals
        res.append((total, time.perf_counter() - t0))
        assert L.check_invariants()
    print(f"{n:>7}{res[0][0]:>28,}{res[0][1]*1e3:>8.1f}ms{res[1][0]:>29,}{res[1][1]*1e3:>9.1f}ms".replace(',', '.'))
print("sem cauda: total = 0 + 1 + ... + (n-2) = (n-1)(n-2)/2 -> O(n²) para construir; com cauda: O(n)")

# 3.a (complemento) — lista encadeada x estrutura baseada em array (list do Python)
import gc
def medir(fn, reps=200):
    """Tempo médio por chamada em microssegundos, com o coletor de lixo desligado durante a medição."""
    gc.collect(); gc.disable()
    t0 = time.perf_counter()
    for _ in range(reps): fn()
    dt = (time.perf_counter() - t0) / reps * 1e6
    gc.enable()
    return dt

operacoes = [   # (nome, operação na lista encadeada, operação no array, explicação)
    ("inserir no início",      lambda L, n: L.insert_first(-1), lambda A, n: A.insert(0, -1),
     "lista O(1) × array O(n): desloca todos os elementos"),
    ("inserir no fim",         lambda L, n: L.insert_last(-1),  lambda A, n: A.append(-1),
     "ambos O(1): ponteiro de cauda × append amortizado"),
    ("ler o elemento do meio", lambda L, n: L.get(n // 2),      lambda A, n: A[n // 2],
     "lista O(n) × array O(1): acesso direto por índice"),
    ("busca linear (ausente)", lambda L, n: L.search(-5),       lambda A, n: linear_search(A, -5),
     "ambos O(n) e n comparações; array é contíguo na memória"),
]
print("\nLista encadeada × array (list do Python) — microssegundos por operação")
print(f"{'n':>8} {'operação':<24}{'SinglyLinkedList':>18}{'list (array)':>14}  explicação")
for n in [1_000, 10_000, 100_000]:
    for nome, f_ll, f_arr, motivo in operacoes:
        L, A = SinglyLinkedList(range(n)), list(range(n))     # estruturas novas a cada medição
        t_ll, t_arr = medir(lambda: f_ll(L, n)), medir(lambda: f_arr(A, n))
        print(f"{n:>8} {nome:<24}{t_ll:>18.2f}{t_arr:>14.2f}  {motivo}")

Comparações de search(value):
     n  alvo na cabeça  alvo no meio  alvo ausente  tempo ausente (ms)
   100               1            51           100               0.011
  1000               1           501          1000               0.130
 10000               1          5001         10000               1.283
100000               1         50001        100000              13.140
razão (ausente, n=100.000 / n=10.000) = 10.0 -> 10x mais dados, 10x mais comparações: O(n)

insert_at / delete_at percorrem até a posição:
  índice     1: insert_at percorreu     0 nós | delete_at percorreu     1 nós
  índice  2500: insert_at percorreu  2499 nós | delete_at percorreu  2500 nós
  índice  5000: insert_at percorreu  4999 nós | delete_at percorreu  5000 nós
  índice  9999: insert_at percorreu  9998 nós | delete_at percorreu  9999 nós

Construir a lista com n chamadas de insert_last:
      n  com cauda: nós percorridos     tempo   sem cauda: nós percorridos      tempo
   1000                     

*Custo por Operação e Decisão sobre o Ponteiro de Cauda*

| Operação | Custo | Observação |
|---|---|---|
| `insert_first` | $O(1)$ | só atualiza `head` (e `tail` se a lista estava vazia) |
| `insert_last` | $O(1)$ com cauda / $O(n)$ sem | ver experimento abaixo |
| `search(value)` | $O(n)$ | $n$ comparações quando o valor está ausente |
| `delete(value)` | $O(n)$ | percorre até achar; atualiza `tail` se remover o último |
| `insert_at(i)` / `delete_at(i)` | $O(i)$ | percorre $i-1$ ou $i$ nós; `delete_at(n-1)` é $O(n)$ mesmo com cauda, pois precisa do predecessor |
| `get(i)` (leitura) | $O(i)$ | percorre $i$ nós; valida o índice como `delete_at` |
| `__len__` | $O(1)$ | contador mantido em cada inserção/remoção |
| `__str__` / iteração | $O(n)$ | |

- Busca linear: as comparações foram exatamente $n$ para alvo ausente e $n/2 + 1$ para o alvo no meio. Com 10× mais dados, foram 10× mais comparações (10.000 → 100.000), e o tempo cresceu na mesma proporção (≈0,44 ms → ≈4,8 ms): $O(n)$.
- Operações posicionais: os nós percorridos por `insert_at` e `delete_at` cresceram linearmente com o índice (2.499, 4.999, 9.998, …).
- Comparação com estrutura baseada em array (`list` do Python), em microssegundos por operação:
  - Inserir no início: a lista encadeada ficou constante (≈0,3 µs para qualquer $n$), e o array cresceu com $n$ (≈0,6 → ≈4 → ≈40–50 µs, conforme a execução), porque `insert(0, x)` desloca todos os elementos: $O(1)$ contra $O(n)$.
  - Ler o elemento do meio: o array ficou constante (≈0,1–0,2 µs), e a lista cresceu linearmente (≈16 → ≈150 → ≈1.540 µs): $O(1)$ contra $O(n)$.
  - Inserir no fim: os dois são $O(1)$ (ponteiro de cauda contra `append` amortizado), com tempos sub-microssegundo nos dois.
  - Busca linear: os dois fazem as mesmas $n$ comparações, mas a lista foi ≈1,4× mais lenta, porque cada nó está em um lugar diferente da memória (sem localidade de cache) e acessá-lo exige seguir um ponteiro.
  - Conclusão: a lista encadeada vence quando as inserções e remoções acontecem nas pontas ou em posições já alcançadas; o array vence no acesso por índice e na varredura.
- Ponteiro de cauda: sem ele, cada `insert_last` percorre a lista inteira, e construir $n$ elementos custa $\sum_{i=0}^{n-2} i = \frac{(n-1)(n-2)}{2}$. Para $n = 10.000$, isso deu 49.985.001 nós percorridos e ≈1,5 s, contra 0 nós e ≈7 ms com a cauda (os tempos variam um pouco a cada execução; as contagens são exatas).

  O custo da cauda é pequeno: um ponteiro a mais e o cuidado de recuar a `tail` ao remover o último nó (o predecessor já está disponível durante a varredura). O invariante é verificado por `check_invariants` em todos os testes. A decisão é manter a cauda, porque transforma a construção e o uso como fila de $O(n^2)$ para $O(n)$.
- Robustez: os casos de borda testados foram lista vazia, remoção da cabeça, da cauda e do meio, lista unitária, índices inválidos (negativos, fora do intervalo, não inteiros, com mensagens claras) e `insert_at(len)` equivalendo a `insert_last`. Também foram executadas 5.000 operações aleatórias comparadas passo a passo com `list` do Python.

### Exercício 11: Lista duplamente encadeada e Deque com especificação rigorosa

**Introdução**  
Deques são fundamentais em arquiteturas de processamento, buffers e algoritmos que exigem inserção e remoção em ambas as extremidades. A implementação eficiente de um deque geralmente exige uma lista duplamente encadeada, com manutenção cuidadosa de ponteiros e invariantes, pois pequenos erros levam a corrupção estrutural difícil de diagnosticar. Neste exercício, você implementará uma lista duplamente encadeada e construirá um Deque sobre ela, garantindo operações $O(1)$ em ambas as pontas. A entrega deve incluir testes que validem invariantes após sequências longas de operações, além de uma análise objetiva de eficiência. Como requisito de qualidade, você deve documentar decisões de projeto, como tratamento de deque vazio e gerenciamento de nós, e demonstrar por evidências que sua implementação é robusta.

**Tarefa**  
1. Implemente `DoublyLinkedList` com nós que tenham `prev` e `next`.
   - a. Inclua `insert_first`, `insert_last`, `delete_first`, `delete_last`, `is_empty`.
2. Implemente `Deque` sobre a lista duplamente encadeada.
   - a. Métodos obrigatórios: `insert_left`, `insert_right`, `remove_left`, `remove_right`, `peek_left`, `peek_right`.
3. Testes e invariantes:
   - a. Escreva uma bateria de testes com sequências que alternem inserções e remoções nas duas pontas;
   - b. Verifique invariantes estruturais (ponteiros consistentes, tamanho correto) após cada sequência e descreva a complexidade por operação.

*Tarefas 1 e 2: `DoublyLinkedList` com Sentinelas e `Deque` sobre Ela*

In [28]:
# Exercício 11 — Tarefas 1 e 2: DoublyLinkedList e Deque

class EmptyStructureError(Exception):
    """Remoção ou consulta em estrutura vazia."""

class _DNode:
    __slots__ = ('value', 'prev', 'next')
    def __init__(self, value=None, prev=None, nxt=None):
        self.value, self.prev, self.next = value, prev, nxt

class DoublyLinkedList:
    """Decisão de projeto: dois nós SENTINELA (header e trailer) que nunca são removidos.
    Toda inserção/remoção ocorre ENTRE dois nós existentes, então não há casos especiais
    para lista vazia ou unitária (nenhum `if head is None`). Custo: 2 nós extras fixos.
    Gerenciamento de nós: o nó removido tem prev/next zerados, evitando referências pendentes."""

    def __init__(self):
        self._header = _DNode()
        self._trailer = _DNode()
        self._header.next = self._trailer
        self._trailer.prev = self._header
        self._size = 0

    def __len__(self):
        return self._size

    def is_empty(self):
        return self._size == 0

    def _insert_between(self, value, pred, succ):          # O(1): 4 atribuições de ponteiro
        node = _DNode(value, pred, succ)
        pred.next = node
        succ.prev = node
        self._size += 1
        return node

    def _delete_node(self, node):                          # O(1)
        node.prev.next = node.next
        node.next.prev = node.prev
        self._size -= 1
        value = node.value
        node.prev = node.next = node.value = None           # desliga o nó removido
        return value

    def insert_first(self, value):
        return self._insert_between(value, self._header, self._header.next)

    def insert_last(self, value):
        return self._insert_between(value, self._trailer.prev, self._trailer)

    def delete_first(self):
        if self.is_empty():
            raise EmptyStructureError("delete_first em lista vazia")
        return self._delete_node(self._header.next)

    def delete_last(self):
        if self.is_empty():
            raise EmptyStructureError("delete_last em lista vazia")
        return self._delete_node(self._trailer.prev)

    def first(self):
        if self.is_empty():
            raise EmptyStructureError("lista vazia")
        return self._header.next.value

    def last(self):
        if self.is_empty():
            raise EmptyStructureError("lista vazia")
        return self._trailer.prev.value

    def __iter__(self):
        cur = self._header.next
        while cur is not self._trailer:
            yield cur.value
            cur = cur.next

    def check_invariants(self):
        """3.b: (1) percurso para frente e para trás têm `size` nós; (2) para todo nó x,
        x.next.prev is x e x.prev.next is x; (3) sentinelas nas pontas;
        (4) os dois percursos visitam os mesmos valores em ordem inversa."""
        h, t = self._header, self._trailer
        assert h.prev is None and t.next is None, "sentinelas corrompidas"
        fwd, cur = [], h
        while cur.next is not None:
            assert cur.next.prev is cur, "cur.next.prev != cur"
            cur = cur.next
            if cur is not t: fwd.append(cur.value)
            assert len(fwd) <= self._size, "ciclo ou tamanho inconsistente (frente)"
        assert cur is t
        bwd, cur = [], t
        while cur.prev is not None:
            assert cur.prev.next is cur, "cur.prev.next != cur"
            cur = cur.prev
            if cur is not h: bwd.append(cur.value)
            assert len(bwd) <= self._size, "ciclo ou tamanho inconsistente (trás)"
        assert cur is h
        assert len(fwd) == len(bwd) == self._size, "size incorreto"
        assert fwd == bwd[::-1], "percursos divergentes"
        return True

    def __repr__(self):
        return "DLL[" + " <-> ".join(map(repr, self)) + "]"


class Deque:
    """Deque sobre a DoublyLinkedList: todas as operações são O(1) nas duas pontas.
    Deque vazio: remove_* e peek_* levantam EmptyStructureError (em vez de devolver None,
    que seria ambíguo quando None é um valor armazenado)."""
    def __init__(self):
        self._list = DoublyLinkedList()
    def insert_left(self, value):  self._list.insert_first(value)
    def insert_right(self, value): self._list.insert_last(value)
    def remove_left(self):
        try: return self._list.delete_first()
        except EmptyStructureError: raise EmptyStructureError("remove_left em deque vazio") from None
    def remove_right(self):
        try: return self._list.delete_last()
        except EmptyStructureError: raise EmptyStructureError("remove_right em deque vazio") from None
    def peek_left(self):
        try: return self._list.first()
        except EmptyStructureError: raise EmptyStructureError("peek_left em deque vazio") from None
    def peek_right(self):
        try: return self._list.last()
        except EmptyStructureError: raise EmptyStructureError("peek_right em deque vazio") from None
    def is_empty(self): return self._list.is_empty()
    def __len__(self):  return len(self._list)
    def __iter__(self): return iter(self._list)
    def check_invariants(self): return self._list.check_invariants()
    def __repr__(self): return "Deque(" + ", ".join(map(repr, self)) + ")"


# Demonstração rápida
d = Deque()
d.insert_right(2); d.insert_left(1); d.insert_right(3); d.insert_left(0)
print(d, "| peek_left =", d.peek_left(), "| peek_right =", d.peek_right(), "| len =", len(d))
print("remove_left ->", d.remove_left(), "| remove_right ->", d.remove_right(), "|", d)
d.remove_left(); d.remove_right()
for op in (d.remove_left, d.remove_right, d.peek_left, d.peek_right):
    try: op()
    except EmptyStructureError as e: print("  ", e)
d.insert_left(None); assert d.peek_right() is None and len(d) == 1     # None é um valor válido
print("None armazenado e lido sem ambiguidade ✓", d)

Deque(0, 1, 2, 3) | peek_left = 0 | peek_right = 3 | len = 4
remove_left -> 0 | remove_right -> 3 | Deque(1, 2)
   remove_left em deque vazio
   remove_right em deque vazio
   peek_left em deque vazio
   peek_right em deque vazio
None armazenado e lido sem ambiguidade ✓ Deque(None)


*Tarefa 3: Bateria de Testes e Verificação de Invariantes*

In [29]:
# Exercício 11 — Tarefa 3: bateria de testes e invariantes
import time, gc
from collections import deque as RefDeque          # usado APENAS como oráculo de referência

def run_sequence(ops):
    """Executa uma sequência de operações no Deque e no oráculo; confere resultados e invariantes."""
    d, ref = Deque(), RefDeque()
    for op, *arg in ops:
        if op == 'L+':   d.insert_left(arg[0]);  ref.appendleft(arg[0])
        elif op == 'R+': d.insert_right(arg[0]); ref.append(arg[0])
        elif op == 'L-':
            if ref: assert d.remove_left() == ref.popleft()
            else:
                try: d.remove_left(); raise AssertionError("deveria falhar")
                except EmptyStructureError: pass
        elif op == 'R-':
            if ref: assert d.remove_right() == ref.pop()
            else:
                try: d.remove_right(); raise AssertionError("deveria falhar")
                except EmptyStructureError: pass
        if ref:
            assert d.peek_left() == ref[0] and d.peek_right() == ref[-1]
    assert list(d) == list(ref) and len(d) == len(ref)
    d.check_invariants()
    return len(d)

# 3.a: sequências que alternam inserções e remoções nas duas pontas
sequencias = {
    "alternada L/R": [('L+', i) if i % 2 else ('R+', i) for i in range(10)] + [('L-',), ('R-',)] * 5,
    "insere esquerda, remove direita (fila)": [('L+', i) for i in range(20)] + [('R-',)] * 20,
    "insere direita, remove direita (pilha)": [('R+', i) for i in range(20)] + [('R-',)] * 20,
    "esvazia e reutiliza": [('R+', 1), ('L-',), ('L+', 2), ('R-',), ('R+', 3), ('R+', 4), ('L-',), ('L-',)],
    "remoções em vazio": [('L-',), ('R-',), ('L+', 'x'), ('R-',), ('L-',)],
    "zigue-zague": [x for i in range(50) for x in (('L+', i), ('R+', -i), ('L-',) if i % 3 == 0 else ('R-',))],
}
for nome, seq in sequencias.items():
    final = run_sequence(seq)
    print(f"✓ {nome:<40} {len(seq):>4} operações | tamanho final {final} | invariantes OK")

# Sequências aleatórias longas, com invariantes verificadas após cada bloco
rnd = random.Random(2026)
total = 0
for bloco in range(20):
    seq = []
    for _ in range(5000):
        r = rnd.random()
        seq.append(('L+', rnd.randint(0, 99)) if r < 0.3 else ('R+', rnd.randint(0, 99)) if r < 0.6
                   else ('L-',) if r < 0.8 else ('R-',))
    run_sequence(seq); total += len(seq)
print(f"✓ {total:_} operações aleatórias em 20 blocos: resultado idêntico ao oráculo e invariantes OK".replace('_', '.'))

# 3.b: evidência de O(1) — tempo médio por operação não cresce com o tamanho do deque
print(f"\n{'tamanho do deque':>17}{'insert_left':>13}{'insert_right':>14}{'remove_left':>13}{'remove_right':>14}  (ns por operação)")
for n in [1_000, 10_000, 100_000, 1_000_000]:
    d = Deque()
    for i in range(n): d.insert_right(i)
    reps, res = 20_000, []
    gc.collect(); gc.disable()          # evita que pausas do coletor de lixo contaminem a medição
    for ins, rem in [(d.insert_left, d.remove_left), (d.insert_right, d.remove_right)]:
        best_ins = best_rem = float('inf')
        for _ in range(3):              # melhor de 3 repetições
            t0 = time.perf_counter()
            for i in range(reps): ins(i)
            best_ins = min(best_ins, (time.perf_counter() - t0) / reps * 1e9)
            t0 = time.perf_counter()
            for i in range(reps): rem()
            best_rem = min(best_rem, (time.perf_counter() - t0) / reps * 1e9)
        res += [best_ins, best_rem]
    gc.enable()
    assert len(d) == n and d.check_invariants()
    print(f"{n:>17,}{res[0]:>13.0f}{res[2]:>14.0f}{res[1]:>13.0f}{res[3]:>14.0f}".replace(',', '.'))

# O verificador de invariantes realmente detecta corrupção? (teste do próprio teste)
for nome, corromper in [("prev quebrado", lambda L: setattr(L._header.next.next, 'prev', L._header)),
                        ("size errado",   lambda L: setattr(L, '_size', L._size + 1)),
                        ("ciclo",         lambda L: setattr(L._trailer.prev, 'next', L._header.next))]:
    L = DoublyLinkedList()
    for i in range(5): L.insert_last(i)
    corromper(L)
    try:
        L.check_invariants(); print(f"✗ {nome}: NÃO detectado")
    except AssertionError as e:
        print(f"✓ corrupção '{nome}' detectada: {e}")

✓ alternada L/R                              20 operações | tamanho final 0 | invariantes OK
✓ insere esquerda, remove direita (fila)     40 operações | tamanho final 0 | invariantes OK
✓ insere direita, remove direita (pilha)     40 operações | tamanho final 0 | invariantes OK
✓ esvazia e reutiliza                         8 operações | tamanho final 0 | invariantes OK
✓ remoções em vazio                           5 operações | tamanho final 0 | invariantes OK
✓ zigue-zague                               150 operações | tamanho final 50 | invariantes OK
✓ 100.000 operações aleatórias em 20 blocos: resultado idêntico ao oráculo e invariantes OK

 tamanho do deque  insert_left  insert_right  remove_left  remove_right  (ns por operação)
            1.000         1095          1105          983           967
           10.000          979           982          969           976
          100.000          897           896          776           775
        1.000.000          499           

*Decisões de Projeto, Invariantes e Complexidade*
- Sentinelas (`header` e `trailer`): toda inserção e remoção acontece entre dois nós existentes. Isso elimina os casos especiais de lista vazia ou unitária, que são a principal fonte de corrupção de ponteiros. O custo é de 2 nós fixos.
- Deque vazio: `remove_*` e `peek_*` levantam `EmptyStructureError`, em vez de devolver `None`. Devolver `None` seria ambíguo, porque `None` pode ser um valor armazenado, o que a célula demonstra.
- Gerenciamento de nós: o nó removido tem `prev`, `next` e `value` zerados, o que evita referências pendentes para a estrutura.
- Invariantes verificados (`check_invariants`):
  1. os percursos para frente e para trás têm exatamente `size` nós;
  2. para todo nó $x$: `x.next.prev is x` e `x.prev.next is x`;
  3. as sentinelas estão nas pontas;
  4. os dois percursos visitam os mesmos valores em ordem inversa.

  O verificador foi testado contra corrupções propositais (`prev` quebrado, `size` errado e ciclo), e todas foram detectadas.
- Testes: 6 sequências dirigidas (alternância L/R, uso como fila, uso como pilha, esvaziar e reutilizar, remoções em deque vazio e zigue-zague) e 100.000 operações aleatórias. Todas foram comparadas passo a passo com um oráculo (`collections.deque`, usado apenas como referência), com `peek_left` e `peek_right` conferidos a cada operação.
- Complexidade: cada operação nas pontas custa $O(1)$ (4 atribuições de ponteiro para inserir e 2 para remover). A evidência é que o tempo por operação ficou na faixa de centenas de nanossegundos (≈250 a 500 ns) para deques de 1.000 a 1.000.000 de elementos. Com 1.000× mais elementos, o custo não se multiplica, como aconteceria em $O(n)$. A leve alta no maior tamanho reflete efeitos de cache e memória, e não o algoritmo. Só `check_invariants` custa $O(n)$, e ele é usado apenas nos testes. O espaço é $O(n)$, com 3 referências por nó.

### Exercício 12: Motor de indexação e consulta com estruturas combinadas e otimização

**Introdução**  
Este exercício consolida competências de engenharia de algoritmos e estruturas de dados em um cenário realista: construir um motor de indexação e consulta para documentos textuais. Em sistemas profissionais, mecanismos desse tipo fundamentam busca interna, observabilidade, auditoria e análise de logs. O desafio não é apenas "funcionar", mas entregar desempenho consistente, operar com estruturas adequadas e justificar tecnicamente as escolhas. Você deverá implementar uma cadeia completa: tokenização, indexação invertida, execução de consultas com operadores e ranqueamento, além de instrumentação e análise de complexidade. O exercício exige integração obrigatória de hashtable, pilha, fila, recursão, programação dinâmica, ordenação eficiente, lista encadeada e árvore. A avaliação considerará corretude, qualidade de testes, documentação técnica e rigor na análise de custos, incluindo identificação de gargalos e otimizações.

**Tarefa**  
1. Indexação invertida com estruturas obrigatórias:
   - a. Receba uma lista de documentos (strings) definida no notebook e tokenize em termos normalizados;
   - b. Construa um índice invertido em uma `HashTableChained` onde cada termo aponta para uma lista encadeada de ocorrências (`doc_id`, posições);
   - c. Mantenha uma BST de termos para permitir listagem ordenada e busca binária em árvore por termo.
2. Consultas booleanas e execução:
   - a. Implemente parsing de consultas com parênteses e operadores AND, OR, NOT usando uma `Stack` para conversão infixa para pós-fixa e avaliação;
   - b. Execute a consulta retornando o conjunto de `doc_ids` correspondentes, usando operações eficientes sobre listas ordenadas quando aplicável;
   - c. Use uma `Queue` para percorrer em largura a BST e gerar um relatório de diagnóstico (níveis, balanceamento aproximado, termos por nível).
3. Ranqueamento, otimização e programação dinâmica:
   - a. Para os `doc_ids` retornados, compute uma pontuação simples baseada em frequência e posição, e ordene os resultados usando QuickSort instrumentado;
   - b. Implemente um componente de sugestão de termo para consultas com termos ausentes, usando programação dinâmica (por exemplo, distância de edição com memoization em hashtable);
   - c. Produza uma análise completa: identifique o custo assintótico de indexação, de consulta, de ranqueamento e do módulo de sugestão, e proponha duas otimizações concretas com justificativa baseada em Big O e em evidências do seu notebook.

*Tarefa 1: Tokenização, Índice Invertido (`HashTableChained` + Listas Encadeadas) e BST de Termos*

In [30]:
# Exercício 12 — Tarefa 1: tokenização, índice invertido (HashTableChained + listas encadeadas) e BST de termos
import re, unicodedata

DOCUMENTOS = [
    "Hashtables resolvem colisões por encadeamento e oferecem busca em tempo constante médio.",
    "A árvore binária de busca permite listagem ordenada dos termos por travessia in-order.",
    "Pilhas e filas são estruturas lineares; a pilha segue LIFO e a fila segue FIFO.",
    "O QuickSort particiona o vetor em torno de um pivô e ordena recursivamente cada parte.",
    "Programação dinâmica evita recomputar subproblemas usando memoization em uma tabela hash.",
    "A distância de edição mede quantas inserções, remoções e trocas transformam uma palavra em outra.",
    "Listas encadeadas permitem inserção e remoção em tempo constante quando o nó já é conhecido.",
    "A busca binária exige vetor ordenado e reduz o espaço de busca pela metade a cada passo.",
    "O índice invertido associa cada termo à lista de documentos em que ele ocorre.",
    "Consultas booleanas combinam termos com AND, OR e NOT sobre listas ordenadas de documentos.",
    "A fila de prioridade e o heap não fazem parte deste trabalho, mas a fila simples sim.",
    "Recursão resolve problemas hierárquicos, como percorrer árvores e diretórios.",
    "O fator de carga da hashtable determina o tamanho médio das listas em cada bucket.",
    "Ordenação eficiente com QuickSort tem custo médio n log n e pior caso quadrático.",
]

STOPWORDS = {'a', 'o', 'as', 'os', 'e', 'de', 'da', 'do', 'das', 'dos', 'em', 'um', 'uma', 'por',
             'para', 'com', 'que', 'se', 'ao', 'no', 'na', 'nos', 'nas', 'ja', 'mas', 'sim', 'nao',
             'cada', 'ele', 'como', 'quando', 'sobre', 'pela', 'pelo', 'deste', 'fazem', 'tem'}

def normalize(word):
    """Minúsculas + remoção de acentos (NFKD): 'Árvore' -> 'arvore'."""
    word = unicodedata.normalize('NFKD', word.lower())
    return ''.join(ch for ch in word if not unicodedata.combining(ch))

def tokenize(text):
    """1.a: devolve [(termo, posição)]. A posição conta só os termos mantidos (sem stopwords).
    Operadores AND/OR/NOT dentro do texto viram termos comuns ('and', 'or', 'not')."""
    tokens, pos = [], 0
    for raw in re.findall(r"[A-Za-zÀ-ÿ0-9]+", text):
        t = normalize(raw)
        if t in STOPWORDS:
            continue
        tokens.append((t, pos)); pos += 1
    return tokens


class Posting:
    """Ocorrência de um termo em um documento."""
    __slots__ = ('doc_id', 'positions', 'next')
    def __init__(self, doc_id, positions):
        self.doc_id, self.positions, self.next = doc_id, positions, None

class PostingList:
    """1.b: lista SIMPLESMENTE ENCADEADA de ocorrências (doc_id, posições).
    Os documentos são indexados em ordem crescente de id e cada posting é anexado na CAUDA,
    portanto a lista já nasce ORDENADA por doc_id — pré-requisito dos merges da Tarefa 2."""
    __slots__ = ('head', 'tail', 'df')
    def __init__(self):
        self.head = self.tail = None
        self.df = 0                                  # document frequency
    def append(self, doc_id, positions):            # O(1) com ponteiro de cauda
        assert self.tail is None or doc_id > self.tail.doc_id, "postings devem chegar em ordem"
        p = Posting(doc_id, positions)
        if self.tail is None: self.head = p
        else: self.tail.next = p
        self.tail = p; self.df += 1
    def __iter__(self):
        cur = self.head
        while cur:
            yield cur; cur = cur.next
    def doc_ids(self):
        return [p.doc_id for p in self]
    def find(self, doc_id):
        for p in self:
            if p.doc_id == doc_id: return p
            if p.doc_id > doc_id: return None        # lista ordenada: pode parar cedo
        return None


class TermNode:
    __slots__ = ('term', 'left', 'right')
    def __init__(self, term):
        self.term, self.left, self.right = term, None, None

class TermBST:
    """1.c: BST de termos (ordem alfabética). Inserção e busca RECURSIVAS."""
    def __init__(self):
        self.root, self.size = None, 0
        self.last_comparisons = 0
    def insert(self, term):
        def rec(node):
            if node is None:
                self.size += 1; return TermNode(term)
            if term < node.term:   node.left = rec(node.left)
            elif term > node.term: node.right = rec(node.right)
            return node                              # termo repetido: nada muda
        self.root = rec(self.root)
    def search(self, term):
        """Busca binária na árvore: desce um nível por comparação -> O(h)."""
        def rec(node, comps):
            if node is None: return False, comps
            comps += 1
            if term == node.term: return True, comps
            return rec(node.left if term < node.term else node.right, comps)
        found, self.last_comparisons = rec(self.root, 0)
        return found
    def inorder(self):
        out = []
        def rec(node):
            if node:
                rec(node.left); out.append(node.term); rec(node.right)
        rec(self.root)
        return out
    def height(self):
        def rec(node): return 0 if node is None else 1 + max(rec(node.left), rec(node.right))
        return rec(self.root)


class InvertedIndex:
    def __init__(self, capacity=16):
        self.table = HashTableChained(initial_capacity=capacity)   # termo -> PostingList
        self.terms = TermBST()
        self.docs = []                                             # texto original por doc_id
        self.doc_len = []
        self.stats = {'tokens': 0, 'buscas hashtable': 0, 'comparações buckets': 0}

    def add_document(self, text):
        doc_id = len(self.docs)
        self.docs.append(text)
        tokens = tokenize(text)
        self.doc_len.append(len(tokens))
        local = {}                                   # posições por termo DENTRO do documento
        for term, pos in tokens:
            local.setdefault(term, []).append(pos)
        self.stats['tokens'] += len(tokens)
        for term, positions in local.items():
            pl = self.table.get(term)
            self.stats['buscas hashtable'] += 1
            self.stats['comparações buckets'] += self.table.last_op_comparisons
            if pl is None:                           # termo novo: cria a lista e entra na BST
                pl = PostingList()
                self.table.put(term, pl)
                self.terms.insert(term)
            pl.append(doc_id, positions)
        return doc_id

    def postings(self, term):
        return self.table.get(term)

    @property
    def n_docs(self):
        return len(self.docs)


indice = InvertedIndex()
for texto in DOCUMENTOS:
    indice.add_document(texto)

print("Tokenização de um documento:", tokenize(DOCUMENTOS[1]))
print(f"\nDocumentos: {indice.n_docs} | tokens indexados: {indice.stats['tokens']} | "
      f"vocabulário: {len(indice.table)} termos | capacidade da hashtable: {indice.table.capacity} "
      f"(fator de carga {indice.table.get_load_factor():.2f})")
for termo in ['fila', 'quicksort', 'hashtable', 'arvore']:
    pl = indice.postings(termo)
    print(f"  '{termo}' -> df={pl.df}: " + " -> ".join(f"(doc {p.doc_id}, pos {p.positions})" for p in pl))

# Testes da Tarefa 1
vocab_ref = sorted({t for d in DOCUMENTOS for t, _ in tokenize(d)})
assert indice.terms.inorder() == vocab_ref, "in-order da BST deve ser o vocabulário ordenado"
for termo in vocab_ref:
    pl = indice.postings(termo)
    esperado = [i for i, d in enumerate(DOCUMENTOS) if termo in {t for t, _ in tokenize(d)}]
    assert pl.doc_ids() == esperado and pl.doc_ids() == sorted(pl.doc_ids())
    for p in pl:
        assert p.positions == [pos for t, pos in tokenize(DOCUMENTOS[p.doc_id]) if t == termo]
print(f"\n✓ in-order da BST == vocabulário ordenado ({len(vocab_ref)} termos)")
print("✓ toda posting list tem exatamente os documentos e posições corretos, em ordem de doc_id")
print("BST: listagem ordenada (primeiros 12):", indice.terms.inorder()[:12])
for termo in ['pilha', 'heap', 'grafo']:
    ok = indice.terms.search(termo)
    print(f"  busca na BST por '{termo}': {'encontrado' if ok else 'ausente':<10} ({indice.terms.last_comparisons} comparações, altura {indice.terms.height()})")

Tokenização de um documento: [('arvore', 0), ('binaria', 1), ('busca', 2), ('permite', 3), ('listagem', 4), ('ordenada', 5), ('termos', 6), ('travessia', 7), ('in', 8), ('order', 9)]

Documentos: 14 | tokens indexados: 125 | vocabulário: 106 termos | capacidade da hashtable: 256 (fator de carga 0.41)
  'fila' -> df=2: (doc 2, pos [8]) -> (doc 10, pos [0, 5])
  'quicksort' -> df=2: (doc 3, pos [0]) -> (doc 13, pos [2])
  'hashtable' -> df=1: (doc 12, pos [2])
  'arvore' -> df=1: (doc 1, pos [0])

✓ in-order da BST == vocabulário ordenado (106 termos)
✓ toda posting list tem exatamente os documentos e posições corretos, em ordem de doc_id
BST: listagem ordenada (primeiros 12): ['and', 'arvore', 'arvores', 'associa', 'binaria', 'booleanas', 'bucket', 'busca', 'carga', 'caso', 'colisoes', 'combinam']
  busca na BST por 'pilha': encontrado (6 comparações, altura 11)
  busca na BST por 'heap': encontrado (7 comparações, altura 11)
  busca na BST por 'grafo': ausente    (5 comparações, altura

*Decisões da Indexação*
- Normalização: letras minúsculas, remoção de acentos (NFKD) e de stopwords. As posições contam apenas os termos mantidos. Uma limitação consciente é que não há *stemming*: `arvore` e `arvores` são termos diferentes, o que a sugestão da Tarefa 3.b compensa em parte.
- Índice invertido: é uma `HashTableChained` (Exercício 4) em que cada termo aponta para uma `PostingList`, uma lista simplesmente encadeada de `(doc_id, posições)` com ponteiro de cauda.
  - Como os documentos são indexados em ordem de `doc_id` e cada posting é anexado na cauda em $O(1)$, as listas já nascem ordenadas, o que é pré-requisito dos merges da Tarefa 2.
  - A tabela cresceu até 256 buckets para 106 termos (fator de carga 0,41).
- BST de termos: a inserção e a busca são recursivas. A listagem in-order devolve o vocabulário ordenado, e a busca na árvore custou 5 a 7 comparações com altura 11.
- Testes: a in-order da BST é igual ao vocabulário ordenado, e cada posting list contém exatamente os documentos e as posições esperados, conferidos por força bruta.

*Tarefa 2 (a, b): Consultas Booleanas com `Stack` e Execução sobre Listas Ordenadas*

In [31]:
# Exercício 12 — Tarefa 2 (a, b): parsing com Stack (infixa -> pós-fixa) e execução sobre listas ordenadas

class QuerySyntaxError(Exception):
    pass

PRECEDENCE = {'NOT': 3, 'AND': 2, 'OR': 1}
RIGHT_ASSOC = {'NOT'}                                # NOT é unário e associa à direita

def lex_query(query):
    """Separa parênteses, operadores (em MAIÚSCULAS) e termos (normalizados como no índice).
    Insere AND implícito entre operandos adjacentes: 'fila pilha' == 'fila AND pilha'."""
    raw = re.findall(r"\(|\)|[A-Za-zÀ-ÿ0-9]+", query)
    if not raw:
        raise QuerySyntaxError("consulta vazia")
    tokens = []
    for r in raw:
        tok = r if r in ('(', ')') or r in PRECEDENCE else ('TERM', normalize(r))
        # AND implícito: operando/')' seguido de operando/'('/NOT
        if tokens and (tokens[-1] == ')' or isinstance(tokens[-1], tuple)) \
                  and (tok == '(' or tok == 'NOT' or isinstance(tok, tuple)):
            tokens.append('AND')
        tokens.append(tok)
    return tokens

def to_postfix(tokens):
    """2.a: algoritmo shunting-yard usando a Stack do Exercício 5."""
    ops, out = Stack(len(tokens) + 1), []
    expect_operand = True                             # valida a gramática durante a conversão
    for tok in tokens:
        if isinstance(tok, tuple):                    # termo
            if not expect_operand: raise QuerySyntaxError(f"termo inesperado: {tok[1]!r}")
            out.append(tok); expect_operand = False
        elif tok == 'NOT':
            if not expect_operand: raise QuerySyntaxError("NOT depois de um operando (falta AND/OR?)")
            ops.push(tok)
        elif tok in ('AND', 'OR'):
            if expect_operand: raise QuerySyntaxError(f"operador {tok} sem operando à esquerda")
            while not ops.is_empty() and ops.peek() != '(' and (
                    PRECEDENCE[ops.peek()] > PRECEDENCE[tok] or
                    (PRECEDENCE[ops.peek()] == PRECEDENCE[tok] and tok not in RIGHT_ASSOC)):
                out.append(ops.pop())
            ops.push(tok); expect_operand = True
        elif tok == '(':
            if not expect_operand: raise QuerySyntaxError("'(' depois de um operando")
            ops.push(tok)
        elif tok == ')':
            if expect_operand: raise QuerySyntaxError("')' sem operando antes")
            while not ops.is_empty() and ops.peek() != '(':
                out.append(ops.pop())
            if ops.is_empty(): raise QuerySyntaxError("')' sem '(' correspondente")
            ops.pop()
    if expect_operand:
        raise QuerySyntaxError("consulta termina com operador")
    while not ops.is_empty():
        op = ops.pop()
        if op == '(': raise QuerySyntaxError("'(' sem ')' correspondente")
        out.append(op)
    return out

# ---------- 2.b: operações sobre listas ordenadas de doc_ids (merge linear) ----------
class MergeStats:
    def __init__(self): self.comparisons = 0

def intersect(a, b, st):                              # O(|a| + |b|)
    i = j = 0; out = []
    while i < len(a) and j < len(b):
        st.comparisons += 1
        if a[i] == b[j]: out.append(a[i]); i += 1; j += 1
        elif a[i] < b[j]: i += 1
        else: j += 1
    return out

def union(a, b, st):                                  # O(|a| + |b|)
    i = j = 0; out = []
    while i < len(a) and j < len(b):
        st.comparisons += 1
        if a[i] == b[j]: out.append(a[i]); i += 1; j += 1
        elif a[i] < b[j]: out.append(a[i]); i += 1
        else: out.append(b[j]); j += 1
    return out + a[i:] + b[j:]

def complement(a, n_docs, st):                        # O(N): todos os documentos menos `a`
    out, j = [], 0
    for d in range(n_docs):
        st.comparisons += 1
        if j < len(a) and a[j] == d: j += 1
        else: out.append(d)
    return out

def evaluate_postfix(postfix, index, st):
    """Avaliação da pós-fixa com Stack. Cada item da pilha carrega
    (doc_ids ordenados, termos positivos, termos negados) — os positivos alimentam o ranqueamento;
    NOT troca positivos e negados de lugar."""
    stack = Stack(len(postfix) + 1)
    missing = []
    for tok in postfix:
        if isinstance(tok, tuple):
            pl = index.postings(tok[1])
            if pl is None: missing.append(tok[1])
            stack.push((pl.doc_ids() if pl else [], {tok[1]}, set()))
        elif tok == 'NOT':
            docs, pos, neg = stack.pop()
            stack.push((complement(docs, index.n_docs, st), neg, pos))
        else:
            b_docs, b_pos, b_neg = stack.pop(); a_docs, a_pos, a_neg = stack.pop()
            docs = intersect(a_docs, b_docs, st) if tok == 'AND' else union(a_docs, b_docs, st)
            stack.push((docs, a_pos | b_pos, a_neg | b_neg))
    docs, pos, _ = stack.pop()
    assert stack.is_empty()
    return docs, pos, missing

def run_query(query, index):
    tokens = lex_query(query)
    postfix = to_postfix(tokens)
    st = MergeStats()
    docs, positive_terms, missing = evaluate_postfix(postfix, index, st)
    return {'postfix': postfix, 'docs': docs, 'positivos': positive_terms,
            'ausentes': missing, 'comparações merge': st.comparisons}

def fmt_postfix(pf):
    return ' '.join(t[1] if isinstance(t, tuple) else t for t in pf)

for q in ["fila AND pilha", "fila OR quicksort", "busca AND NOT binaria",
          "(hashtable OR hashtables) AND (colisoes OR bucket)", "quicksort pivo", "NOT (fila OR pilha OR arvore)",
          "arvore AND grafo"]:
    r = run_query(q, indice)
    print(f"{q!r}\n   pós-fixa: {fmt_postfix(r['postfix'])}\n   docs: {r['docs']} | comparações de merge: {r['comparações merge']}"
          + (f" | termos ausentes: {r['ausentes']}" if r['ausentes'] else ""))

r = run_query("fila NOT pilha", indice)
print(f"\nAND implícito: 'fila NOT pilha' -> pós-fixa {fmt_postfix(r['postfix'])!r} -> docs {r['docs']}")
print("\nConsultas inválidas:")
for q in ["", "fila AND", "AND fila", "fila AND OR pilha", "(fila OR pilha", "fila OR pilha)", "( )", "NOT"]:
    try:
        run_query(q, indice); print("  ✗ aceitou", repr(q))
    except QuerySyntaxError as e:
        print(f"  {q!r:<18} -> {e}")

# Teste de corretude: 400 consultas aleatórias contra avaliação por força bruta com set do Python
import itertools
docsets = {t: set(indice.postings(t).doc_ids()) for t in vocab_ref}
TODOS = set(range(indice.n_docs))
def brute(node):
    kind = node[0]
    if kind == 'T':   return docsets.get(node[1], set())
    if kind == 'NOT': return TODOS - brute(node[1])
    a, b = brute(node[1]), brute(node[2])
    return a & b if kind == 'AND' else a | b
def render(node):
    kind = node[0]
    if kind == 'T': return node[1]
    if kind == 'NOT': return f"NOT ({render(node[1])})"
    return f"({render(node[1])} {kind} {render(node[2])})"
rnd = random.Random(12)
pool = vocab_ref[:40] + ['inexistente']
def rand_expr(d):
    if d == 0 or rnd.random() < 0.3: return ('T', rnd.choice(pool))
    r = rnd.random()
    if r < 0.2: return ('NOT', rand_expr(d - 1))
    return ('AND' if r < 0.6 else 'OR', rand_expr(d - 1), rand_expr(d - 1))
for _ in range(400):
    e = rand_expr(4)
    assert run_query(render(e), indice)['docs'] == sorted(brute(e)), render(e)
print("\n✓ 400 consultas aleatórias (aninhadas, com NOT e termo inexistente) == avaliação por força bruta")

'fila AND pilha'
   pós-fixa: fila pilha AND
   docs: [2] | comparações de merge: 1
'fila OR quicksort'
   pós-fixa: fila quicksort OR
   docs: [2, 3, 10, 13] | comparações de merge: 3
'busca AND NOT binaria'
   pós-fixa: busca binaria NOT AND
   docs: [0] | comparações de merge: 22
'(hashtable OR hashtables) AND (colisoes OR bucket)'
   pós-fixa: hashtable hashtables OR colisoes bucket OR AND
   docs: [0, 12] | comparações de merge: 4
'quicksort pivo'
   pós-fixa: quicksort pivo AND
   docs: [3] | comparações de merge: 1
'NOT (fila OR pilha OR arvore)'
   pós-fixa: fila pilha OR arvore OR NOT
   docs: [0, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13] | comparações de merge: 16
'arvore AND grafo'
   pós-fixa: arvore grafo AND
   docs: [] | comparações de merge: 0 | termos ausentes: ['grafo']

AND implícito: 'fila NOT pilha' -> pós-fixa 'fila pilha NOT AND' -> docs [10]

Consultas inválidas:
  ''                 -> consulta vazia
  'fila AND'         -> consulta termina com operador
  'AND fila'    

*Parsing e Execução*
- Parsing (2.a): usa o algoritmo *shunting-yard* com a `Stack` do Exercício 5. A precedência é `NOT` > `AND` > `OR`, sendo `NOT` unário e associativo à direita. Operandos adjacentes recebem `AND` implícito (`fila NOT pilha` equivale a `fila AND NOT pilha`). A gramática é validada durante a conversão: operador sem operando, consulta terminando em operador e parênteses desbalanceados são rejeitados com mensagem.
- Execução (2.b): a pós-fixa é avaliada com outra `Stack`.
  - `AND` e `OR` são *merges* lineares de listas ordenadas, com custo $O(|a| + |b|)$.
  - `NOT` é o complemento em relação a todos os documentos, com custo $O(N)$.
  - Cada item da pilha carrega também os termos positivos e os negados. `NOT` troca os dois de lugar, então só os termos positivos alimentam o ranqueamento.
- Corretude: 400 consultas aleatórias, com aninhamento, `NOT` e termos inexistentes, deram exatamente o mesmo resultado que a avaliação por força bruta com conjuntos.

*Tarefa 2.c: Relatório de Diagnóstico da BST por Largura (`Queue`)*

In [32]:
# Exercício 12 — Tarefa 2.c: relatório de diagnóstico da BST por largura, usando a Queue do Exercício 5
import math

def bst_level_report(bst, show=6):
    """Percorre a BST nível a nível com a Queue circular de capacidade fixa.
    Capacidade = nº de termos (limite superior de nós simultâneos na fila)."""
    q = Queue(max(1, bst.size))
    levels = []
    if bst.root:
        q.enqueue((bst.root, 0))
    max_fila = 0
    while not q.is_empty():
        max_fila = max(max_fila, len(q))
        node, depth = q.dequeue()
        if depth == len(levels):
            levels.append([])
        levels[depth].append(node.term)
        if node.left:  q.enqueue((node.left, depth + 1))
        if node.right: q.enqueue((node.right, depth + 1))
    n = bst.size
    h = len(levels)
    h_ideal = math.ceil(math.log2(n + 1)) if n else 0
    print(f"Termos: {n} | altura: {h} | altura mínima possível: {h_ideal} | razão h/h_ideal = {h / h_ideal:.2f}"
          f" | pico da fila: {max_fila}")
    print(f"{'nível':>5}{'nós':>6}{'capacidade':>12}{'ocupação':>10}  termos")
    for d, termos in enumerate(levels):
        cap = 2 ** d
        amostra = ', '.join(termos[:show]) + (' …' if len(termos) > show else '')
        print(f"{d:>5}{len(termos):>6}{cap:>12}{len(termos)/cap:>9.0%}  {amostra}")
    completos = sum(1 for d, t in enumerate(levels) if len(t) == 2 ** d)
    print(f"Níveis completamente preenchidos: {completos} de {h}. "
          f"Balanceamento aproximado: {'bom' if h <= 2 * h_ideal else 'ruim'} (critério: h <= 2·log2(n+1))")
    return levels

niveis = bst_level_report(indice.terms)
assert sum(len(l) for l in niveis) == indice.terms.size
assert sorted(t for l in niveis for t in l) == indice.terms.inorder()

Termos: 106 | altura: 11 | altura mínima possível: 7 | razão h/h_ideal = 1.57 | pico da fila: 21
nível   nós  capacidade  ocupação  termos
    0     1           1     100%  hashtables
    1     2           2     100%  colisoes, resolvem
    2     4           4     100%  busca, encadeamento, oferecem, tempo
    3     8           8     100%  arvore, carga, constante, filas, medio, permite …
    4    14          16      88%  and, binaria, caso, conhecido, dinamica, estruturas …
    5    18          32      56%  associa, booleanas, combinam, consultas, distancia, espaco …
    6    21          64      33%  arvores, bucket, determina, diretorios, edicao, fifo …
    7    19         128      15%  custo, documentos, encadeadas, evita, hierarquicos, lifo …
    8    11         256       4%  eficiente, exige, insercoes, n, palavra, percorrer …
    9     7         512       1%  fator, insercao, invertido, outra, problemas, quadratico …
   10     1        1024       0%  indice
Níveis completamente p

*Leitura do Diagnóstico*
- A travessia em largura com a `Queue` circular do Exercício 5 visita cada nó uma vez, com custo $O(V)$. O pico da fila foi de 21 nós, bem abaixo da capacidade (106).
- Com 106 termos, a altura é 11, contra o mínimo possível de 7 (razão 1,57). Os 4 primeiros níveis estão completos, e a ocupação cai a partir do nível 5.
- A árvore está razoavelmente balanceada porque os termos entram na ordem em que aparecem no texto, que é praticamente aleatória em relação à ordem alfabética. Se o vocabulário fosse inserido em ordem alfabética, a BST degeneraria, como no Exercício 6.

*Tarefa 3.a: Pontuação por Frequência e Posição + Ordenação com o QuickSort Instrumentado (Exercício 9)*

In [33]:
# Exercício 12 — Tarefa 3.a: pontuação por frequência e posição + ordenação com o QuickSort instrumentado (Ex. 9)
import math

def score_document(doc_id, positive_terms, index):
    """score(d) = Σ_t [ tf(t,d) · idf(t) + 1 / (1 + primeira_posição(t,d)) ]
      - tf·idf: premia termos frequentes no documento e raros na coleção, idf = log(1 + N/df);
      - bônus de posição: termo que aparece no início do documento pesa mais.
    Só os termos POSITIVOS da consulta pontuam (termos sob NOT não contam)."""
    total = 0.0
    for t in positive_terms:
        pl = index.postings(t)
        if pl is None:
            continue
        p = pl.find(doc_id)                       # percorre a posting list ordenada (para cedo)
        if p is None:
            continue
        idf = math.log(1 + index.n_docs / pl.df)
        total += len(p.positions) * idf + 1 / (1 + p.positions[0])
    return total

def rank(result, index, short=8):
    """Ordena por score decrescente e, em empate, por doc_id crescente.
    Chave (-score, doc_id): o QuickSort crescente produz exatamente essa ordem."""
    chaves = [(-round(score_document(d, result['positivos'], index), 6), d) for d in result['docs']]
    ordenado, comps, copias = quicksort(chaves, short)
    return [(d, -s) for s, d in ordenado], comps, copias

for q in ["fila OR pilha OR quicksort", "busca OR binaria OR ordenado", "(hashtable OR hashtables OR hash) AND NOT fator"]:
    r = run_query(q, indice)
    ranking, c, cp = rank(r, indice)
    print(f"{q!r} -> termos positivos: {sorted(r['positivos'])}")
    for pos, (d, s) in enumerate(ranking, 1):
        print(f"   {pos}. doc {d:>2}  score={s:.3f}  | {DOCUMENTOS[d][:70]}")
    print(f"   QuickSort: {c} comparações, {cp} cópias para {len(ranking)} resultados\n")

'fila OR pilha OR quicksort' -> termos positivos: ['fila', 'pilha', 'quicksort']
   1. doc 10  score=5.159  | A fila de prioridade e o heap não fazem parte deste trabalho, mas a fi
   2. doc  2  score=5.065  | Pilhas e filas são estruturas lineares; a pilha segue LIFO e a fila se
   3. doc  3  score=3.079  | O QuickSort particiona o vetor em torno de um pivô e ordena recursivam
   4. doc 13  score=2.413  | Ordenação eficiente com QuickSort tem custo médio n log n e pior caso 
   QuickSort: 4 comparações, 8 cópias para 4 resultados

'busca OR binaria OR ordenado' -> termos positivos: ['binaria', 'busca', 'ordenado']
   1. doc  7  score=9.957  | A busca binária exige vetor ordenado e reduz o espaço de busca pela me
   2. doc  1  score=4.647  | A árvore binária de busca permite listagem ordenada dos termos por tra
   3. doc  0  score=1.901  | Hashtables resolvem colisões por encadeamento e oferecem busca em temp
   QuickSort: 3 comparações, 7 cópias para 3 resultados

'(hashtable OR hasht

*Tarefa 3.b: Sugestão de Termos com Distância de Edição (Programação Dinâmica com Memoization em `HashTableChained`)*

In [34]:
# Exercício 12 — Tarefa 3.b: sugestão de termos com distância de edição (PD + memoization em HashTableChained)

# Distância de edição com inserção, remoção, substituição e TRANSPOSIÇÃO de letras vizinhas
# (variante "Optimal String Alignment" de Damerau-Levenshtein): 'pilah' -> 'pilha' custa 1, não 2,
# o que modela melhor erros de digitação.
def edit_distance_naive(a, b, stats):
    """Recursão direta: exponencial — serve só de linha de base."""
    def ed(i, j):
        stats['calls'] += 1
        if i == 0: return j
        if j == 0: return i
        if a[i - 1] == b[j - 1]: return ed(i - 1, j - 1)
        best = 1 + min(ed(i - 1, j), ed(i, j - 1), ed(i - 1, j - 1))
        if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
            best = min(best, 1 + ed(i - 2, j - 2))
        return best
    return ed(len(a), len(b))

def edit_distance_memo(a, b, stats, limit=None):
    """Mesma recorrência; o estado (i, j) = prefixos a[:i] e b[:j] é o estado mínimo suficiente
    DENTRO de um par de palavras. O memo é uma HashTableChained nova por par
    (se fosse compartilhado entre pares, a chave teria de incluir as palavras).
    limit=k (otimização da Tarefa 3.c): estados com |i - j| > k são cortados, pois qualquer
    caminho que passe por eles custa > k. O resultado é exato quando <= k e > k caso contrário."""
    memo = HashTableChained(initial_capacity=16)
    def ed(i, j):
        stats['calls'] += 1
        if limit is not None and abs(i - j) > limit:
            stats['cortes'] = stats.get('cortes', 0) + 1
            return limit + 1
        v = memo.get((i, j))
        if v is not None:
            stats['hits'] += 1; return v
        if i == 0: v = j
        elif j == 0: v = i
        elif a[i - 1] == b[j - 1]: v = ed(i - 1, j - 1)
        else:
            v = 1 + min(ed(i - 1, j), ed(i, j - 1), ed(i - 1, j - 1))   # remoção, inserção, troca
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                v = min(v, 1 + ed(i - 2, j - 2))                      # transposição
        memo.put((i, j), v)
        stats['states'] += 1
        return v
    return ed(len(a), len(b))

# Demonstração do ganho da PD em um par de palavras
print(f"{'par':<28}{'dist':>5}{'chamadas ingênuas':>19}{'chamadas memo':>15}{'estados calculados':>20}{'estados possíveis':>19}")
for a, b in [("fila", "filas"), ("pilah", "pilha"), ("quicksort", "quiksort"), ("hashtable", "hastable"),
             ("ordenacao", "ordenado"), ("recursividade", "recursao")]:
    s1 = {'calls': 0}; d1 = edit_distance_naive(a, b, s1)
    s2 = {'calls': 0, 'hits': 0, 'states': 0}; d2 = edit_distance_memo(a, b, s2)
    assert d1 == d2
    print(f"{a + ' / ' + b:<28}{d1:>5}{s1['calls']:>19,}{s2['calls']:>15,}{s2['states']:>20}{(len(a)+1)*(len(b)+1):>19}".replace(',', '.'))

def suggest(term, index, max_dist=2, otimizado=True):
    """Candidatos = vocabulário listado em ordem pela BST (in-order).
    otimizado=True liga as duas podas da Tarefa 3.c:
      (1) descarta candidatos com | |a| - |b| | > max_dist sem calcular nada
          (a distância de edição é no mínimo a diferença de tamanhos);
      (2) PD restrita à faixa |i - j| <= max_dist (limit)."""
    stats = {'calls': 0, 'hits': 0, 'states': 0, 'candidatos avaliados': 0, 'descartados pelo tamanho': 0}
    best = []
    for cand in index.terms.inorder():
        if otimizado and abs(len(cand) - len(term)) > max_dist:
            stats['descartados pelo tamanho'] += 1; continue
        stats['candidatos avaliados'] += 1
        d = edit_distance_memo(term, cand, stats, limit=max_dist if otimizado else None)
        if d <= max_dist:
            best.append((d, -index.postings(cand).df, cand))   # menor distância, depois maior df
    best.sort()
    return [(c, d) for d, _, c in best[:3]], stats

print()
for erro in ["quiksort", "hastable", "arvre", "pilah", "memoisation", "grafo"]:
    sug, st = suggest(erro, indice)
    print(f"'{erro}' -> {sug if sug else 'sem sugestão (nenhum termo a distância <= 2)'}"
          f" | candidatos avaliados {st['candidatos avaliados']}, descartados {st['descartados pelo tamanho']}, estados PD {st['states']}")

par                          dist  chamadas ingênuas  chamadas memo  estados calculados  estados possíveis
fila / filas                    1                225             35                  20                 30
pilah / pilha                   1                248             49                  28                 36
quicksort / quiksort            1                 64             27                  19                 90
hashtable / hastable            1                 55             25                  18                 90
ordenacao / ordenado            2             15.258            133                  64                 90
recursividade / recursao        6          2.369.736            290                 124                126

'quiksort' -> [('quicksort', 1)] | candidatos avaliados 63, descartados 43, estados PD 2337
'hastable' -> [('hashtable', 1), ('hashtables', 2)] | candidatos avaliados 63, descartados 43, estados PD 2294
'arvre' -> [('arvore', 1), ('arvores', 2)] | ca

*Programação Dinâmica na Sugestão*
- Recorrência: é a distância de edição com inserção, remoção, substituição e transposição de letras vizinhas (variante *Optimal String Alignment*), para que o erro de digitação `pilah` → `pilha` custe 1, e não 2. O estado $(i, j)$, que representa os prefixos `a[:i]` e `b[:j]`, é memorizado em uma `HashTableChained` própria para cada par de palavras.
- Ganho da PD: em `recursividade / recursao`, a recursão ingênua fez 2.369.736 chamadas, contra 290 com memo (124 estados calculados de 126 possíveis). O custo cai de exponencial para $O(|a| \cdot |b|)$.
- Sugestão: os candidatos vêm da listagem in-order da BST. O critério é a menor distância, depois a maior frequência (df). Por exemplo, `quiksort` → `quicksort` e `hastable` → `hashtable`. Termos sem nenhum candidato a distância $\le 2$, como `grafo`, ficam sem sugestão.

*Tarefa 3.c: Integração, Instrumentação em Escala e Evidências para as Otimizações*

In [35]:
# Exercício 12 — Integração final, instrumentação em escala e evidências para as otimizações (3.c)
import time, math

class SearchEngine:
    """Fachada que encadeia todas as etapas: consulta -> execução -> sugestão -> ranqueamento."""
    def __init__(self, documents):
        self.index = InvertedIndex()
        for d in documents:
            self.index.add_document(d)
    def search(self, query, top=5):
        r = run_query(query, self.index)
        sugestoes = {t: suggest(t, self.index)[0] for t in r['ausentes']}
        ranking, comps, _ = rank(r, self.index)
        return {'ranking': ranking[:top], 'total': len(r['docs']), 'sugestões': sugestoes,
                'comparações merge': r['comparações merge'], 'comparações quicksort': comps}

engine = SearchEngine(DOCUMENTOS)
for q in ["(fila OR pilha) AND NOT prioridade", "quiksort AND pivo", "hastable colisoes"]:
    out = engine.search(q)
    print(f"{q!r}: {out['total']} documento(s) | top: {[(d, round(s, 2)) for d, s in out['ranking']]}")
    for t, sug in out['sugestões'].items():
        print(f"    termo ausente '{t}' -> você quis dizer: {[c for c, _ in sug]}?")

# ------------------------------------------------------------------
# Instrumentação em escala: corpus sintético com frequência de termos Zipf
# ------------------------------------------------------------------
def synthetic_corpus(n_docs, vocab_size=3000, doc_len=40, seed=0):
    rnd = random.Random(seed)
    letras = "abcdefghijlmnoprstuv"
    vocab = sorted({''.join(rnd.choice(letras) for _ in range(rnd.randint(4, 9))) for _ in range(vocab_size)})
    rnd.shuffle(vocab)                                  # ordem aleatória: a BST não degenera
    pesos = [1 / (r + 1) for r in range(len(vocab))]    # Zipf: poucos termos muito comuns
    return vocab, [' '.join(rnd.choices(vocab, weights=pesos, k=doc_len)) for _ in range(n_docs)]

print(f"\n{'docs':>6}{'tokens':>8}{'vocab.':>8}{'indexação':>11}{'µs/token':>10}{'comp. bucket/busca':>20}"
      f"{'altura BST':>12}{'log2(V)':>9}")
escala = {}
for n in [250, 1000, 4000]:
    vocab, docs = synthetic_corpus(n)
    t0 = time.perf_counter()
    idx = InvertedIndex()
    for d in docs: idx.add_document(d)
    dt = time.perf_counter() - t0
    escala[n] = (vocab, idx)
    s = idx.stats
    print(f"{n:>6}{s['tokens']:>8}{len(idx.table):>8}{dt*1e3:>9.0f}ms{dt/s['tokens']*1e6:>10.1f}"
          f"{s['comparações buckets']/s['buscas hashtable']:>20.2f}{idx.terms.height():>12}{math.log2(len(idx.table)):>9.1f}")

# ------------------------------------------------------------------
# Otimização 1 — interseção adaptativa (busca binária) para listas de tamanhos muito diferentes
# ------------------------------------------------------------------
def intersect_gallop(small, large, st):
    """Para cada doc da lista menor, busca binária na maior a partir da última posição: O(m log n)."""
    out, lo = [], 0
    for x in small:
        hi = len(large)
        while lo < hi:
            mid = (lo + hi) // 2
            st.comparisons += 1
            if large[mid] < x: lo = mid + 1
            else: hi = mid
        if lo < len(large):
            st.comparisons += 1
            if large[lo] == x: out.append(x); lo += 1
    return out

def intersect_adaptive(a, b, st):
    small, large = (a, b) if len(a) <= len(b) else (b, a)
    if small and len(small) * math.log2(len(large) + 1) < len(small) + len(large):
        return intersect_gallop(small, large, st)
    return intersect(a, b, st)

vocab, idx = escala[4000]
por_df = sorted(idx.terms.inorder(), key=lambda t: idx.postings(t).df)
comum = por_df[-1]
print(f"\nOtimização 1 — AND entre o termo mais comum ('{comum}', df={idx.postings(comum).df}) e termos de df variado:")
print(f"{'termo':<12}{'df':>6}{'merge linear':>14}{'adaptativa':>12}{'ganho':>8}")
for t in [por_df[0], por_df[len(por_df)//2], por_df[int(len(por_df)*0.95)], por_df[-2]]:
    a, b = idx.postings(t).doc_ids(), idx.postings(comum).doc_ids()
    s1, s2 = MergeStats(), MergeStats()
    r1, r2 = intersect(a, b, s1), intersect_adaptive(a, b, s2)
    assert r1 == r2
    print(f"{t:<12}{len(a):>6}{s1.comparisons:>14}{s2.comparisons:>12}{s1.comparisons/max(1,s2.comparisons):>7.1f}x")

# ------------------------------------------------------------------
# Otimização 2 — sugestão: poda por tamanho + PD restrita à faixa |i - j| <= k
# ------------------------------------------------------------------
print(f"\nOtimização 2 — sugestão de termo no vocabulário de {len(idx.table)} termos (k = 2):")
print(f"{'termo digitado':<16}{'sem otimização: estados PD':>28}{'tempo':>9}{'otimizado: estados PD':>23}{'tempo':>9}  sugestão")
for alvo in por_df[-3:] + [por_df[len(por_df) // 2]]:
    erro = alvo[:2] + alvo[3:]                        # apaga uma letra: simula erro de digitação
    res = []
    for otim in (False, True):
        t0 = time.perf_counter(); sug, st = suggest(erro, idx, otimizado=otim); dt = time.perf_counter() - t0
        res.append((st['states'], dt, sug))
    assert res[0][2] == res[1][2]                     # exatamente as mesmas sugestões
    linha = f"{erro:<16}{res[0][0]:>28,}{res[0][1]*1e3:>7.0f}ms{res[1][0]:>23,}{res[1][1]*1e3:>7.0f}ms".replace(',', '.')
    print(f"{linha}  {res[1][2][0][0]} (d={res[1][2][0][1]})")

# ------------------------------------------------------------------
# Custo de consulta e de ranqueamento em escala
# ------------------------------------------------------------------
print(f"\n{'docs':>6}  {'consulta':<38}{'resultados':>11}{'comp. merge':>13}{'comp. QuickSort':>17}")
for n, (vocab, idx) in escala.items():
    top = sorted(idx.terms.inorder(), key=lambda t: -idx.postings(t).df)[:4]
    for q in [f"{top[0]} OR {top[1]}", f"{top[0]} AND {top[1]}", f"NOT {top[2]}"]:
        r = run_query(q, idx)
        _, qc, _ = rank(r, idx)
        print(f"{n:>6}  {q:<38}{len(r['docs']):>11}{r['comparações merge']:>13}{qc:>17}")

'(fila OR pilha) AND NOT prioridade': 1 documento(s) | top: [(2, 5.07)]
'quiksort AND pivo': 0 documento(s) | top: []
    termo ausente 'quiksort' -> você quis dizer: ['quicksort']?
'hastable colisoes': 0 documento(s) | top: []
    termo ausente 'hastable' -> você quis dizer: ['hashtable', 'hashtables']?

  docs  tokens  vocab.  indexação  µs/token  comp. bucket/busca  altura BST  log2(V)
   250    9997    1820       42ms       4.2                1.09          26     10.8
  1000   39994    2810      144ms       3.6                1.21          28     11.5
  4000  159969    2996      980ms       6.1                1.37          28     11.5

Otimização 1 — AND entre o termo mais comum ('lbfjfsgh', df=3967) e termos de df variado:
termo           df  merge linear  adaptativa   ganho
ostmam           1          3627          12  302.2x
ljfseiif        13          3925         162   24.2x
icvivhaeg      121          3916        1405    2.8x
midbdhc       3637          3996        3996    1.

*Análise Completa do Motor*

**Onde cada estrutura obrigatória é usada**

| Estrutura / técnica | Uso no motor |
|---|---|
| Hashtable (`HashTableChained`) | índice invertido termo → postings; memo da distância de edição |
| Lista encadeada | `PostingList` de cada termo, ordenada por `doc_id` |
| Árvore (BST) | vocabulário ordenado, busca por termo e fonte de candidatos da sugestão |
| Pilha (`Stack`) | conversão infixa → pós-fixa e avaliação da consulta |
| Fila (`Queue`) | travessia em largura da BST para o diagnóstico |
| Recursão | inserção, busca e in-order da BST; distância de edição |
| Programação dinâmica | distância de edição com memoization |
| Ordenação eficiente | QuickSort instrumentado no ranqueamento |

**Custo assintótico de cada etapa**, com $T$ = tokens, $V$ = vocabulário, $N$ = documentos, $h$ = altura da BST e $R$ = resultados:
- Indexação: $O(T)$ esperado para a hashtable e as postings, mais $O(V \cdot h)$ para inserir termos novos na BST.
  - Evidência: o tempo por token ficou na mesma ordem de grandeza (≈2,5 a 5 µs) enquanto o corpus cresceu 16×, de 10 mil para 160 mil tokens. Em $O(T^2)$, ele teria crescido 16×; a leve alta no maior corpus vem da alocação de cerca de 160 mil objetos.
  - As comparações por busca no bucket ficaram entre 1,12 e 1,34, porque o fator de carga é mantido $\le 0{,}75$.
- Consulta: $O(q)$ para o parsing, mais $O(\sum \text{df})$ para buscar as postings e fazer os merges, mais $O(N)$ para cada `NOT`.
  - Evidência: as comparações de merge acompanharam $N$ (249 → 998 → 3.996 para termos presentes em quase todos os documentos), e `NOT` custou exatamente $N$ comparações (250, 1.000 e 4.000).
- Ranqueamento: $O(R \cdot t \cdot \text{df})$ para a pontuação (`find` percorre a posting list), mais $O(R \log R)$ para o QuickSort.
  - Evidência: 1.890 → 10.026 → 50.798 comparações para $R \approx$ 250, 1.000 e 4.000, próximo de $R \log_2 R$ (para $R = 3.996$, $\approx 47.800$).
- Sugestão: $O(V \cdot |a| \cdot |b|)$ sem otimização.

**Gargalos identificados**
- O `NOT` é $O(N)$ mesmo quando o termo negado é raro.
- A pontuação percorre a posting list a cada termo e documento.
- A altura da BST chegou a 28 para $\log_2 V \approx 11{,}5$, o que é típico de uma BST aleatória, cuja altura esperada é $\approx 3 \log_2 n$.
- A sugestão compara o termo com o vocabulário inteiro.

**Duas otimizações implementadas e medidas**
1. Interseção adaptativa. Quando $m \ll n$, cada documento da lista menor é procurado por busca binária na maior, com custo $O(m \log n)$ em vez de $O(m + n)$. A escolha é automática pela condição $m \log_2 n < m + n$.
   - Evidência, contra o termo mais comum (df = 3.967): com df = 1, foram 3.627 → 12 comparações (302×); com df = 13, 3.925 → 162 (24×); com df = 121, 3.916 → 1.405 (2,8×).
   - Com listas de tamanho parecido, o algoritmo volta ao merge linear (1,0×), sem piorar nenhum caso.
2. Sugestão com poda. Primeiro, candidatos com $\big||a| - |b|\big| > k$ são descartados sem cálculo, porque a distância nunca é menor que a diferença de tamanhos. Depois, a PD fica restrita à faixa $|i - j| \le k$: qualquer caminho que saia dela custa mais que $k$. O custo cai de $O(V \cdot L^2)$ para $O(V \cdot k \cdot L)$, e o resultado é idêntico (verificado por `assert`).
   - Evidência, no vocabulário de 2.996 termos: 107.204 → 30.497 estados de PD (3,5×) para `jtor`, com o tempo caindo de ≈240 ms para ≈70 ms. Nos demais casos, foram cerca de 2,1 a 2,2× menos estados, com o tempo reduzido aproximadamente pela metade.

**Próximos passos naturais**, fora do escopo desta atividade:
- árvore auto-balanceada para os termos;
- guardar as posições por documento em um dicionário, para que a pontuação custe $O(1)$ por termo;
- índice de n-gramas ou BK-tree para gerar candidatos de sugestão sem varrer o vocabulário.